# Libraries

In [1]:
import os
# Set up the download directory to be the same as the Python script directory
try:
    #Python file
    download_directory = os.path.dirname(os.path.realpath(__file__))
except:
    #Jupyter notebook
    download_directory = os.getcwd()

# Define the path for the new folder
download_directory = os.path.join(download_directory, "Download_folder")


# Create the folder if it does not exist
if not os.path.exists(download_directory):
    os.makedirs(download_directory)

download_directory

'h:\\Shared drives\\DevOps Projects\\Python projecs\\python-automations\\Amz - Wmt Pricing\\Download_folder'

In [2]:
import sys

def is_python_script():
    try:
        os.path.basename(__file__)
        print("Running python file...")
        return True
    except NameError:
        return False
    
# Navigate up from the current directory and then into the "Modules" directory

is_python_script = is_python_script()

if is_python_script:
    # Get the directory of the current Python file
    current_directory = os.path.dirname(os.path.realpath(__file__))

    # Navigate up from the current directory and then into the "Modules" directory
    path_to_append = os.path.join(current_directory, '..', 'modules')

else:
    path_to_append = os.path.join(os.getcwd(), '..', 'modules')

# Normalize the path to resolve any ".."
normalized_path = os.path.normpath(path_to_append)

# Append the normalized path to sys.path
sys.path.append(normalized_path)

In [3]:
#Import libraries
import pandas as pd
pd.set_option('display.max_columns', None)

import GoogleAPI_functions as gAPI #Here we also have all required libraries for google API.
import hybris as hybris
import pricing_module as pricing
# import math
import DW_connection as dw
import FTP_Connection as ftp_server

import numpy as np
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta

import json
import xml.etree.ElementTree as ET
import re
pd.options.mode.copy_on_write = True

Running python file...


# Params

In [4]:
days_before = 7

test = False
min_margin_update_prices = 0.059
max_margin_update_prices = 0.203

#New params to implement as checks: if any of these is met, then stop the code and let me know.
threshold_new_nodes_flag = 20000
threshold_price_increases_flag = 10000
threshold_price_decreases_flag = 10000

#This is used in the errors part
perc_failure_flag = 0.015

# Dates calcs

In [5]:
today_str = pd.to_datetime('today').strftime('%Y-%m-%d')
date_str = today_str

start_date_str = (pd.to_datetime(date_str) - relativedelta(days=days_before)).strftime('%Y-%m-%d')
end_date_str = date_str
dates_str_list = pd.date_range(start=start_date_str, end=end_date_str).strftime('%Y-%m-%d').tolist()
print(dates_str_list)
yesterday_str = (pd.to_datetime(date_str) - relativedelta(days=1)).strftime('%Y-%m-%d')
last_week_str = (pd.to_datetime(date_str) - relativedelta(days=7)).strftime('%Y-%m-%d')

['2026-03-26', '2026-03-27', '2026-03-28', '2026-03-29', '2026-03-30', '2026-03-31', '2026-04-01', '2026-04-02']


# Input files

## Drive connection

In [6]:
#Connect to google drive

SCOPES = ["https://www.googleapis.com/auth/drive.metadata.readonly", "https://www.googleapis.com/auth/drive.readonly",
          "https://www.googleapis.com/auth/drive",'https://www.googleapis.com/auth/spreadsheets']

file_name = "google_credentials.json"
folder_name = "credentials"

# credentials_file = os.path.join('..', folder_name, file_name)
credentials_file = gAPI.create_path_cred()

service, service_sheets = gAPI.connect_drive(SCOPES,credentials_file)

## Buybox

In [7]:
sqlQuery = f"""SELECT *
FROM pricing_tests.walmart_item_report WHERE date = (SELECT MAX(date) FROM pricing_tests.walmart_item_report WHERE date <= '{date_str}') 
    AND "1p_offer_status" = 'PUBLISHED'"""

dfWmtItemReport = dw.runQuery(sqlQuery, newCredentials=True)
dfWmtItemReport.rename(columns={"item_id": "Item ID", "product_code": "Product Code"}, inplace=True)
dfWmtItemReport

,walmart_item_number,Item ID,gtin,product_code_wmt,unit_cost,customer_review_count,content_quality_score,1p_offer_status,walmart_buybox_winner,offer_price,walmart_dot_com_price,date,Product Code,unpublished_reason,average_customer_rating
0,664909762,936350496,08903094004539,BKTA058147B,112.26,6.0,0.953261,PUBLISHED,Yes,126.04,126.04,2026-04-02,BKTA058147B,None,4.8
1,664569488,986452845,08903094004744,BKTA0582295C,245.86,26.0,0.957703,PUBLISHED,No,313.68,194.61,2026-04-02,BKTA0582295C,None,4.8
2,664568742,466049908,08903094004621,BKTA0612095C,249.17,24.0,0.954659,PUBLISHED,Yes,250.09,250.09,2026-04-02,BKTA0612095C,None,4.6
3,677623084,15215216560,00796519037872,BKTA0612095D,261.77,8.0,0.959000,PUBLISHED,Yes,267.09,NaN,2026-04-02,BKTA0612095D,None,4.9
4,662185386,463471529,08903094003044,BKTA06124112D,218.46,14.0,0.839353,PUBLISHED,No,237.46,393.25,2026-04-02,BKTA06124112D,None,4.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73607,683897392,750171695,00636313208804,ZETA0411826560V~wm,92.61,NaN,0.889000,PUBLISHED,Yes,107.07,107.07,2026-04-02,ZETA0411826560V,None,NaN
73608,676036415,2327086177,00841623102685,ZETA0411826565H,96.27,7.0,0.973300,PUBLISHED,Yes,117.06,117.06,2026-04-02,ZETA0411826565H,None,4.9
73609,679253337,5262965474,00641126577238,ZETA0411826565H~wm,96.27,NaN,0.973300,PUBLISHED,Yes,117.13,117.13,2026-04-02,ZETA0411826565H,None,NaN
73610,676036416,3188494995,00841623102692,ZETA0411827565H,101.21,8.0,0.973300,PUBLISHED,Yes,123.94,123.94,2026-04-02,ZETA0411827565H,None,4.9


In [8]:
dfAvgWmtPrice = dfWmtItemReport.groupby("Product Code").agg({"offer_price":"mean","unit_cost":"mean"}).reset_index()
dfAvgWmtPrice

,Product Code,offer_price,unit_cost
0,ACCE0041523570E,105.480000,92.27
1,ACCE0071523575E,110.090000,81.98
2,ACCE0131318560V,71.070000,62.35
3,ACCE0131419550H,66.810000,39.29
4,ACCE0131820535YXL,68.360000,61.79
...,...,...,...
54664,ZETA0411823565HXL,94.075000,80.74
54665,ZETA0411825555VXL,93.936667,79.57
54666,ZETA0411826560V,112.125000,92.61
54667,ZETA0411826565H,117.095000,96.27


In [9]:
# sqlQuery = f"""SELECT *
# FROM pricing_tests.walmart_item_report WHERE date = (SELECT MAX(date) FROM pricing_tests.walmart_item_report WHERE date <= '2026-03-20') 
#     AND "1p_offer_status" = 'PUBLISHED'"""

# dfWmtItemReportPrev = dw.runQuery(sqlQuery, newCredentials=True)
# dfWmtItemReportPrev.rename(columns={"item_id": "Item ID", "product_code": "Product Code"}, inplace=True)
# dfWmtItemReportPrev

In [10]:
# dfCheck = dfWmtItemReport.merge(dfWmtItemReportPrev[["product_code_wmt", "offer_price"]], on="product_code_wmt", how="left", suffixes=("", "_prev"))
# dfCheck["Brand code"] = dfCheck["Product Code"].str[:4]
# dfCheck["Offer price change"] = dfCheck["offer_price"] - dfCheck["offer_price_prev"]
# dfCheck["Offer price change %"] = round(dfCheck["Offer price change"] / dfCheck["offer_price_prev"], 3)
# dfCheck

In [11]:
# dfCheck[dfCheck["Brand code"]== "FIRE"]#["Offer price change %"].mean()

## MAP

In [12]:
queryMAP= f"""  SELECT map_prices.sku,
    map_prices.map
   FROM pricing_tests.map_prices
  WHERE map_prices.date = (SELECT MAX(date) FROM pricing_tests.map_prices WHERE date <= '{date_str}')"""

dfMAP = dw.runQuery(queryMAP, newCredentials = True)

dfMAP = dfMAP.rename(columns={'sku': 'Product Code', 'map': 'MAP'})

dfMAP

,Product Code,MAP
0,ATTU0531527560W,239.0
1,ATTU0531727540WXL,249.0
2,ATTU0531730545W,309.0
3,ATTU0531731535YXL,279.0
4,ATTU0531831540YXL,429.0
...,...,...
17054,NITT05217421350C,749.0
17055,NITT0671530950,239.0
17056,NITT0671532950,261.0
17057,NITT0671533950,278.0


## Rollbacks (to exclude from NLCs)

In [13]:
path = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_03 Rollbacks\WalmartB2B Rollbacks tracker.xlsx"

dfRollbacksAll = pd.read_excel(path)
dfRollbacks = dfRollbacksAll[dfRollbacksAll["End date"]> pd.to_datetime(today_str)].copy()
print(f"Number of rollbacks active as of today: {len(dfRollbacks['Product Code'].unique())}")
dfRollbacks

Number of rollbacks active as of today: 587


,Item ID,Brand,Product Code,Final cost used,Time,Start date,End date,Planned end date,Unit cost,Source,Walmart SKU,New RB 3-18,Cost for DSV
0,15674650903,Advanta,ADVN0611620565H,65.46,90 days,2026-01-30,2026-04-30,2026-04-30,65.46,Sole Source,ADVN0611620565H~wm,No,65.46
1,15691958892,Advanta,ADVN0611622560H,88.50,90 days,2026-01-30,2026-04-30,2026-04-30,88.50,Sole Source,ADVN0611622560H~wm,No,88.50
2,5317408156,Advanta,ADVN0611721545WXL,62.88,90 days,2026-01-30,2026-04-30,2026-04-30,62.88,Dual Source,ADVN0611721545WXL,No,62.88
3,15701902922,Advanta,ADVN0611721545WXL,76.61,90 days,2026-01-30,2026-04-30,2026-04-30,76.61,Sole Source,ADVN0611721545WXL~wm,No,76.61
4,8616201882,Advanta,ADVN0611723555VXL,73.60,90 days,2026-01-30,2026-04-30,2026-04-30,73.60,Sole Source,ADVN0611723555VXL~wm,No,73.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...
777,14888903045,Achilles,ACHI0361624575S,87.86,90 days,2026-01-30,2026-04-30,2026-04-30,87.86,Sole Source,ACHI0361624575S~wm,No,87.86
778,5178959122,Achilles,ACHI0361722560H,61.41,90 days,2026-01-30,2026-04-30,2026-04-30,61.41,Sole Source,ACHI0361722560H,No,61.41
779,5429311514,Achilles,ACHI0361823565H,75.59,90 days,2026-01-30,2026-04-30,2026-04-30,75.59,Sole Source,ACHI0361823565H,No,75.59
780,5439168730,Achilles,ACHI0361826570H,94.91,90 days,2026-01-30,2026-04-30,2026-04-30,94.91,Sole Source,ACHI0361826570H,No,94.91


In [14]:
# days_check = ["2026-03-13","2026-03-15"]
# days_check_date = [pd.to_datetime(day) for day in days_check]
# days_check_date
# dfRollbacksEnded = dfRollbacksAll[dfRollbacksAll["End date"].isin(days_check_date)].copy()
# dfRollbacksEnded

In [15]:
# dfRollbacksEndedSKUs = dfRollbacksEnded[["Product Code"]].drop_duplicates()
# dfRollbacksEndedSKUs = dfRollbacksEndedSKUs[~dfRollbacksEndedSKUs["Product Code"].isin(dfRollbacks["Product Code"])]
# dfRollbacksEndedSKUs

In [16]:
# dfNewDSV[(dfNewDSV["SKU"].isin(dfRollbacksEndedSKUs["Product Code"])) & (dfNewDSV["Source"].notna())]

In [17]:
dfRollbacksSKUPrice = dfRollbacks.groupby(["Product Code"]).agg({"Cost for DSV": "min"}).reset_index()
dfRollbacksSKUPrice = dfRollbacksSKUPrice.rename(columns={"Product Code": "SKU", "Cost for DSV": "RB price"})
dfRollbacksSKUPrice

,SKU,RB price
0,ACHI0031726570E,136.30
1,ACHI0361623585S,92.40
2,ACHI0361624575S,87.86
3,ACHI0361722560H,61.41
4,ACHI0361823565H,75.59
...,...,...
582,ZEET0251621560HXL,48.62
583,ZENN0012027555VXL,102.42
584,ZENN0012226535WXL,86.81
585,ZENN0012228545WXL,106.37


In [18]:
# dfRollbacksSKUPriceWithDSV = dfRollbacksSKUPrice.merge(dfCurrDSV[["SKU", "walmart_dsv_price"]], how="left", on="SKU")
# dfRollbacksSKUPriceWithDSV["Delta %"] = round((dfRollbacksSKUPriceWithDSV["Price"] - dfRollbacksSKUPriceWithDSV["walmart_dsv_price"]) / dfRollbacksSKUPriceWithDSV["walmart_dsv_price"],4)
# dfRollbacksSKUPriceWithDSV["Delta %"].value_counts()

## Wh node mapping

In [19]:
idFolderWhStatus = "1Y4drFI84j2eNQM_XTNb9nY9vBCngszZq"
dtype = {"Identifier": str}

dfWHnodeMappingAll = gAPI.get_last_df(service, idFolderWhStatus, sep = ";", dtype=dtype)
dfWHnodeMappingAll = dfWHnodeMappingAll[dfWHnodeMappingAll["Channel"]== "WalmartB2B"]

dfWHnodeMappingAll["Wh-Identifier"] = dfWHnodeMappingAll["Warehouse Code"] + "-" + dfWHnodeMappingAll["Identifier"]

dfWHnodeMappingAll["Overall status"] = np.where((dfWHnodeMappingAll["Warehouse Status"] == "ENABLED") &
                                                (dfWHnodeMappingAll["Identifier Status"] == "ENABLED") &
                                                (dfWHnodeMappingAll["Inventory Enabled"] ==1), "ENABLED", "DISABLED")

dfWHnodeMapping = dfWHnodeMappingAll[dfWHnodeMappingAll["Overall status"] == "ENABLED"].copy()
# keepCols = ["Warehouse Code","Identifier", "Type", "Inventory Threshold"]
# dfWHnodeMapping = dfWHnodeMapping[keepCols]
dfWHnodeMapping = dfWHnodeMapping.rename(columns={"Type": "Node type", "Inventory Threshold": "Zero Out Threshold"})

dfWHnodeMapping

Last modified time 2026-04-02T10:00:21.009Z
Download 100.


,Channel,Warehouse Code,Warehouse Status,Identifier Status,Source,Identifier,Node type,Inventory Enabled,Inventory Type,Zero Out Threshold,Wh-Identifier,Overall status
0,WalmartB2B,5019,ENABLED,ENABLED,NaN,62832654,HUB,1,SUM,7,5019-62832654,ENABLED
7,WalmartB2B,502,ENABLED,ENABLED,NaN,62832637,HUB,1,SUM,4,502-62832637,ENABLED
37,WalmartB2B,5022,ENABLED,ENABLED,NaN,628326670,HUB,1,SUM,4,5022-628326670,ENABLED
71,WalmartB2B,5024,ENABLED,ENABLED,NaN,628326514,HUB,1,SUM,7,5024-628326514,ENABLED
77,WalmartB2B,5024,ENABLED,ENABLED,NaN,6283261190,HUB,1,SUM,7,5024-6283261190,ENABLED
...,...,...,...,...,...,...,...,...,...,...,...,...
23272,WalmartB2B,5128-Vans,ENABLED,ENABLED,NaN,628326686,SPOKE,1,AGGREGATE,4,5128-Vans-628326686,ENABLED
23274,WalmartB2B,5520-Vans,ENABLED,ENABLED,NaN,628326685,SPOKE,1,AGGREGATE,4,5520-Vans-628326685,ENABLED
23275,WalmartB2B,5520-Vans,ENABLED,ENABLED,NaN,6283261052,SPOKE,1,AGGREGATE,4,5520-Vans-6283261052,ENABLED
23277,WalmartB2B,5983-Vans,ENABLED,ENABLED,NaN,628326947,SPOKE,1,AGGREGATE,4,5983-Vans-628326947,ENABLED


In [20]:
# idFolderWhStatus = "1GTnjzGzF2yEypw1CnN7Qwk5r8Ay5zqHl"  #3-30

# dtype = {"Identifier": str}

# dfWHnodeMappingAllPrev = gAPI.get_df_of_file(service, idFolderWhStatus,"csv", sep = ";", dtype=dtype)
# dfWHnodeMappingAllPrev = dfWHnodeMappingAllPrev[dfWHnodeMappingAllPrev["Channel"]== "WalmartB2B"]

# dfWHnodeMappingAllPrev["Wh-Identifier"] = dfWHnodeMappingAllPrev["Warehouse Code"] + "-" + dfWHnodeMappingAllPrev["Identifier"]
# dfWHnodeMappingAllPrev = dfWHnodeMappingAllPrev.rename(columns={"Type": "Node type", "Inventory Threshold": "Zero Out Threshold"})

# dfWHnodeMappingAllPrev["Overall status"] = np.where((dfWHnodeMappingAllPrev["Warehouse Status"] == "ENABLED") &
#                                                 (dfWHnodeMappingAllPrev["Identifier Status"] == "ENABLED") &
#                                                 (dfWHnodeMappingAllPrev["Inventory Enabled"] ==1), "ENABLED", "DISABLED")

# dfWHnodeMappingPrev = dfWHnodeMappingAllPrev[dfWHnodeMappingAllPrev["Overall status"] == "ENABLED"].copy()
# # keepCols = ["Warehouse Code","Identifier", "Type", "Inventory Threshold"]
# # dfWHnodeMappingPrev = dfWHnodeMappingPrev[keepCols]
# dfWHnodeMappingPrev = dfWHnodeMappingPrev.rename(columns={"Type": "Node type", "Inventory Threshold": "Zero Out Threshold"})
# dfWHnodeMappingPrev["Wh-Identifier"] = dfWHnodeMappingPrev["Warehouse Code"] + "-" + dfWHnodeMappingPrev["Identifier"]

# dfWHnodeMappingPrev

In [21]:
# dfWHnodeMappingAllComp =  dfWHnodeMappingAll.merge(dfWHnodeMappingAllPrev, how = "outer", on="Wh-Identifier", suffixes=("", "_Prev"), indicator=True).merge(dfVendorRef, on="Warehouse Code", how="left")
# dfWHnodeMappingAllComp["Overall status_Prev"] = dfWHnodeMappingAllComp["Overall status_Prev"].fillna("Not found")
# dfWHnodeMappingAllComp["Status change"] = "From " + dfWHnodeMappingAllComp["Overall status_Prev"] + " to " + dfWHnodeMappingAllComp["Overall status"]
# dfWHnodeMappingAllComp["Status change"].value_counts()

In [22]:
# #add a column to wh maping all comp to check if the column Warehouse code contiains any of these values: 5458, 5459, 5460, 5461, 5466, 6052, 5467,5097
# # 5456
# # 5457
# # 5462
# # 5463
# dfWHnodeMappingAllComp["WH code flag"] = np.where(dfWHnodeMappingAllComp["Warehouse Code"].astype(str).str.contains("5458|5459|5460|5461|5466|6052|5467|5097|5456|5457|5462|5463"), "In whs list", "Not in whs list")
# dfWHnodeMappingAllComp["WH code flag"].value_counts()

In [23]:
# dfWHnodeEnabltoDisabl = dfWHnodeMappingAllComp[dfWHnodeMappingAllComp["Status change"] == "From ENABLED to DISABLED"].copy()
# display(dfWHnodeEnabltoDisabl)
# display(dfWHnodeEnabltoDisabl["Vendor Group"].value_counts())
# display(dfWHnodeEnabltoDisabl["WH code flag"].value_counts())

# new_enabled_list = ["From Not found to ENABLED", "From DISABLED to ENABLED"]
# dfWHnodeNewEnabled = dfWHnodeMappingAllComp[dfWHnodeMappingAllComp["Status change"].isin(new_enabled_list)].copy()
# display(dfWHnodeNewEnabled["Vendor Group"].value_counts())
# display(dfWHnodeNewEnabled["WH code flag"].value_counts())

# #put in excel to 2 tabs
# with pd.ExcelWriter("WH-identifier combinations comparison.xlsx") as writer:
#     dfWHnodeEnabltoDisabl.to_excel(writer, sheet_name="Enabled to disabled", index=False)
#     dfWHnodeNewEnabled.to_excel(writer, sheet_name="Newly enabled", index=False)

In [24]:
# dfWHnodeMappingAllComp.to_excel("WH-identifier combinations comparison.xlsx", index=False)

In [25]:
# dfVendor = dw.get_vendor_code_table()
# dfVendor = dfVendor.rename(columns={"warehouse_code": "Warehouse Code"})
# dfVendorRef = dfVendor[["Warehouse Code", "Vendor Group", "vendor_code","warehouse_name"]].copy()

# # dfVendor = dfVendor[["Warehouse Code", "vendor_code"]].copy()
# dfVendorRef

In [26]:
# dfWHnodeMappingNew = dfWHnodeMapping[dfWHnodeMapping["Wh-Identifier"].isin(dfWHnodeMappingPrev["Wh-Identifier"]) == False].copy()
# dfWHnodeMappingNew = dfWHnodeMappingNew.merge(dfVendorRef, on="Warehouse Code", how="left")
# # dfWHnodeMappingNew.to_excel("New active WH-identifier combinations.xlsx", index=False)
# dfWHnodeMappingWithPrevStatus = dfWHnodeMapping.merge(dfWHnodeMappingAllPrev, how = "left", on="Wh-Identifier", suffixes=("", "_Prev")).merge(dfVendorRef, on="Warehouse Code", how="left")
# dfWHnodeMappingWithPrevStatus["Was fully enabled before?"] = np.where(dfWHnodeMappingWithPrevStatus["Wh-Identifier"].isin(dfWHnodeMappingPrev["Wh-Identifier"]), "Yes", "No")
# dfWHnodeMappingWithPrevStatus["Was fully enabled before?"].value_counts() 
# # dfWHnodeMappingWithPrevStatus.to_excel("WH-identifier combinations with previous status.xlsx", index=False)
# dfWHnodeMappingNew["Vendor Group"].value_counts()


In [27]:
# dfWHnodeMappingOld = dfWHnodeMappingPrev[dfWHnodeMappingPrev["Wh-Identifier"].isin(dfWHnodeMapping["Wh-Identifier"]) == False].copy()
# dfWHnodeMappingOld = dfWHnodeMappingOld.merge(dfVendorRef, on="Warehouse Code", how="left")

# dfWHnodeMappingOld["Vendor Group"].value_counts()

In [28]:
# OLD:
# # id = "1uQKncC10D0cVgLOXWQcIxNnogNrIKzKZ" #10-09-2025 file
# # id = "14Ahcn-M7G6fBWJH4PbbIN3tOiOnTCxyT" #10-30-2025 file 
# # id = "15zrneTcAiUJJv7YJO-_gqia8QBGHqPd6" #11-13-2025 file
# # id = "1MtBERWyq3kvcQfEZyiQKmzXAm0U8y0Cs" #11-24-2025 file 
# # id = "1ELvSFQ-pqhb7j70BQJLuar6N68Cn_6ya" #12-04-2025 file 
# # id = "1NOh38WnyG-sFtG_EGX52UCWzL7dB3I-z" #12-11-2025 file
# # id = "1SoKqC_odyUj6t4NwjAjAh-31UQsNG7lv" #12-30-2025 file
# # id = "1LUQS3droPFVV56kmn7HIpth-pNHDIg9I" #1-14-2026 file
# id = "1fxZOdTyygUkLipYQO7ty3xJks0hvnmIn" #1-19-2026 file including SPOKES



# usecols = ["Warehouse", "Identifier", "Inventory Enabled","Inventory Type","Zero Out Threshold","Type"]
# dtype = {"Identifier": str}
# dfWHnodeMappingAll = gAPI.get_df_of_file(service, id, "csv", sep = ";", dtype=dtype, read_cols=usecols)

# dfWHnodeMapping = dfWHnodeMappingAll[dfWHnodeMappingAll["Inventory Enabled"] == True].copy()
# dfWHnodeMapping =dfWHnodeMapping.drop(columns=["Inventory Enabled","Inventory Type"]).copy()

# dfWHnodeMapping["Warehouse"] = dfWHnodeMapping["Warehouse"].str.split(" [", n=1, regex=False).str[0]
# dfWHnodeMapping.rename(columns={"Warehouse": "Warehouse Code", "Type": "Node type"}, inplace=True)
# dfWHnodeMapping

## Shipping costs

In [29]:
path = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Config files\Costs to add by node.xlsx"
dfCostNode = pd.read_excel(path, dtype = {"Identifier": str}, usecols=["Identifier", "Shipping cost"])
dfCostNode

,Identifier,Shipping cost
0,628326761,10
1,6283261135,7
2,6283261137,7
3,6283261207,7
4,6283261132,7
5,6283261134,7
6,6283261147,7
7,6283261109,7
8,6283261235,7


In [30]:
dfCostNodeCheck = dfCostNode[dfCostNode["Identifier"]!= "628326761"]

## Inventory

In [31]:
def process_inventory_nlc(dfInventoryRelevant, metricGroup = "min", min_units = 4):

    dfInventoryRelevant["Brand code"] = dfInventoryRelevant["Product Code"].str[:4]

    dfInventoryRelevant = dfInventoryRelevant[dfInventoryRelevant["Available"] >= min_units].copy()

    dfInventoryRelevantFiltered = dfInventoryRelevant[dfInventoryRelevant["Product Code"].isin(dfCurrDSV["SKU"])].copy()

    dfInventoryRelevantFilteredWithNodes = dfInventoryRelevantFiltered.merge(dfWHnodeMapping, how="left", on="Warehouse Code")
    dfInvAfterThreshold= dfInventoryRelevantFilteredWithNodes[dfInventoryRelevantFilteredWithNodes["Available"] >= dfInventoryRelevantFilteredWithNodes["Zero Out Threshold"]].copy()
    dfInvAfterThreshold = dfInvAfterThreshold[dfInvAfterThreshold["Node type"].notna()].copy()
    dfInvMinPrices = dfInvAfterThreshold.groupby(['Product Code', 'Identifier',"date"]).agg({'Purchase Price+FET':metricGroup})
    dfInvNLC = dfInvAfterThreshold.merge(dfInvMinPrices, how="inner", on=['Product Code', 'Identifier', 'Purchase Price+FET',"date"])
    dfInvNLC = dfInvNLC.drop_duplicates(subset=["Product Code", "Identifier", "Purchase Price+FET","date"], keep="first")
    dfInvNLC = dfInvNLC[["Product Code", "Identifier", "Warehouse Code", "Available", "Purchase Price+FET","date","Node type"]].copy()
    
    dfInvFinal = (
        dfInvNLC.sort_values('date', ascending=False)
        .groupby(['Product Code', 'Identifier'], as_index=False)
        .first()
    )

    return dfInvFinal

In [32]:
dfInvAll = pd.DataFrame()
for date_i in dates_str_list:
    dfInvDate = pricing.get_inventory(date_i, add_rebates = False, amazon = False, greater3=True)
    dfInvDate["date"] = pd.to_datetime(date_i)
    print(f"Date {date_i} done, rows retrieved: {len(dfInvDate)}")

    dfInvAll = pd.concat([dfInvAll, dfInvDate], ignore_index=True)

dfInvAll = dfInvAll[~dfInvAll["Product Code"].isin(dfRollbacks["Product Code"])]

dfInvAll

Download 100.
Last modified time 2026-03-26T10:01:17.993Z
Download 96.
Download 100.
Date 2026-03-26 done, rows retrieved: 2179024
Download 100.
Last modified time 2026-03-27T10:01:28.905Z
Download 95.
Download 100.
Date 2026-03-27 done, rows retrieved: 2207614
Download 100.
Last modified time 2026-03-28T10:01:35.059Z
Download 95.
Download 100.
Date 2026-03-28 done, rows retrieved: 2216883
Download 100.
Last modified time 2026-03-29T10:01:35.053Z
Download 97.
Download 100.
Date 2026-03-29 done, rows retrieved: 2158571
Download 100.
Last modified time 2026-03-30T10:01:28.916Z
Download 95.
Download 100.
Date 2026-03-30 done, rows retrieved: 2206758
Download 100.
Last modified time 2026-03-31T10:01:00.426Z
Download 96.
Download 100.
Date 2026-03-31 done, rows retrieved: 2200307
Download 100.
Last modified time 2026-04-01T10:01:26.154Z
Download 96.
Download 100.
Date 2026-04-01 done, rows retrieved: 2192248
Download 100.
Last modified time 2026-04-02T10:01:15.756Z
Download 97.
Download 100

,Product Code,Warehouse Code,Purchase Price,Available,FET,Purchase Price+FET,name,State_Code,Postal code,Zone,closed,walmartb2b_enabled,walmartb2b_threshold,Can show inv?,date
0,ACCE0131820535YXL,5614,56.95,87,0.0,56.95,Quality Tire Distributors,US-FL,33610,4,No,Yes,4.0,True,2026-03-26
1,ACCE0221316580T,5232,33.35,16,0.0,33.35,Hesselbein Tire - San Antonio,US-TX,78219,3,No,Yes,4.0,True,2026-03-26
2,ACCE0221316580T,5233,33.35,16,0.0,33.35,Hesselbein Tire Southwest - Mission,US-TX,78572,3,No,Yes,4.0,True,2026-03-26
3,ACCE0221316580T,5234,33.35,8,0.0,33.35,Hesselbein Tire - Houston,US-TX,77040,3,No,Yes,4.0,True,2026-03-26
4,ACCE0221316580T,5235,33.35,16,0.0,33.35,Hesselbein Tire Southwest - Corpus Christi,US-TX,78401,3,No,Yes,4.0,True,2026-03-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17537451,ZETA0411827565H,549,89.95,63,0.0,89.95,"Economy Tire, Inc - Mesquite, TX",US-TX,75126,3,No,Yes,4.0,True,2026-04-02
17537452,ZETA0411827565H,568,90.86,40,0.0,90.86,Freedom Tire Distributing,US-CA,92882,2,No,Yes,7.0,True,2026-04-02
17537453,ZETA0411827565H,8186,99.00,26,0.0,99.00,"Tires Depot Inc Memphis, TN",US-TN,38134,8,No,Yes,4.0,True,2026-04-02
17537454,ZETA0411827565H,5570,99.53,24,0.0,99.53,"GTW Tires Tampa, Fl",US-FL,33605,4,No,Yes,4.0,True,2026-04-02


## Current DSV

In [33]:
compare_dsvs = False

In [34]:
def read_dsv(read_date = None):
    folderWmtDSV = "1piuawZRpppmoD-Qdkd1IUj3x4rs-LKny"
    readCols = ["SKU", "Price", "Source"]
    dtype = {"SKU": str, "Price": float,"Source": str}
    dfWmatDSVFiles = gAPI.get_all_files_folder(service, folderWmtDSV)

    dfWmatDSVFiles["Date"] = dfWmatDSVFiles["Name"].apply(lambda x: re.search(r"\d{4}-\d{2}-\d{2}", x).group())
    dfWmatDSVFiles["Date"] = pd.to_datetime(dfWmatDSVFiles["Date"], format = "%Y-%m-%d")

    dfWmatDSVFiles = dfWmatDSVFiles[(dfWmatDSVFiles["Date"] >= '2025-12-30')]

    if read_date == None:
        read_date = max(dfWmatDSVFiles["Date"])
        print(f"Max DSV date: {read_date}")

    dfWmatDSVFiles = dfWmatDSVFiles[dfWmatDSVFiles["Date"] == read_date]

    idWmtFile = dfWmatDSVFiles["ID"].iloc[0]

    dfCurrDSVOriginal = gAPI.get_df_of_file(service, idWmtFile, "csv", read_cols = readCols, dtype = dtype)

    #Drop duplicates
    dfCurrDSVOriginal  = dfCurrDSVOriginal.drop_duplicates(subset=["SKU", "Source"], keep="first")

    dfCurrDSVOriginal = dfCurrDSVOriginal.rename(columns={"SKU": "sku", "Price": "walmart_dsv_price","Source":"source"})


    # queryCurrDSV = f"""SELECT *
    # FROM pricing_tests.walmart_dsv WHERE date = (SELECT MAX(date) FROM pricing_tests.walmart_dsv WHERE date <= '{date_str}')"""
    # dfCurrDSVOriginal = dw.runQuery(queryCurrDSV, newCredentials=True)

    dfCurrDSVNLC = dfCurrDSVOriginal[dfCurrDSVOriginal['source'].notna()].copy()
    dfCurrDSVNLC = dfCurrDSVNLC.rename(columns={"sku": "Product Code", "source": "Identifier","walmart_dsv_price":"current_nlc_price"})#.drop(columns=["date"])

    dfCurrDSV = dfCurrDSVOriginal[dfCurrDSVOriginal['source'].isna()].copy()

    # dfCurrDSV = dfCurrDSV.drop_duplicates(subset=["SKU", "Vendor Code"], keep="last")
    dfCurrDSV = dfCurrDSV.rename(columns={"sku": "SKU"})
    return dfCurrDSVOriginal, dfCurrDSVNLC, dfCurrDSV

In [35]:
def dsv_changes(dfCurrDSVOriginal, dfPrevDSVOriginal):

    dfCurrDSVOriginalCheck = dfCurrDSVOriginal.copy()
    dfCurrDSVOriginalCheck["source"] =  dfCurrDSVOriginalCheck["source"].fillna("National")
    dfCurrDSVOriginalCheck["SKU-Node"] = dfCurrDSVOriginalCheck["sku"] + "-" + dfCurrDSVOriginalCheck["source"]

    dfPrevDSVOriginalCheck = dfPrevDSVOriginal.copy()
    dfPrevDSVOriginalCheck["source"] =  dfPrevDSVOriginalCheck["source"].fillna("National")
    dfPrevDSVOriginalCheck["SKU-Node"] = dfPrevDSVOriginalCheck["sku"] + "-" + dfPrevDSVOriginalCheck["source"]

    dfDSVComp = dfCurrDSVOriginalCheck.merge(dfPrevDSVOriginalCheck, how="left", on=["SKU-Node"], suffixes=("_current", "_previous"))
    dfDSVComp["Price change %"] = (dfDSVComp["walmart_dsv_price_current"] - dfDSVComp["walmart_dsv_price_previous"]) / dfDSVComp["walmart_dsv_price_previous"]

    dfDSVComp["Price change % category"] = np.where(dfDSVComp["Price change %"] >0,"Increase",
                                            np.where(dfDSVComp["Price change %"] <0,"Decrease", np.where(dfDSVComp["Price change %"] == 0, "No change", "New")))

    dfDSVComp["Price change % category"].value_counts()

    #drop sku_previous and source_previous + rename sku_current to sku and source_current to source
    dfDSVComp = dfDSVComp.drop(columns=["sku_previous", "source_previous"]).rename(columns={"sku_current": "sku", "source_current": "source"})

    # dfDSVChanges = dfDSVComp[dfDSVComp["Price change % category"] != "No change"].copy()
    # dfDSVChanges = dfDSVChanges[["SKU-Node", "walmart_dsv_price_current","Price change %", "Price change % category","sku", "source"]]
    return dfDSVComp

In [36]:
dfCurrDSVOriginal, dfCurrDSVNLC, dfCurrDSV = read_dsv()
dfCurrDSVOriginal

Max DSV date: 2026-03-31 00:00:00
Download 86.
Download 100.


,sku,walmart_dsv_price,source
0,IRON0081417565H,40.11,628326503
1,IRON0061417565H,40.11,628326106
2,IRON0061417565H,40.11,6283261141
3,APLS0031518555V,40.11,628326916
4,IRON0061417565H,40.11,6283261227
...,...,...,...
3296321,MATR0011520575C2,53.99,NaN
3296322,MATR0011522575D2,69.39,NaN
3296323,MATR0011623585E2,80.09,NaN
3296324,RADA0101824545WXL,79.43,NaN


In [37]:
if compare_dsvs:
    dfPrevDSVOriginal1, dfPrevDSVNLC1, dfPrevDSV1 = read_dsv('2026-03-05')
    dfPrevDSVOriginal, dfPrevDSVNLC, dfPrevDSV = read_dsv('2026-03-06')

    dfDSVChanges = dsv_changes(dfPrevDSVOriginal, dfPrevDSVOriginal1)


## Sales

### Get data

In [38]:
days_sales = 90
today_minus_x = (pd.to_datetime(date_str) - relativedelta(days=days_sales)).strftime('%Y-%m-%d')

print("start date", today_minus_x)

query = f"""SELECT order_id, sku, externalwarehouseid, order_date, supplier_id,
       quantity, total_inv_amount, profit, external_price, order_type
FROM warehouse.vw_virtual_node_tracker
WHERE shop = 'WalmartB2B'
  AND order_type IN ('Sale')
  AND orderstatus NOT IN ('WILL_CALL_NOT_PAID', 'CANCELLED_BEFORE_SHIPPING', 'ERROR')
  AND order_date >= '{today_minus_x}'
  AND order_date < CURRENT_DATE;
"""

dfSalesSKUNode = dw.runQuery(query, newCredentials=False)
dfSalesSKUNode["SKU-Node"] = dfSalesSKUNode["sku"] + "-" + dfSalesSKUNode["externalwarehouseid"].astype(str)
dfSalesSKUNode["order_date"] = pd.to_datetime(dfSalesSKUNode["order_date"])


dfSales = dfSalesSKUNode.groupby(["order_date", "sku"]).agg({"quantity":"sum", "total_inv_amount":"sum", "profit":"sum"}).reset_index()
dfSales = dfSales.rename(columns={ "total_inv_amount": "revenue", "sku": "Product Code"})
dfSKURevenue = dfSales.copy().groupby("Product Code").agg({"revenue": "sum"}).reset_index()
dfSKURevenue = dfSKURevenue.rename(columns={"revenue": "sku_revenue"})


dfSKUNodeRevenue = dfSalesSKUNode.groupby(["SKU-Node"]).agg({"total_inv_amount":"sum"}).reset_index()
total_revenue = dfSKUNodeRevenue["total_inv_amount"].sum()
print(f"Total revenue from sales with node-level costs in the last 90 days: {total_revenue}")
dfSKUNodeRevenue = dfSKUNodeRevenue.rename(columns={"total_inv_amount": "sku_node_revenue"})
dfSKUNodeRevenue

start date 2026-01-02
Total revenue from sales with node-level costs in the last 90 days: 91144591.39999999


,SKU-Node,sku_node_revenue
0,ACCE0131318560V-6283261264,141.52
1,ACCE0131418555V-6283261117,252.25
2,ACCE0131418555V-628326864,100.90
3,ACCE0131419550H-6283261117,196.00
4,ACCE0131419550H-628326433,222.24
...,...,...
117788,ZETA0411823560VXL-6283261138,350.56
117789,ZETA0411823560VXL-6283261395,175.28
117790,ZETA0411823560VXL-628326985,178.94
117791,ZETA0411823565HXL-6283261094,93.26


In [39]:
# How many Product Code are in the dfSales (use dfSales['Product Code'].nunique()) over the last 14 days, 30 days, 90 days and 150 days?
days = [days_sales]
for day in days:
    date_threshold = pd.to_datetime(date_str) - timedelta(days=day)
    count_product_codes = dfSales[dfSales['order_date'] >= date_threshold]['Product Code'].nunique()
    print(f"Number of unique Product Codes in the last {day} days: {count_product_codes}")

#Use the above to calculate how many skus represent 20% of sales, 50% of sales and 80% of sales for the same days in the list
for day in days:
    date_threshold = pd.to_datetime(date_str) - timedelta(days=day)
    dfSalesFiltered = dfSales[dfSales['order_date'] >= date_threshold].copy()
    dfSalesFiltered = dfSalesFiltered.groupby("Product Code").agg({"revenue": "sum"}).reset_index()
    dfSalesFiltered = dfSalesFiltered.sort_values("revenue", ascending=False).reset_index(drop=True)
    dfSalesFiltered["cumulative_revenue"] = dfSalesFiltered["revenue"].cumsum()
    total_revenue_filt = dfSalesFiltered["revenue"].sum()
    dfSalesFiltered["cumulative_revenue_pct"] = dfSalesFiltered["cumulative_revenue"] / total_revenue_filt
    skus_20_pct = (dfSalesFiltered[dfSalesFiltered["cumulative_revenue_pct"] <= 0.20].shape[0])
    skus_50_pct = (dfSalesFiltered[dfSalesFiltered["cumulative_revenue_pct"] <= 0.50].shape[0])
    skus_80_pct = (dfSalesFiltered[dfSalesFiltered["cumulative_revenue_pct"] <= 0.80].shape[0])
    
    print(f"In the last {day} days, {skus_20_pct} SKUs represent 20% of sales, {skus_50_pct} SKUs represent 50% of sales, and {skus_80_pct} SKUs represent 80% of sales.")
    #Also create a column "Sales category" that is top 20%, top 50% top 80% and the rest, and count how many skus are in each category.
    dfSalesFiltered["sku_sales_category"] = pd.cut(dfSalesFiltered["cumulative_revenue_pct"], bins=[0, 0.05,0.1,0.2,0.3,0.4, 0.50,0.6,0.7,0.8,0.9,0.95,1], labels=["0-5%","5-10%","10-20%","20-30%","30-40%","40-50%","50-60%","60-70%","70-80%","80-90%","90-95%","95-100%"])
    sales_category_counts = dfSalesFiltered["sku_sales_category"].value_counts()
    print(f"Sales category counts in the last {day} days:\n{sales_category_counts}\n")

dfSKURevenue = dfSKURevenue.merge(dfSalesFiltered[["Product Code", "sku_sales_category"]], on="Product Code", how="left")
dfSKURevenue

Number of unique Product Codes in the last 90 days: 23116
In the last 90 days, 117 SKUs represent 20% of sales, 895 SKUs represent 50% of sales, and 4642 SKUs represent 80% of sales.
Sales category counts in the last 90 days:
sku_sales_category
95-100%    10819
80-90%      3868
90-95%      3787
70-80%      1956
60-70%      1124
50-60%       667
40-50%       400
30-40%       236
20-30%       142
10-20%        80
5-10%         23
0-5%          14
Name: count, dtype: int64



,Product Code,sku_revenue,sku_sales_category
0,ACCE0131318560V,141.52,95-100%
1,ACCE0131418555V,353.15,95-100%
2,ACCE0131419550H,418.24,95-100%
3,ACCE0131820535YXL,137.98,95-100%
4,ACCE0221316580T,309.02,95-100%
...,...,...,...
23111,ZETA0411726570S,2277.89,80-90%
23112,ZETA0411823555V,834.04,95-100%
23113,ZETA0411823560VXL,704.78,95-100%
23114,ZETA0411823565HXL,93.26,95-100%


### Stats less sales

In [40]:
# df = dfSales.copy()
# df['order_date'] = pd.to_datetime(df['order_date'])

# # --- anchors (same logic as before) ---
# as_of = df['order_date'].max().normalize()

# l2w_start = as_of - pd.Timedelta(days=13)   # 14 days inclusive
# p90_end   = l2w_start - pd.Timedelta(days=1)
# p90_start = p90_end - pd.Timedelta(days=89) # 90 days inclusive

# # --- slice periods ---
# l2w = df[df['order_date'].between(l2w_start, as_of)]
# p90 = df[df['order_date'].between(p90_start, p90_end)]

# # --- global totals ---
# l2w_qty = l2w['quantity'].sum()
# p90_qty = p90['quantity'].sum()

# l2w_rev = l2w['revenue'].sum()
# p90_rev = p90['revenue'].sum()

# l2w_prof = l2w['profit'].sum()
# p90_prof = p90['profit'].sum()

# # --- global metrics (fixed days) ---
# global_avg_daily_qty_L2W = l2w_qty / 14
# global_avg_daily_qty_P90 = p90_qty / 90

# global_margin_L2W = l2w_prof / l2w_rev
# global_margin_P90 = p90_prof / p90_rev

# # --- comparisons ---
# global_qty_pct_change_L2W_vs_P90 = global_avg_daily_qty_L2W / global_avg_daily_qty_P90 - 1
# global_margin_delta_L2W_vs_P90 = global_margin_L2W - global_margin_P90

# # --- nice output dataframe ---
# overall = pd.DataFrame([{
#     'as_of': as_of.date(),
#     'L2W_start': l2w_start.date(),
#     'P90_start': p90_start.date(),
#     'P90_end': p90_end.date(),
#     'avg_daily_qty_L2W': global_avg_daily_qty_L2W,
#     'avg_daily_qty_P90': global_avg_daily_qty_P90,
#     'qty_pct_change_L2W_vs_P90': global_qty_pct_change_L2W_vs_P90,
#     'margin_L2W': global_margin_L2W,
#     'margin_P90': global_margin_P90,
#     'margin_delta_L2W_vs_P90': global_margin_delta_L2W_vs_P90
# }])

# # round
# num_cols = overall.select_dtypes(include='number').columns
# overall[num_cols] = overall[num_cols].round(4)

# overall


In [41]:
# rb = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_03 Rollbacks\All RBs tracker.xlsx"
# dfRb = pd.read_excel(rb, usecols = ["Product Code", "Start date", "End date"]).rename(columns = {"Start date":"Rb start date", "End date":"Rb end date"})
# # dfRb["Ended RB"] = np.where(dfRb["Rb end date"] < pd.to_datetime('today'), "Yes", "No")
# # dfRb["Active RB"] = np.where(dfRb["Rb end date"] >= pd.to_datetime('today'), "Yes", "No")

# # dfRb["RB Status"] = np.where(dfRb["Ended RB"] == "Yes", "Ended", np.where(dfRb["Active RB"] == "Yes", "Active", "Unknown"))

# # dfRb = dfRb.drop(columns=["Rb start date", "Rb end date", "Ended RB", "Active RB"])

# dfRb

In [42]:
# threshold = -0.0
# totalTreshold = threshold + global_qty_pct_change_L2W_vs_P90

# marginTreshold = 0.08

In [43]:
# df = dfSales.copy()
# df['order_date'] = pd.to_datetime(df['order_date'])

# # -----------------------
# # 1) L2W vs P90 (fixed days)
# # -----------------------
# as_of = df['order_date'].max().normalize()

# l2w_start = as_of - pd.Timedelta(days=13)   # 14 days inclusive
# p90_end   = l2w_start - pd.Timedelta(days=1)
# p90_start = p90_end - pd.Timedelta(days=89) # 90 days inclusive

# df['period'] = np.select(
#     [
#         df['order_date'].between(l2w_start, as_of),
#         df['order_date'].between(p90_start, p90_end)
#     ],
#     ['L2W', 'P90'],
#     default=None
# )

# agg = (
#     df[df['period'].notna()]
#       .groupby(['Product Code', 'period'], as_index=False)
#       .agg(qty=('quantity', 'sum'),
#            rev=('revenue', 'sum'),
#            prof=('profit', 'sum'))
# )

# agg['days'] = agg['period'].map({'L2W': 14, 'P90': 90}).astype(int)
# agg['avg_daily_qty'] = agg['qty'] / agg['days']
# agg['margin'] = agg['prof'] / agg['rev']
# agg = agg.drop(columns=['qty', 'rev', 'prof', 'days'])

# wide = agg.pivot(index='Product Code', columns='period', values=['avg_daily_qty', 'margin'])
# wide.columns = [f'{m}_{p}' for (m, p) in wide.columns]
# wide = wide.reset_index()

# wide['qty_pct_change_L2W_vs_P90'] = wide['avg_daily_qty_L2W'] / wide['avg_daily_qty_P90'] - 1
# wide["qty_abs_change_L2W_vs_P90"] = wide['avg_daily_qty_L2W'] - wide['avg_daily_qty_P90']

# wide['margin_delta_L2W_vs_P90'] = wide['margin_L2W'] - wide['margin_P90']

# wide = wide.merge(dfRb, how="left", left_on="Product Code", right_on="Product Code")

# wide["Sales below threshold"] = np.where(wide["qty_pct_change_L2W_vs_P90"] < totalTreshold, "Yes", "No")
# wide["margin_L2W > target"] = np.where(wide["margin_L2W"] >= marginTreshold, "Yes", "No")
# wide["Rb active in analysis period"] = np.where((wide["Rb end date"] >= p90_start), "Yes", "No")
# wide["Target less sales"] = np.where((wide["Sales below threshold"] == "Yes") & 
#                                      (wide["margin_L2W > target"] == "Yes") &
#                                      (wide["Rb active in analysis period"]!="Yes"), "Yes", "No")

# # -----------------------
# # 2) Monthly reference for ALL months in data
# #    (avg_daily_qty = monthly qty / # calendar days in that month)
# # -----------------------
# df['month'] = df['order_date'].dt.to_period('M')

# monthly = (
#     df.groupby(['Product Code', 'month'], as_index=False)
#       .agg(qty=('quantity', 'sum'),
#            rev=('revenue', 'sum'),
#            prof=('profit', 'sum'))
# )

# monthly['days_in_month'] = monthly['month'].dt.days_in_month
# monthly['avg_daily_qty'] = monthly['qty'] / monthly['days_in_month']
# monthly['margin'] = monthly['prof'] / monthly['rev']

# monthly_wide = monthly.pivot(index='Product Code', columns='month', values=['avg_daily_qty', 'margin'])
# monthly_wide.columns = [f'{m}_{str(mon)}' for (m, mon) in monthly_wide.columns]  # e.g. avg_daily_qty_2025-10
# monthly_wide = monthly_wide.reset_index()

# # -----------------------
# # 3) Merge monthly reference to the right
# # -----------------------
# final = wide.merge(monthly_wide, on='Product Code', how='left')

# # -----------------------
# # 4) Round to 2 decimals
# # -----------------------
# num_cols = final.select_dtypes(include='number').columns
# final[num_cols] = final[num_cols].round(4)


# finalTarget = final[final["Target less sales"] == "Yes"].copy()

# final


In [44]:
# targetVolume = finalTarget["avg_daily_qty_L2W"].sum()
# totalVolume = final["avg_daily_qty_L2W"].sum()
# percVolume = targetVolume / totalVolume
# print(percVolume)

In [45]:
# # finalTarget categories for margin_L2W for 11% to 13%, 13 to 15, 15 to 17 and 17+
# finalTarget["margin_L2W cat"] = pd.cut(finalTarget["margin_L2W"], bins=[-np.inf,.08, 0.11, 0.13, 0.15, 0.17, np.inf], labels=["<8%", "8% to <11%", "11% to <13%", "13% to <15%", "15% to <17%", ">=17%"])
# finalTarget["margin_L2W cat"].value_counts()

In [46]:
# final["qty_abs_change_L2W_vs_P90"].sum()

In [47]:
# final[final["Target less sales"] == "Yes"]["qty_abs_change_L2W_vs_P90"].sum()

In [48]:
# final.to_excel("Walmart B2B SKU-level sales and margin analysis.xlsx", index=False)

<!-- ### Assign groups -->

In [49]:
# import random
# from typing import Dict, List, Optional, Sequence, Tuple

# import pandas as pd


# def split_skus_into_groups(
#     df: pd.DataFrame,
#     n_groups: int,
#     code_col: str = "Product Code",
#     qty_col: str = "Total quantity",
#     seed: int = 42,
#     max_iters: int = 200_000,
#     patience: int = 40_000,
#     allow_equal_sum_only_if_sizes_met: bool = True,
# ) -> pd.DataFrame:
#     """
#     Partition rows into exactly `n_groups` groups such that:
#       - group sizes are as equal as possible (fixed targets)
#       - group total quantities are as close as possible

#     Returns: df copy with new column 'group' in [1..n_groups].

#     Algorithm (fast + good quality):
#       1) Size-constrained greedy assignment (largest qty first -> lowest current sum among non-full groups)
#       2) Local search with pairwise swaps across groups to reduce sum imbalance

#     Notes:
#       - Size targets are fixed: e.g. n=10, k=3 -> [4,3,3]; n=10, k=4 -> [3,3,2,2]
#       - Handles non-numeric qty by coercing to 0.
#     """
#     if n_groups < 2:
#         raise ValueError("n_groups must be >= 2")
#     if code_col not in df.columns or qty_col not in df.columns:
#         raise ValueError(f"df must contain columns: '{code_col}' and '{qty_col}'")

#     work = df[[code_col, qty_col]].copy()
#     work[qty_col] = pd.to_numeric(work[qty_col], errors="coerce").fillna(0)

#     n = len(work)
#     out = df.copy()
#     if n == 0:
#         out["group"] = pd.Series(dtype="int64")
#         return out

#     # --- Target sizes: as equal as possible (largest groups first) ---
#     base = n // n_groups
#     r = n % n_groups
#     target_sizes = [base + (1 if i < r else 0) for i in range(n_groups)]

#     rng = random.Random(seed)

#     # --- 1) Greedy init (largest qty first, tie-broken by shuffle) ---
#     idxs = list(work.index)
#     rng.shuffle(idxs)
#     work = work.loc[idxs].sort_values(qty_col, ascending=False)

#     groups: List[List[int]] = [[] for _ in range(n_groups)]
#     sums: List[float] = [0.0] * n_groups

#     for row_idx, qty in zip(work.index, work[qty_col].tolist()):
#         eligible = [g for g in range(n_groups) if len(groups[g]) < target_sizes[g]]
#         # Choose lowest sum; tie-break on smaller size; then random
#         eligible.sort(key=lambda g: (sums[g], len(groups[g]), rng.random()))
#         g = eligible[0]
#         groups[g].append(row_idx)
#         sums[g] += float(qty)

#     qty_map: Dict[int, float] = work[qty_col].to_dict()

#     def spread(s: Sequence[float]) -> float:
#         return float(max(s) - min(s))

#     def sse(s: Sequence[float]) -> float:
#         # sum of squared error vs mean (helps overall balance, not just min/max)
#         m = sum(s) / len(s)
#         return float(sum((x - m) ** 2 for x in s))

#     # Objective: prioritize lowering max-min spread; use SSE as tie-breaker
#     def objective(s: Sequence[float]) -> Tuple[float, float]:
#         return (spread(s), sse(s))

#     best_groups = [g[:] for g in groups]
#     best_sums = sums[:]
#     best_obj = objective(best_sums)

#     # --- 2) Local search via swaps (keeps sizes exactly fixed) ---
#     group_of = {idx: g for g in range(n_groups) for idx in groups[g]}
#     group_lists = [g[:] for g in groups]
#     current_sums = sums[:]
#     current_obj = objective(current_sums)

#     # Precompute which groups are currently highest / lowest sum often helps,
#     # but we keep it simple + fast with random sampling.
#     no_improve = 0

#     def try_swap(a: int, b: int) -> bool:
#         nonlocal current_obj, best_obj, best_groups, best_sums

#         ga, gb = group_of[a], group_of[b]
#         if ga == gb:
#             return False

#         qa = float(qty_map[a])
#         qb = float(qty_map[b])

#         new_sums = current_sums[:]
#         new_sums[ga] = new_sums[ga] - qa + qb
#         new_sums[gb] = new_sums[gb] - qb + qa

#         new_obj = objective(new_sums)

#         # Accept if strictly better objective
#         if new_obj < current_obj:
#             # apply swap
#             group_lists[ga].remove(a)
#             group_lists[gb].remove(b)
#             group_lists[ga].append(b)
#             group_lists[gb].append(a)
#             group_of[a], group_of[b] = gb, ga
#             current_sums[:] = new_sums
#             current_obj = new_obj

#             if new_obj < best_obj:
#                 best_obj = new_obj
#                 best_sums = new_sums[:]
#                 best_groups = [g[:] for g in group_lists]
#             return True

#         return False

#     for _ in range(max_iters):
#         # Pick two different groups; bias toward swapping from high-sum to low-sum groups.
#         # This usually improves convergence vs totally random.
#         order = sorted(range(n_groups), key=lambda g: current_sums[g])
#         low = rng.choice(order[: max(1, n_groups // 3)])
#         high = rng.choice(order[-max(1, n_groups // 3) :])

#         if low == high:
#             continue
#         if not group_lists[low] or not group_lists[high]:
#             continue

#         a = rng.choice(group_lists[high])  # item from high-sum group
#         b = rng.choice(group_lists[low])   # item from low-sum group

#         improved = try_swap(a, b)

#         if improved:
#             no_improve = 0
#         else:
#             no_improve += 1
#             if no_improve >= patience:
#                 break

#     # --- Build assignment column ---
#     assignment = {idx: (g + 1) for g in range(n_groups) for idx in best_groups[g]}
#     out["group"] = out.index.map(assignment).astype("int64")

#     return out


# # --- Example ---
# # df_with_groups = split_skus_into_groups(dfSalesSKUJanuary, n_groups=5,
# #                                        code_col="Product Code", qty_col="January quantity")


In [50]:
# # #use functino split_skus_into_3_groups to finalTarget dataframe based on "Product Code" and "avg_daily_qty_L2W"
# finalTargetWithGroups = split_skus_into_groups(df = finalTarget, n_groups=3, code_col="Product Code", qty_col="avg_daily_qty_L2W", seed=10)
# finalTargetWithGroupsMerge = finalTargetWithGroups[["Product Code", "group","qty_pct_change_L2W_vs_P90"]].copy()
# display(finalTargetWithGroups)
# finalTargetWithGroupsCheck = finalTargetWithGroups.groupby("group").agg({"Product Code": "count", "avg_daily_qty_L2W": "sum"})
# finalTargetWithGroupsCheck

### Prev

In [51]:
# dfSales["Brand code"] = dfSales["Product Code"].str[:4]

# target_brands = ["BLHK", "IRON", "LION", "MICH", "ARMS", "WATF", "FIRE"]


# dfSalesBrands = dfSales[dfSales["Brand code"].isin(target_brands)].copy()
# dfSalesBrands

In [52]:
# dfSalesSKU = dfSalesBrands.groupby(["Product Code","Brand code"]).agg({"quantity": "sum"}).reset_index()
# dfSalesSKU = dfSalesSKU.rename(columns={"quantity": "Total quantity"})
# dfSalesSKU

In [53]:
# dfSalesSKUJanuary = dfSalesBrands[dfSalesBrands["order_date"].dt.month == 1].groupby(["Product Code","Brand code"]).agg({"quantity": "sum"}).reset_index()
# dfSalesSKUJanuary = dfSalesSKUJanuary.rename(columns={"quantity": "Total quantity"})
# dfSalesSKUJanuary

In [54]:
# dfSalesSKUNotJan = dfSalesSKU[dfSalesSKU["Product Code"].isin(dfSalesSKUJanuary["Product Code"]) == False].copy()
# dfSalesSKUNotJan

In [55]:
# def selection_by_brand(dfSales: pd.DataFrame, target_brands: List[str]) -> pd.DataFrame:

#     selection = pd.DataFrame()
#     for brand in target_brands:
#         # print(f"Processing brand: {brand}")
#         dfBrand = dfSales[dfSales["Brand code"] == brand].copy()
#         dfBrandWithGroups = split_skus_into_3_groups(dfBrand, seed=42)
#         # print(dfBrandWithGroups[["Product Code", "Total quantity", "group"]].to_string(index=False))
#         # print("\n")

#         selection = pd.concat([selection, dfBrandWithGroups], ignore_index=True)

#     #BY GROUP DISPLAY # SKUS AND TOTAL QTY
#     display(
#         selection.groupby("group", as_index=False)
#         .agg(
#             sku_count=("Product Code", "count"),
#             total_qty=("Total quantity", "sum"),
#         )
#         .sort_values("group")
#     )

#     return selection

In [56]:
# selectionJan = selection_by_brand(dfSalesSKUJanuary, target_brands)
# selectionJan

In [57]:
# selectionNotJan = selection_by_brand(dfSalesSKUNotJan, target_brands)
# selectionNotJan

In [58]:
# finalGroups = pd.concat([selectionJan, selectionNotJan], ignore_index=True).drop(columns=["Total quantity","Brand code"])
# finalGroups

In [59]:
# dfSalesSKUWithGroups = dfSalesSKU.merge(finalGroups, how="left", on="Product Code")
# dfSalesSKUWithGroups["group"].value_counts()

# # dfSalesSKUWithGroups sum sales by group
# dfSalesByGroup = dfSalesSKUWithGroups.groupby("group").agg({"Total quantity": "sum","Product Code": "count"}).reset_index()
# dfSalesByGroup

## Current tests tracker

In [60]:
# id = "12jQyVDfPnR9qUxfbwmFPWjHxTMZNk59I"
# dfCurrentTestsAll = gAPI.get_df_of_file(service, id, "csv", dtype = {"Identifier": str})

pathTracker = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Final node level costs tracker.csv"

dfCurrentTestsAll = pd.read_csv(pathTracker, dtype = {"Identifier": str})

dfCurrentTestsAll["SKU-Node"] = dfCurrentTestsAll["Product Code"] + "-" + dfCurrentTestsAll["Identifier"]


dfCurrentTest = dfCurrentTestsAll.copy()
dfCurrentTests = dfCurrentTest[["SKU-Node", "Final target","Final price category","Start date", "Sub-group"]].copy()
dfCurrentTestsAll["Final target"].value_counts()

C:\Users\franc\AppData\Local\Temp\ipykernel_18248\4003682945.py:6: DtypeWarning: Columns (3,6,7,11,12,13,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  dfCurrentTestsAll = pd.read_csv(pathTracker, dtype = {"Identifier": str})


Final target
Added                       1146693
Normal strategy              515714
Updated                      334807
Margin test                  303443
DSVD test                    291097
No sales test                248736
Decreased - margin > 20%     143075
Updates test                 112817
Increase test                101521
Wm margin split test          95983
Shipping cost added           31917
Less sales back               14592
Less sales                    14193
Name: count, dtype: int64

In [61]:
if compare_dsvs:
    dfDSVChanges["Product Code"] = dfDSVChanges["SKU-Node"].str.split("-", n=1, regex=False).str[0]
    dfDSVChanges["Identifier"] = dfDSVChanges["SKU-Node"].str.split("-", n=1, regex=False).str[1]
    dfDSVChanges = dfDSVChanges.merge(dfCurrentTests[["Final target", "SKU-Node"]], on=["SKU-Node"], how="left")

    dfDSVChanges["Final target"] = np.where(dfDSVChanges["Price change % category"] == "Increase", "Updated",
                                                np.where((dfDSVChanges["Price change % category"] == "Decrease") & (dfDSVChanges["Final target"]!= "Wm margin split test"), "Decreased - margin > 20%", 
                                                        dfDSVChanges["Final target"]))

    dfDSVChanges["Final price category"] = np.where(dfDSVChanges["Price change % category"] == "Decrease", "20%","11%")
    dfDSVChanges = dfDSVChanges.rename(columns={"Price change %": "Final delta vs current %",
                                                    "walmart_dsv_price_current":"Final price",
                                                    "Price change % category": "Final price change category final"})
    dfDSVChanges["Node type"] = "HUB"
    dfDSVChanges["Min units"] = 8
    dfDSVChanges["Start date"] = "2026-03-13"
    display(dfDSVChanges["Final target"].value_counts())                        

In [62]:
# dfCurrentTestsAll["Final price category"] = (
#     dfCurrentTestsAll["Final price category"]
#     .astype(str)
#     .str.strip()
#     .str.replace(r"%$", "", regex=True)   # remove existing trailing %
#     .str.extract(r"(\d+)", expand=False)  # keep just the number
#     .add("%")
# )

In [63]:
# dfCurrentTestsAll["Sub-group"] = dfCurrentTestsAll["Sub-group"].fillna(dfCurrentTestsAll["Final price category"])

# Initial output

## Function

In [64]:
def calculate_new_node_level_cost(dfInvAll, dfAvgWmtPrice, dfMAP, minWmMargin=0, distanceMAP=0.05, min_units = 4):

    dfInvFinal = process_inventory_nlc(dfInvAll, metricGroup="min", min_units = min_units)

    dfOutput = dfInvFinal.merge(dfAvgWmtPrice, how="left", on="Product Code").merge(dfMAP, how="left", on="Product Code").merge(dfCostNode, how="left", on=["Identifier"])
    dfOutput["Shipping cost"] = dfOutput["Shipping cost"].fillna(0)
    dfOutput["Cost+Shipping"] = dfOutput["Purchase Price+FET"] + dfOutput["Shipping cost"]
    dfOutput["Is MAP now?"] = np.where(dfOutput["MAP"].isna(), "No", "Yes")

    margins = [0.06, 0.08,0.11,0.12,0.13,0.14,0.15, 0.2]
    minWmMargin = 0 #-.05  # 0 -0.05
    distanceMAP = 0.05  # 0.05

    for margin in margins:
        margin_suffix = f" - {int(margin*100)}% margin"
        dfOutput["Node level cost" + margin_suffix] = round(dfOutput["Purchase Price+FET"] /(1-margin),2) 
        dfOutput["Walmart Margin" + margin_suffix] = round((dfOutput["offer_price"] - dfOutput["Node level cost" + margin_suffix]) / dfOutput["offer_price"],4)
        dfOutput["Node cost < MAP - min margin%" + margin_suffix] = np.where(dfOutput["Node level cost" + margin_suffix] < dfOutput["MAP"] *(1-distanceMAP), "Yes", "No")
        dfOutput["Node cost - MAP" + margin_suffix] = dfOutput["Node level cost" + margin_suffix] - dfOutput["MAP"]
        dfOutput["Target for node level cost?" + margin_suffix] = np.where(dfOutput["Is MAP now?"] == "Yes",
                                                                        np.where(dfOutput["Node cost < MAP - min margin%" + margin_suffix]== "No", "No", "Yes"), #This happens if it is MAP now
                                                                                np.where(dfOutput["Walmart Margin" + margin_suffix]> minWmMargin, "Yes", "No")) #This happens if it is not MAP now
             
        dfOutput["Node level cost" + margin_suffix] = dfOutput["Node level cost" + margin_suffix] + dfOutput["Shipping cost"]

    dfOutput = dfOutput.merge(dfCurrDSVNLC, how="left", on=["Product Code", "Identifier"])


    dfOutput["Final node level cost category"] = np.where(dfOutput["Target for node level cost? - 11% margin"] == "Yes", "11%",
                                                np.where(dfOutput["Target for node level cost? - 8% margin"] == "Yes", "8%",
                                                        np.where(dfOutput["Target for node level cost? - 6% margin"] == "Yes", "6%", "N/A")))

    dfOutput["Final node level cost"] = np.where(dfOutput["Target for node level cost? - 11% margin"] == "Yes", dfOutput["Node level cost - 11% margin"],
                                                np.where(dfOutput["Target for node level cost? - 8% margin"] == "Yes", dfOutput["Node level cost - 8% margin"],
                                                        np.where(dfOutput["Target for node level cost? - 6% margin"] == "Yes", dfOutput["Node level cost - 6% margin"], np.nan)))


    
    dfOutput["Final walmart margin"] = np.where(dfOutput["Target for node level cost? - 11% margin"] == "Yes", dfOutput["Walmart Margin - 11% margin"],
                                                np.where(dfOutput["Target for node level cost? - 8% margin"] == "Yes", dfOutput["Walmart Margin - 8% margin"],
                                                        np.where(dfOutput["Target for node level cost? - 6% margin"] == "Yes", dfOutput["Walmart Margin - 6% margin"], np.nan)))


    dfOutput["Final price change %"] = round((dfOutput["Final node level cost"] - dfOutput["current_nlc_price"])/ dfOutput["current_nlc_price"], 3)
    dfOutput["Final price change category"] = np.where(dfOutput["Final price change %"] < 0, "Decrease",
                                                     np.where(dfOutput["Final price change %"] > 0, "Increase", "No change"))

    dfOutput["SKU-Node"] = dfOutput["Product Code"] + "-" + dfOutput["Identifier"]

    dfOutput["Brand code"] = dfOutput["Product Code"].str[:4]

    dfOutput["New NLC"] = np.where(dfOutput["current_nlc_price"].isna(), "Yes", "No")

    dfOutput["Min units"] = min_units

    dfOutput["current_nlc_margin"] = round((dfOutput["current_nlc_price"] - dfOutput["Cost+Shipping"])/dfOutput["current_nlc_price"],4)  

    #dfOutput["current_nlc_margin category" make categories: 0 to 6%, 6% to 8%, 8% to 10.8%,10.8% to 11.2%, 11.2% to 13%, 13 to 15, 15 to 17 and 17+%
    dfOutput["current_nlc_margin category"] = pd.cut(dfOutput["current_nlc_margin"], bins=[-np.inf, 0, 0.06, 0.08, 0.108, 
                                                                                            0.112, 0.13, 0.15, 0.17, 0.198, np.inf], 
                                                                                            labels=["<0%", "0% to <6%", "6% to <8%", "8% to <10.8%", "10.8% to <11.2%", 
                                                                                                     "11.2% to <13%", "13% to <15%", "15% to <17%", "17% to <20%", ">=20%"])


    dfOutput["Current walmart margin at National"] = round((dfOutput["offer_price"] - dfOutput["unit_cost"])/dfOutput["offer_price"],4)
    dfOutput["Current walmart margin at NLC"] = round((dfOutput["offer_price"] - dfOutput["current_nlc_price"])/dfOutput["offer_price"],4)
    dfOutput["Current walmart margin at NLC category"] = pd.cut(dfOutput["Current walmart margin at NLC"], bins=[-np.inf, -.1, 0,0.1, 0.15, 0.2, 0.25, 0.3, np.inf], labels=["<-10%", "-10% to 0%", "0% to 10%", "10% to 15%", "15% to 20%", "20% to 25%", "25% to 30%", ">=30%"])

    dfOutput["Walmart margin at new NLC category"] = pd.cut(dfOutput["Final walmart margin"], bins=[-np.inf, -.1, 0,0.1, 0.15, 0.2, 0.25, 0.3, np.inf], labels=["<-10%", "-10% to 0%", "0% to 10%", "10% to 15%", "15% to 20%", "20% to 25%", "25% to 30%", ">=30%"])


    dfOutput["Total margin"] = round(dfOutput["Current walmart margin at NLC"] + dfOutput["current_nlc_margin"],4)
    margin_splits = [0.50, 0.60]

    for margin in margin_splits:
        dfOutput[f"Wmt margin {int(margin * 100)}% group"] = round(dfOutput["Total margin"] * (1-margin), 4)
        dfOutput[f"Price margin split {int(margin * 100)}%"] = round(dfOutput["offer_price"] * (1- dfOutput[f"Wmt margin {int(margin * 100)}% group"]),2)
        dfOutput[f"Price margin split {int(margin * 100)}%"] = np.minimum(dfOutput[f"Price margin split {int(margin * 100)}%"], dfOutput["Node level cost - 20% margin"])

    dfOutput = dfOutput.merge(dfSKUNodeRevenue, how = "left", on = "SKU-Node").merge(dfSKURevenue, how = "left", on = "Product Code")

    dfOutputAll = dfOutput.copy()
    print("Total of rows in dfOutputAll:", len(dfOutputAll))

    dfOutput = dfOutput[dfOutput["Final node level cost category"] != "N/A"].copy()

    print("Rows we can work with (not N/A):", len(dfOutput))

    display(dfOutput["New NLC"].value_counts())

    return dfOutputAll

## Df Output

In [65]:
dfOutputAll4 = calculate_new_node_level_cost(dfInvAll, dfAvgWmtPrice, dfMAP, minWmMargin=0, distanceMAP=0.05, min_units=4)
print("---- Done with min_units = 4 ----")
dfOutputAll8 = calculate_new_node_level_cost(dfInvAll, dfAvgWmtPrice, dfMAP, minWmMargin=0, distanceMAP=0.05, min_units=8)
print("---- Done with min_units = 8 ----")

dfOutput4 = dfOutputAll4[dfOutputAll4["Final node level cost category"] != "N/A"].copy()
dfOutput8 = dfOutputAll8[dfOutputAll8["Final node level cost category"] != "N/A"].copy()

dfOutput4Notin8 = dfOutput4[~dfOutput4["SKU-Node"].isin(dfOutput8["SKU-Node"])].copy()
dfOutputInitial = pd.concat([dfOutput8, dfOutput4Notin8], ignore_index=True)
print(len(dfOutputInitial))

dfOutput = dfOutputInitial.merge(dfCurrentTests[["SKU-Node","Final target","Start date","Sub-group"]], how="left", left_on="SKU-Node", right_on="SKU-Node")

dfOutput = dfOutput[dfOutput["Product Code"].isin(dfRollbacks["Product Code"]) == False].copy()

dfOutput

Total of rows in dfOutputAll: 2317293
Rows we can work with (not N/A): 2204166


New NLC
No     2139169
Yes      64997
Name: count, dtype: int64

---- Done with min_units = 4 ----
Total of rows in dfOutputAll: 1525748
Rows we can work with (not N/A): 1447405


New NLC
No     1404884
Yes      42521
Name: count, dtype: int64

---- Done with min_units = 8 ----
2204451


,Product Code,Identifier,Warehouse Code,Available,Purchase Price+FET,date,Node type,offer_price,unit_cost,MAP,Shipping cost,Cost+Shipping,Is MAP now?,Node level cost - 6% margin,Walmart Margin - 6% margin,Node cost < MAP - min margin% - 6% margin,Node cost - MAP - 6% margin,Target for node level cost? - 6% margin,Node level cost - 8% margin,Walmart Margin - 8% margin,Node cost < MAP - min margin% - 8% margin,Node cost - MAP - 8% margin,Target for node level cost? - 8% margin,Node level cost - 11% margin,Walmart Margin - 11% margin,Node cost < MAP - min margin% - 11% margin,Node cost - MAP - 11% margin,Target for node level cost? - 11% margin,Node level cost - 12% margin,Walmart Margin - 12% margin,Node cost < MAP - min margin% - 12% margin,Node cost - MAP - 12% margin,Target for node level cost? - 12% margin,Node level cost - 13% margin,Walmart Margin - 13% margin,Node cost < MAP - min margin% - 13% margin,Node cost - MAP - 13% margin,Target for node level cost? - 13% margin,Node level cost - 14% margin,Walmart Margin - 14% margin,Node cost < MAP - min margin% - 14% margin,Node cost - MAP - 14% margin,Target for node level cost? - 14% margin,Node level cost - 15% margin,Walmart Margin - 15% margin,Node cost < MAP - min margin% - 15% margin,Node cost - MAP - 15% margin,Target for node level cost? - 15% margin,Node level cost - 20% margin,Walmart Margin - 20% margin,Node cost < MAP - min margin% - 20% margin,Node cost - MAP - 20% margin,Target for node level cost? - 20% margin,current_nlc_price,Final node level cost category,Final node level cost,Final walmart margin,Final price change %,Final price change category,SKU-Node,Brand code,New NLC,Min units,current_nlc_margin,current_nlc_margin category,Current walmart margin at National,Current walmart margin at NLC,Current walmart margin at NLC category,Walmart margin at new NLC category,Total margin,Wmt margin 50% group,Price margin split 50%,Wmt margin 60% group,Price margin split 60%,sku_node_revenue,sku_revenue,sku_sales_category,Final target,Start date,Sub-group
0,ACCE0131820535YXL,628326659,5614,85,56.95,2026-04-02,HUB,68.360000,61.79,NaN,0.0,56.95,No,60.59,0.1137,No,NaN,Yes,61.90,0.0945,No,NaN,Yes,63.99,0.0639,No,NaN,Yes,64.72,0.0532,No,NaN,Yes,65.46,0.0424,No,NaN,Yes,66.22,0.0313,No,NaN,Yes,67.00,0.0199,No,NaN,Yes,71.19,-0.0414,No,NaN,No,63.99,11%,63.99,0.0639,0.000,No change,ACCE0131820535YXL-628326659,ACCE,No,8,0.1100,10.8% to <11.2%,0.0961,0.0639,0% to 10%,0% to 10%,0.1739,0.0870,62.41,0.0696,63.60,NaN,137.98,95-100%,Updates test,2026-02-12,Update
1,ACCE0221316580T,6283261113,5237,16,38.11,2026-04-01,HUB,45.930000,37.94,NaN,0.0,38.11,No,40.54,0.1174,No,NaN,Yes,41.42,0.0982,No,NaN,Yes,42.82,0.0677,No,NaN,Yes,43.31,0.0570,No,NaN,Yes,43.80,0.0464,No,NaN,Yes,44.31,0.0353,No,NaN,Yes,44.84,0.0237,No,NaN,Yes,47.64,-0.0372,No,NaN,No,43.25,11%,42.82,0.0677,-0.010,Decrease,ACCE0221316580T-6283261113,ACCE,No,8,0.1188,11.2% to <13%,0.1740,0.0583,0% to 10%,0% to 10%,0.1771,0.0886,41.86,0.0708,42.68,NaN,309.02,95-100%,Normal strategy,2026-01-05,11%
2,ACCE0221316580T,6283261117,5266,20,32.21,2026-04-02,HUB,45.930000,37.94,NaN,0.0,32.21,No,34.27,0.2539,No,NaN,Yes,35.01,0.2378,No,NaN,Yes,36.19,0.2121,No,NaN,Yes,36.60,0.2031,No,NaN,Yes,37.02,0.1940,No,NaN,Yes,37.45,0.1846,No,NaN,Yes,37.89,0.1750,No,NaN,Yes,40.26,0.1234,No,NaN,Yes,39.21,11%,36.19,0.2121,-0.077,Decrease,ACCE0221316580T-6283261117,ACCE,No,8,0.1785,17% to <20%,0.1740,0.1463,10% to 15%,20% to 25%,0.3248,0.1624,38.47,0.1299,39.96,NaN,309.02,95-100%,Normal strategy,2026-01-05,11%
3,ACCE0221316580T,6283261134,5234,8,38.61,2026-04-02,HUB,45.930000,37.94,NaN,7.0,45.61,No,48.07,0.1058,No,NaN,Yes,48.97,0.0862,No,NaN,Yes,50.38,0.0555,No,NaN,Yes,50.88,0.0446,No,NaN,Yes,51.38,0.0337,No,NaN,Yes,51.90,0.0224,No,NaN,Yes,52.42,0.0111,No,NaN,Yes,55.26,-0.0507,No,NaN,No,41.69,11%,50.38,0.0555,0.208,Increase,ACCE0221316580T-6283261134,ACCE,No,8,-0.0940,<0%,0.1740,0.0923,0% to 10%,0% to 10%,-0.0017,-0.0008,45.97,-0.0007,45.96,177.88,309.02,95-1

In [66]:
dfOutput["Final target"].value_counts()

Final target
Added                       560030
Normal strategy             295250
Margin test                 260841
DSVD test                   257891
Updated                     242472
No sales test               155577
Decreased - margin > 20%    128341
Increase test                95843
Wm margin split test         86470
Updates test                 66000
Shipping cost added          23484
Less sales back               2705
Less sales                    2221
Name: count, dtype: int64

In [67]:
dfOutput["current_nlc_margin category"].value_counts()

current_nlc_margin category
10.8% to <11.2%    474924
11.2% to <13%      394413
8% to <10.8%       348037
13% to <15%        242680
15% to <17%        175739
6% to <8%          154591
17% to <20%        140717
>=20%               87712
0% to <6%           73683
<0%                 46950
Name: count, dtype: int64

In [68]:
dfAnalyze = dfOutput[dfOutput["current_nlc_margin"] >0.13].copy()#
dfAnalyze

,Product Code,Identifier,Warehouse Code,Available,Purchase Price+FET,date,Node type,offer_price,unit_cost,MAP,Shipping cost,Cost+Shipping,Is MAP now?,Node level cost - 6% margin,Walmart Margin - 6% margin,Node cost < MAP - min margin% - 6% margin,Node cost - MAP - 6% margin,Target for node level cost? - 6% margin,Node level cost - 8% margin,Walmart Margin - 8% margin,Node cost < MAP - min margin% - 8% margin,Node cost - MAP - 8% margin,Target for node level cost? - 8% margin,Node level cost - 11% margin,Walmart Margin - 11% margin,Node cost < MAP - min margin% - 11% margin,Node cost - MAP - 11% margin,Target for node level cost? - 11% margin,Node level cost - 12% margin,Walmart Margin - 12% margin,Node cost < MAP - min margin% - 12% margin,Node cost - MAP - 12% margin,Target for node level cost? - 12% margin,Node level cost - 13% margin,Walmart Margin - 13% margin,Node cost < MAP - min margin% - 13% margin,Node cost - MAP - 13% margin,Target for node level cost? - 13% margin,Node level cost - 14% margin,Walmart Margin - 14% margin,Node cost < MAP - min margin% - 14% margin,Node cost - MAP - 14% margin,Target for node level cost? - 14% margin,Node level cost - 15% margin,Walmart Margin - 15% margin,Node cost < MAP - min margin% - 15% margin,Node cost - MAP - 15% margin,Target for node level cost? - 15% margin,Node level cost - 20% margin,Walmart Margin - 20% margin,Node cost < MAP - min margin% - 20% margin,Node cost - MAP - 20% margin,Target for node level cost? - 20% margin,current_nlc_price,Final node level cost category,Final node level cost,Final walmart margin,Final price change %,Final price change category,SKU-Node,Brand code,New NLC,Min units,current_nlc_margin,current_nlc_margin category,Current walmart margin at National,Current walmart margin at NLC,Current walmart margin at NLC category,Walmart margin at new NLC category,Total margin,Wmt margin 50% group,Price margin split 50%,Wmt margin 60% group,Price margin split 60%,sku_node_revenue,sku_revenue,sku_sales_category,Final target,Start date,Sub-group
2,ACCE0221316580T,6283261117,5266,20,32.21,2026-04-02,HUB,45.930,37.9400,NaN,0.0,32.21,No,34.27,0.2539,No,NaN,Yes,35.01,0.2378,No,NaN,Yes,36.19,0.2121,No,NaN,Yes,36.60,0.2031,No,NaN,Yes,37.02,0.1940,No,NaN,Yes,37.45,0.1846,No,NaN,Yes,37.89,0.1750,No,NaN,Yes,40.26,0.1234,No,NaN,Yes,39.21,11%,36.19,0.2121,-0.077,Decrease,ACCE0221316580T-6283261117,ACCE,No,8,0.1785,17% to <20%,0.1740,0.1463,10% to 15%,20% to 25%,0.3248,0.1624,38.47,0.1299,39.96,NaN,309.02,95-100%,Normal strategy,2026-01-05,11%
11,ACCE0221316580T,628326864,5266,20,32.21,2026-04-02,HUB,45.930,37.9400,NaN,0.0,32.21,No,34.27,0.2539,No,NaN,Yes,35.01,0.2378,No,NaN,Yes,36.19,0.2121,No,NaN,Yes,36.60,0.2031,No,NaN,Yes,37.02,0.1940,No,NaN,Yes,37.45,0.1846,No,NaN,Yes,37.89,0.1750,No,NaN,Yes,40.26,0.1234,No,NaN,Yes,39.21,11%,36.19,0.2121,-0.077,Decrease,ACCE0221316580T-628326864,ACCE,No,8,0.1785,17% to <20%,0.1740,0.1463,10% to 15%,20% to 25%,0.3248,0.1624,38.47,0.1299,39.96,NaN,309.02,95-100%,Normal strategy,2026-01-05,11%
12,ACCE0221316580T,628326916,6012,16,38.71,2026-04-02,HUB,45.930,37.9400,NaN,0.0,38.71,No,41.18,0.1034,No,NaN,Yes,42.08,0.0838,No,NaN,Yes,43.49,0.0531,No,NaN,Yes,43.99,0.0422,No,NaN,Yes,44.49,0.0314,No,NaN,Yes,45.01,0.0200,No,NaN,Yes,45.54,0.0085,No,NaN,Yes,48.39,-0.0536,No,NaN,No,46.69,11%,43.49,0.0531,-0.069,Decrease,ACCE0221316580T-628326916,ACCE,No,8,0.1709,17% to <20%,0.1740,-0.0165,-10% to 0%,0% to 10%,0.1544,0.0772,42.38,0.0618,43.09,93.38,309.02,95-100%,Normal strategy,2026-01-05,11%
13,ACCE0221316580T,628326937,6072,48,35.78,2026-04-02,HUB,45.930,37.9400,NaN,0.0,35.78,No,38.06,0.1713,No,NaN,Yes,38.89,0.1533,No,NaN,Yes,40.20,0.1248,No,NaN,Yes,40.66,0.1147,No,NaN,Yes,41.13,0.1045,No,NaN,Yes,41.60,0.0943,No,NaN,Yes,42.09,0.0836,No,NaN,Yes,44.72,0.0263,No,NaN,Yes,44.72,11%,40.20,0.1248,-0.101,Decrease,ACCE0221316580T-628326937,ACCE,No,8,0.1999,>=20%,0.1740,0.0263,0% to 10%,10% to 15%,0.2262,0.1131,40.74,0.0905,41.77,NaN,309.02,95-100%,Decreas

In [69]:
#summary = dfAnalyze group by "Final target" and "current_nlc_margin category" and count number of skus in each group and sum of sku_node_revenue and % sku_node_revenue/total_revenue (total revenue already defined)
summary = dfAnalyze.groupby(["Final target"], as_index=False).agg(
    sku_count=("SKU-Node", "count"),
    total_sku_node_revenue=("sku_node_revenue", "sum")
)

summary["% of total revenue"] = summary["total_sku_node_revenue"] / total_revenue
summary.sort_values("total_sku_node_revenue", ascending=False, inplace=True)
summary

,Final target,sku_count,total_sku_node_revenue,% of total revenue
12,Wm margin split test,45902,4220346.92,0.046304
6,Margin test,69917,3362369.23,0.036890
8,Normal strategy,103772,3296828.86,0.036171
2,Decreased - margin > 20%,90588,1321084.14,0.014494
10,Updated,60315,1296386.60,0.014223
0,Added,116431,1291631.41,0.014171
1,DSVD test,72088,888307.41,0.009746
11,Updates test,25479,764010.38,0.008382
7,No sales test,55372,236078.80,0.002590
5,Less sales back,899,147577.91,0.001619


In [70]:
summary["% of total revenue"].sum()

0.18704768125166008

## New sku-nodes + checks

In [71]:
dfOutputNew = dfOutput[dfOutput["New NLC"] == "Yes"].copy()
display(dfOutputNew["Min units"].value_counts())
dfOutputNew["Brand code"].value_counts()

Min units
8    42521
4    22484
Name: count, dtype: int64

Brand code
CONT    10665
SAIL     7058
GOYR     5147
NITT     5084
TOYO     4683
        ...  
GRTM        1
NEBL        1
ZEEM        1
ATLN        1
TOST        1
Name: count, Length: 144, dtype: int64

In [72]:
#How many lines has dfOutput where Product Code is in Product Code of dfOutputNew

In [73]:
dfOutputNew[["Product Code","sku_revenue"]].drop_duplicates()["sku_revenue"].sum()/total_revenue

0.3286296070882381

In [74]:
# dfRollbacksEnded = dfRollbacksAll[dfRollbacksAll["End date"] == '2026-03-13'].copy()


In [75]:
# dfOutputNew[dfOutputNew["Product Code"].isin(dfRollbacksEnded["Product Code"])]["Product Code"].value_counts()

In [76]:
dfOutputNew["Identifier"].value_counts()

Identifier
6283261560    2050
6283261561    1951
6283261562    1075
628326470      976
6283261147     976
              ... 
6283261158       1
628326987        1
628326631        1
628326638        1
62832659         1
Name: count, Length: 1135, dtype: int64

In [77]:
dfOutputNew["Warehouse Code"].value_counts()

Warehouse Code
8195              2050
5232              1952
8196              1951
5239              1698
5234              1620
                  ... 
5116-Express         1
5999-Secondary       1
5169-Cooper          1
5469-Vans            1
5675                 1
Name: count, Length: 1563, dtype: int64

## Update current nlc margins + if is in stock or not

In [155]:
update_cols = ["current_nlc_margin_date", "current_nlc_margin", "Current walmart margin at NLC category","sku_sales_category"]
dfCurrentTestsAll = dfCurrentTestsAll.drop(columns=update_cols)
dfCurrentTestsAll = dfCurrentTestsAll.merge(dfOutputAll4[["SKU-Node","current_nlc_margin", "Current walmart margin at NLC category","sku_sales_category"]], how="left", left_on="SKU-Node", right_on="SKU-Node")
dfCurrentTestsAll["current_nlc_margin_date"] = today_str
dfCurrentTestsAll["sku_sales_category"] = dfCurrentTestsAll["sku_sales_category"].astype(str).replace("nan", "No sales")
dfCurrentTestsAll["Is in stock?"] = np.where(dfCurrentTestsAll["current_nlc_margin"].isna(), "No", "Yes")
dfCurrentTestsAll

,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,Last price update,SKU-Node,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date
0,ACCE0131318560V,6283261261,Normal strategy,11%,70.06,0.00,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,ACCE0131318560V-6283261261,NaN,NaN,No sales,2026-04-02
1,ACCE0131318560V,6283261264,Normal strategy,11%,70.76,0.00,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,ACCE0131318560V-6283261264,NaN,NaN,No sales,2026-04-02
2,ACCE0131318560V,6283261391,Normal strategy,11%,67.87,NaN,New,NaN,HUB,2026-01-05,8,11%,No,NaN,ACCE0131318560V-6283261391,NaN,NaN,No sales,2026-04-02
3,ACCE0131418555V,6283261391,Normal strategy,11%,56.13,-0.02,Decrease,NaN,HUB,2026-01-05,8,11%,No,NaN,ACCE0131418555V-6283261391,NaN,NaN,No sales,2026-04-02
4,ACCE0221316580T,6283261113,Normal strategy,11%,43.25,0.03,Increase,NaN,HUB,2026-01-05,8,11%,Yes,NaN,ACCE0221316580T-6283261113,0.1188,0% to 10%,95-100%,2026-04-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3354583,YOKO0391722565T,62832669,Increase test,6%,NaN,0.00,No change,NaN,HUB,2026-03-30,4,Increased,Yes,2026-03-31,YOKO0391722565T-62832669,0.1100,0% to 10%,80-90%,2026-04-02
3354584,YOKO0391722565T,628326922,Increase test,6%,NaN,0.00,No change,NaN,HUB,2026-03-30,4,Increased,Yes,2026-03-31,YOKO0391722565T-628326922,0.1060,0% to 10%,80-90%,2026-04-02
3354585,YOKO0401722555T,6283261177,Increase test,6%,NaN,0.00,No change,NaN,HUB,2026-03-30,4,Increased,Yes,2026-03-31,YOKO0401722555T-6283261177,0.1100,-10% to 0%,80-90%,2026-04-02
3354586,YOKO0401722555T,628326527,Increase test,6%,NaN,0.00,No change,NaN,HUB,2026-03-30,4,Increased,Yes,2026-03-31,YOKO0401722555T-628326527,0.1100,-10% to 0%,80-90%,2026-04-02


## Inv check

In [ ]:
# dfInvDate = pricing.get_inventory(today_str, add_rebates = False, amazon = False, greater3=True)
# dfInvDate1 = pricing.get_inventory('2026-03-30', add_rebates = False, amazon = False, greater3=True)

dfVendor = dw.get_vendor_code_table()
dfVendor = dfVendor.rename(columns={"warehouse_code": "Warehouse Code"})
dfVendor = dfVendor[["Warehouse Code", "vendor_code", "Vendor Group"]].copy()

dfInvComp = dfInvDate.merge(dfInvDate1, how="outer", on=["Product Code", "Warehouse Code"], suffixes=("_current", "_prev")).merge(dfVendor, how="left", on="Warehouse Code")
dfInvComp["Delta price%"] = round((dfInvComp["Purchase Price+FET_current"] - dfInvComp["Purchase Price+FET_prev"])/ dfInvComp["Purchase Price+FET_prev"],4)
dfInvComp["Delta price category"] = np.where(dfInvComp["Delta price%"] < -0.02, "Decrease",
                                              np.where(dfInvComp["Delta price%"] > 0.02, "Increase", np.where(dfInvComp["Delta price%"] == 0, "No change", 
                                                                                                           np.where(dfInvComp["Purchase Price+FET_prev"].isna(), "Only in current file", 
                                                                                                                     np.where(dfInvComp["Purchase Price+FET_current"].isna(), "Only in prev file", "Small price change (-1% to 1%)")))))
#keep only cols Product Code, Warehouse Code, Purchase Price+FET_13th, Purchase Price+FET_12th, Delta price%, Delta price category, Available_13th, Available_12th
dfInvComp = dfInvComp[["Product Code", "Warehouse Code", "Purchase Price+FET_current", "Purchase Price+FET_prev", "Delta price%", "Delta price category", "Available_current", "Available_prev","vendor_code","Vendor Group"]].copy()
dfInvComp["Brand code"] = dfInvComp["Product Code"].str[:4]
dfInvCompAnalysis = dfInvComp.groupby(["Delta price category"]).agg({"Product Code": "count", "Delta price%": "mean"}).reset_index().rename(columns={"Product Code": "Count SKU-Whs", "Delta price%": "Avg price change %"})
dfInvCompAnalysis["Avg price change %"] = dfInvCompAnalysis["Avg price change %"].round(4)
dfInvCompAnalysis

,Delta price category,Count SKU-Whs,Avg price change %
0,Decrease,168274,-0.0373
1,Increase,511659,0.0734
2,No change,1064161,0.0000
3,Only in current file,113493,NaN
4,Only in prev file,144200,NaN
5,Small price change (-1% to 1%),318464,0.0014


In [149]:
dfInvCompIncr.to_excel("Price increases above 1%.xlsx", index=False)

In [147]:
vendor_col = "vendor_code"
brand_col = "Brand code"
col_group = vendor_col
category = "Increase"

dfInvCompTotalGrouped = dfInvComp.groupby(col_group).agg({"Product Code": "count"}).rename(columns={"Product Code": "Total wh-sku lines"}).reset_index()
dfInvCompIncr = dfInvComp[dfInvComp["Delta price category"] ==category].copy()

dfInvCompIncrBrand = dfInvCompIncr.groupby(col_group).agg({"Product Code": "count","Delta price%": "mean"}).reset_index().rename(columns={"Product Code": f"Count of wh-sku price {category}","Delta price%": f"Avg price {category} %"})
dfInvCompIncrBrand = dfInvCompIncrBrand.sort_values(f"Count of wh-sku price {category}", ascending=False)
dfInvCompIncrBrand[f"Avg price {category} %"] = dfInvCompIncrBrand[f"Avg price {category} %"].round(3)
dfInvCompIncrBrand = dfInvCompIncrBrand.merge(dfInvCompTotalGrouped, how="left", on=col_group)
dfInvCompIncrBrand[f"% Lines {category}"] = round((dfInvCompIncrBrand[f"Count of wh-sku price {category}"] / dfInvCompIncrBrand["Total wh-sku lines"]),3)
# dfInvCompIncrBrand = dfInvCompIncrBrand.sort_values(f"% Lines {category}", ascending=False)
dfFilt = dfInvCompIncrBrand[dfInvCompIncrBrand["Total wh-sku lines"]>1000].copy()
dfFilt

,vendor_code,Count of wh-sku price Increase,Avg price Increase %,Total wh-sku lines,% Lines Increase
0,VTP,263120,0.085,384264,0.685
1,PST,92995,0.037,450580,0.206
2,GYR,62796,0.139,288326,0.218
3,COP,38853,0.019,190703,0.204
4,ATD,18723,0.014,163441,0.115
...,...,...,...,...,...
76,SGT,3,0.087,4067,0.001
79,WDT,2,0.080,1901,0.001
82,TTW,2,0.030,3309,0.001
83,GTD,1,0.128,1567,0.001


# Tests (no)

## DSVD shipping cost test

In [ ]:
# pathNodes = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_09 Documentation\Price tests docs & analysis\DSVD cost test\DSVD cost test v3.xlsx"
# dfDSVDTestAll = pd.read_excel(pathNodes, dtype={"node": str})
# dfDSVDTest = dfDSVDTestAll[dfDSVDTestAll["Target"] == "Yes"].copy()
# dfDSVDTest = dfDSVDTest.rename(columns={"node": "Identifier"})

# dfDSVDTest

,Identifier,Min date DSVD order,Max date DSVD order,Max date overall order,Days no DSVD orders,Total dsvd orders,Total orders after DSVD start date (node),Profit % before DSVD,Profit % after DSVD,Delta profit% overall,Profit % dsvd orders,Profit % not dsvd,Average unit shipping cost DSVD,% dsvd orders,% dsvd orders last 30 days,Average shipping overall last 30 days,Average shipping overall,Orders after DSVD date (all nodes),% of orders this node vs overall,Target
35,6283261032,2026-01-03,2026-03-12,2026-03-20,8.0,11.0,30.0,0.138901,0.104941,-0.033959,0.076943,0.126677,10.000000,0.366667,0.416667,4.166670,3.666667,272652.0,0.000110,Yes
36,6283261033,2025-12-26,2026-03-15,2026-03-15,0.0,25.0,44.0,0.116776,0.092474,-0.024302,0.081580,0.109021,10.000000,0.568182,0.769231,7.692310,5.681818,298042.0,0.000148,Yes
37,6283261034,2025-12-26,2026-03-12,2026-03-12,0.0,19.0,31.0,0.139994,0.102517,-0.037477,0.094411,0.116453,10.000000,0.612903,0.857143,8.571430,6.129032,298042.0,0.000104,Yes
41,628326104,2025-09-29,2026-03-20,2026-03-22,2.0,91.0,252.0,0.128246,0.093534,-0.034712,0.059164,0.111291,6.000000,0.361111,0.372093,2.232558,2.166667,642888.0,0.000392,Yes
72,6283261069,2025-12-27,2026-03-14,2026-03-14,0.0,22.0,48.0,0.142685,0.103240,-0.039446,0.092069,0.113753,10.000000,0.458333,0.642857,6.428570,4.583333,294707.0,0.000163,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1198,62832693,2025-09-24,2026-03-22,2026-03-22,0.0,306.0,507.0,NaN,0.081052,NaN,0.058468,0.113028,6.000000,0.603550,0.500000,3.000000,3.621302,658051.0,0.000770,Yes
1207,62832694,2025-09-25,2026-03-22,2026-03-22,0.0,248.0,433.0,NaN,0.085237,NaN,0.065225,0.110569,5.968421,0.572748,0.610390,3.643064,3.418403,654514.0,0.000662,Yes
1229,62832696,2025-09-24,2026-03-16,2026-03-22,6.0,176.0,316.0,NaN,0.090941,NaN,0.073279,0.113717,6.000000,0.556962,0.538462,3.230772,3.341772,658051.0,0.000480,Yes
1248,62832698,2025-09-28,2026-03-16,2026-03-22,6.0,133.0,220.0,0.071332,0.082878,0.011546,0.059733,0.117395,6.000000,0.604545,0.600000,3.600000,3.627273,645430.0,0.000341,Yes


In [ ]:
# dfDSVDTest["Average unit shipping cost DSVD"].mean()

7.754760502538071

In [ ]:
# dfDSVDTest["Average shipping overall last 30 days"].mean()

4.545345403233215

In [ ]:
# dfDSVDTestUseful = dfDSVDTest[["Identifier", "Average shipping overall last 30 days", "% of orders this node vs overall"]]
# dfDSVDTestUseful

,Identifier,Average shipping overall last 30 days,% of orders this node vs overall
35,6283261032,4.166670,0.000110
36,6283261033,7.692310,0.000148
37,6283261034,8.571430,0.000104
41,628326104,2.232558,0.000392
72,6283261069,6.428570,0.000163
...,...,...,...
1198,62832693,3.000000,0.000770
1207,62832694,3.643064,0.000662
1229,62832696,3.230772,0.000480
1248,62832698,3.600000,0.000341


In [ ]:
# dfTestAll = dfOutput[dfOutput["Identifier"].isin(dfDSVDTest["Identifier"])].copy()
# exclude_tests = ["Margin test", "Wm margin split test"]
# dfTest = dfTestAll[dfTestAll["Final target"].isin(exclude_tests) == False].copy()
# col_shipping_cost = "Average unit shipping cost DSVD" #
# col_shipping_cost = "Average shipping overall last 30 days"
# dfTest = dfTest.merge(dfDSVDTest[["Identifier", col_shipping_cost]], how="left", on="Identifier").rename(columns={col_shipping_cost: "Shipping cost DSVD"})
# dfTest["Test price"] = round(dfTest["Final node level cost"] + 1 * dfTest["Shipping cost DSVD"], 2)
# dfTest["Delta % test"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"], 3)
# dfTest["Delta % test category"] = np.where(dfTest["Delta % test"] < 0, "Decrease",
#                                                      np.where(dfTest["Delta % test"] > 0, "Increase", "No change"))


# dfTest["Delta % test category bins"] = pd.cut(dfTest["Delta % test"], bins=[-np.inf, -.1,  -.05, 0, .05, .1, np.inf], labels=["<-10%", "-10% to -5%", "-5% to 0%", "0% to 5%", "5% to 10%", ">10%"])

# dfTest = dfTest[dfTest["Delta % test"] > -0.03].copy()

# print("% revenue:",dfTest["sku_node_revenue"].sum()/total_revenue)   

# dfTest["Delta % test category bins"].value_counts()

# print(len(dfTest),"total lines")

# dfTestNode = dfTest.groupby(["Identifier","Shipping cost DSVD"]).agg({"sku_node_revenue": "sum", "Delta % test": "mean",
#                                                                       "SKU-Node": "count","Final price change %": "mean"
# }).reset_index()
# dfTestNode.sort_values("sku_node_revenue", ascending=True)

# price_change_test = dfTestNode["Delta % test"].mean()
# final_price_change = dfTestNode["Final price change %"].mean()
# print(f"Average price change for test price vs current NLC price: {price_change_test:.2%}")
# print(f"Average price change for final node level cost vs current NLC price: {final_price_change:.2%}")
# print("Delta between price changes:", round(price_change_test - final_price_change, 4))

0.03075764500698527
291097 total lines
Average price change for test price vs current NLC price: 2.38%
Average price change for final node level cost vs current NLC price: -0.89%
Delta between price changes: 0.0327


In [ ]:
# import pandas as pd
# import numpy as np

# df = dfTestNode[["Identifier", "sku_node_revenue", "Delta % test"]].copy()

# # Clean
# df["sku_node_revenue"] = pd.to_numeric(df["sku_node_revenue"], errors="coerce").fillna(0)
# df["Delta % test"] = pd.to_numeric(df["Delta % test"], errors="coerce").fillna(0)

# # Sort biggest nodes first so balancing works better
# df = df.sort_values("sku_node_revenue", ascending=False).reset_index(drop=True)

# group_a = []
# group_b = []

# rev_a = rev_b = 0.0
# nodes_a = nodes_b = 0

# # for weighted avg delta
# weighted_sum_a = weighted_sum_b = 0.0
# weight_a = weight_b = 0.0

# for _, row in df.iterrows():
#     rev = row["sku_node_revenue"]
#     delta = row["Delta % test"]

#     # simulate adding to A
#     sim_nodes_a = nodes_a + 1
#     sim_rev_a = rev_a + rev
#     sim_weighted_sum_a = weighted_sum_a + rev * delta
#     sim_weight_a = weight_a + rev

#     avg_a_if = sim_weighted_sum_a / sim_weight_a if sim_weight_a > 0 else 0
#     avg_b_now = weighted_sum_b / weight_b if weight_b > 0 else 0

#     score_if_a = (
#         abs(sim_nodes_a - nodes_b) +
#         abs(sim_rev_a - rev_b) / max(df["sku_node_revenue"].sum(), 1) +
#         abs(avg_a_if - avg_b_now)
#     )

#     # simulate adding to B
#     sim_nodes_b = nodes_b + 1
#     sim_rev_b = rev_b + rev
#     sim_weighted_sum_b = weighted_sum_b + rev * delta
#     sim_weight_b = weight_b + rev

#     avg_b_if = sim_weighted_sum_b / sim_weight_b if sim_weight_b > 0 else 0
#     avg_a_now = weighted_sum_a / weight_a if weight_a > 0 else 0

#     score_if_b = (
#         abs(nodes_a - sim_nodes_b) +
#         abs(rev_a - sim_rev_b) / max(df["sku_node_revenue"].sum(), 1) +
#         abs(avg_a_now - avg_b_if)
#     )

#     # assign to the group with better balance score
#     if score_if_a <= score_if_b:
#         group_a.append(row["Identifier"])
#         nodes_a = sim_nodes_a
#         rev_a = sim_rev_a
#         weighted_sum_a = sim_weighted_sum_a
#         weight_a = sim_weight_a
#     else:
#         group_b.append(row["Identifier"])
#         nodes_b = sim_nodes_b
#         rev_b = sim_rev_b
#         weighted_sum_b = sim_weighted_sum_b
#         weight_b = sim_weight_b

# # Final assignment
# df["test_group"] = np.where(df["Identifier"].isin(group_a), "A", "B")

# # Summary check
# summary = df.groupby("test_group").apply(
#     lambda x: pd.Series({
#         "nodes": x["Identifier"].nunique(),
#         "sku_node_revenue_sum": x["sku_node_revenue"].sum(),
#         "weighted_avg_delta": np.average(x["Delta % test"], weights=x["sku_node_revenue"])
#     })
# ).reset_index()

# print(summary)

  test_group  nodes  sku_node_revenue_sum  weighted_avg_delta
0          A   98.0            1394196.09            0.024922
1          B   98.0            1369995.94            0.024484


C:\Users\franc\AppData\Local\Temp\ipykernel_20344\4282595211.py:75: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df.groupby("test_group").apply(


In [ ]:
# dfTestFinal = dfTest.merge(df[["Identifier", "test_group"]], how="left", on="Identifier")
# # dfTestFinal where test_group is A Then Final test Price = Test price, else Final test Price = Final node level cost

# dfTestFinal["Sub-group"] = np.where(dfTestFinal["test_group"] == "A","Shipping cost added", "No shipping")

# dfTestFinal["Final test price"] = np.where(dfTestFinal["Sub-group"] == "Shipping cost added", dfTestFinal["Test price"], dfTestFinal["Final node level cost"])
# dfTestFinal["Final price change %"] = round((dfTestFinal["Final test price"] - dfTestFinal["current_nlc_price"])/dfTestFinal["current_nlc_price"], 3)
# dfTestFinal["Final price change category"] = np.where(dfTestFinal["Final price change %"] < 0, "Decrease",
#                                                      np.where(dfTestFinal["Final price change %"] > 0, "Increase", "No change"))

# #calculate sum of sku_node revenue by gruop, and final price change % average by group
# summary_df = dfTestFinal.groupby("Sub-group").agg({"sku_node_revenue": "sum", "Final price change %": "mean"}).reset_index()
# summary_df["Final price change %"] = summary_df["Final price change %"].round(4)
# summary_df

,Sub-group,sku_node_revenue,Final price change %
0,No shipping,1369995.94,-0.0079
1,Shipping cost added,1394196.09,0.0231


In [ ]:
# dfTestDSV= dfTestFinal[["Product Code", "Identifier","Final test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Final test price": "Price"})
# dfTestDSV

,SKU,Source,Price,SKU-Node
0,ACCE0221316580T,628326443,37.47,ACCE0221316580T-628326443
1,ACCE0221316580T,628326470,41.93,ACCE0221316580T-628326470
2,ACCE0221316580T,628326570,37.47,ACCE0221316580T-628326570
3,ACCE0221517560H,628326570,35.36,ACCE0221517560H-628326570
4,ACCE0221521565HXL,628326443,58.13,ACCE0221521565HXL-628326443
...,...,...,...,...
291092,YOKO39620351250E,628326173,339.45,YOKO39620351250E-628326173
291093,YOKO39620351250E,628326179,339.45,YOKO39620351250E-628326179
291094,YOKO39620351250E,62832693,339.45,YOKO39620351250E-62832693
291095,ZETA0021822550WXL,62832612,79.18,ZETA0021822550WXL-62832612


In [ ]:
# usecols = ["Product Code", "Identifier", "Final test price", "Final price change %", "Final price change category", "Node type", "Min units","Sub-group","Final node level cost category"]
# dfTestTracker = dfTestFinal[usecols].copy()

# dfTestTracker["Final target"] = "DSVD test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Final node level cost category": "Final price category",
#     "Final price change category": "Final price change category final",
#    "Final price change %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

,Product Code,Identifier,Final test price,Final delta vs current %,Final price change category final,Node type,Min units,Sub-group,Final price category,Final target,SKU-Node,Start date
0,ACCE0221316580T,628326443,37.47,-0.010,Decrease,HUB,8,No shipping,11%,DSVD test,ACCE0221316580T-628326443,2026-03-24
1,ACCE0221316580T,628326470,41.93,0.108,Increase,HUB,8,Shipping cost added,11%,DSVD test,ACCE0221316580T-628326470,2026-03-24
2,ACCE0221316580T,628326570,37.47,-0.010,Decrease,HUB,8,No shipping,11%,DSVD test,ACCE0221316580T-628326570,2026-03-24
3,ACCE0221517560H,628326570,35.36,-0.010,Decrease,HUB,8,No shipping,11%,DSVD test,ACCE0221517560H-628326570,2026-03-24
4,ACCE0221521565HXL,628326443,58.13,-0.010,Decrease,HUB,8,No shipping,11%,DSVD test,ACCE0221521565HXL-628326443,2026-03-24
...,...,...,...,...,...,...,...,...,...,...,...,...
291092,YOKO39620351250E,628326173,339.45,0.000,No change,SPOKE,4,No shipping,11%,DSVD test,YOKO39620351250E-628326173,2026-03-24
291093,YOKO39620351250E,628326179,339.45,0.000,No change,SPOKE,4,No shipping,11%,DSVD test,YOKO39620351250E-628326179,2026-03-24
291094,YOKO39620351250E,62832693,339.45,0.000,No change,SPOKE,4,No shipping,11%,DSVD test,YOKO39620351250E-62832693,2026-03-24
291095,ZETA0021822550WXL,62832612,79.18,0.052,Increase,HUB,4,Shipping cost added,11%,DSVD test,ZETA0021822550WXL-62832612,2026-03-24


## Test increase from 6-10% to 11%

In [ ]:

# dont_update_final_targets = ["Margin test", "Wm margin split test", "Shipping cost added","DSVD test"]


# #exclude sales category nan and 95-100%
# exclude_sales_categories = ["95-100%", np.nan]

# dfTest = dfOutput[(dfOutput["current_nlc_margin"]>= min_margin_update_prices)
#                                               & (dfOutput["current_nlc_margin"]< 0.1)
#                                               & (~dfOutput["Final target"].isin(dont_update_final_targets))
#                                               & (~dfOutput["sku_sales_category"].isin(exclude_sales_categories))].copy()

# dfTest["Test price"] = round(dfTest["Final node level cost"], 2)
# dfTest["Delta % test"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"], 3)
# dfTest["Delta % test category"] = np.where(dfTest["Delta % test"] < 0, "Decrease",
#                                                      np.where(dfTest["Delta % test"] > 0, "Increase", "No change"))


# #remove where is decrease
# dfTest = dfTest[dfTest["Delta % test"] >= 0].copy()

# dfTest["Delta % test category bins"] = pd.cut(dfTest["Delta % test"], bins=[-np.inf, -.1,  -.05, 0, .05, .1, np.inf], labels=["<-10%", "-10% to -5%", "-5% to 0%", "0% to 5%", "5% to 10%", ">10%"])

# dfTest = dfTest[dfTest["Delta % test"] > -0.03].copy()

# print("% revenue:",dfTest["sku_node_revenue"].sum()/total_revenue)   

# dfTest["Delta % test category bins"].value_counts()

# print(len(dfTest),"total lines")

# price_change_test = dfTest["Delta % test"].mean()
# final_price_change = dfTest["Final price change %"].mean()
# print(f"Average price change for test price vs current NLC price: {price_change_test:.2%}")
# print(f"Average price change for final node level cost vs current NLC price: {final_price_change:.2%}")
# print("Delta between price changes:", round(price_change_test - final_price_change, 4))

% revenue: 0.05658169133833221
101521 total lines
Average price change for test price vs current NLC price: 2.71%
Average price change for final node level cost vs current NLC price: 2.71%
Delta between price changes: 0.0


In [ ]:
# group_col = "Product Code"

# dfTestSKU = dfTest.groupby([group_col]).agg({"sku_node_revenue": "sum", "Delta % test": "mean",
#                                                                       "SKU-Node": "count","Final price change %": "mean"}).reset_index()


# df = dfTestSKU[[group_col, "sku_node_revenue", "Delta % test"]].copy()

# # Clean
# df["sku_node_revenue"] = pd.to_numeric(df["sku_node_revenue"], errors="coerce").fillna(0)
# df["Delta % test"] = pd.to_numeric(df["Delta % test"], errors="coerce").fillna(0)

# # Sort biggest nodes first so balancing works better
# df = df.sort_values("sku_node_revenue", ascending=False).reset_index(drop=True)

# group_a = []
# group_b = []

# rev_a = rev_b = 0.0
# nodes_a = nodes_b = 0

# # for weighted avg delta
# weighted_sum_a = weighted_sum_b = 0.0
# weight_a = weight_b = 0.0

# for _, row in df.iterrows():
#     rev = row["sku_node_revenue"]
#     delta = row["Delta % test"]

#     # simulate adding to A
#     sim_nodes_a = nodes_a + 1
#     sim_rev_a = rev_a + rev
#     sim_weighted_sum_a = weighted_sum_a + rev * delta
#     sim_weight_a = weight_a + rev

#     avg_a_if = sim_weighted_sum_a / sim_weight_a if sim_weight_a > 0 else 0
#     avg_b_now = weighted_sum_b / weight_b if weight_b > 0 else 0

#     score_if_a = (
#         abs(sim_nodes_a - nodes_b) +
#         abs(sim_rev_a - rev_b) / max(df["sku_node_revenue"].sum(), 1) +
#         abs(avg_a_if - avg_b_now)
#     )

#     # simulate adding to B
#     sim_nodes_b = nodes_b + 1
#     sim_rev_b = rev_b + rev
#     sim_weighted_sum_b = weighted_sum_b + rev * delta
#     sim_weight_b = weight_b + rev

#     avg_b_if = sim_weighted_sum_b / sim_weight_b if sim_weight_b > 0 else 0
#     avg_a_now = weighted_sum_a / weight_a if weight_a > 0 else 0

#     score_if_b = (
#         abs(nodes_a - sim_nodes_b) +
#         abs(rev_a - sim_rev_b) / max(df["sku_node_revenue"].sum(), 1) +
#         abs(avg_a_now - avg_b_if)
#     )

#     # assign to the group with better balance score
#     if score_if_a <= score_if_b:
#         group_a.append(row[group_col])
#         nodes_a = sim_nodes_a
#         rev_a = sim_rev_a
#         weighted_sum_a = sim_weighted_sum_a
#         weight_a = sim_weight_a
#     else:
#         group_b.append(row[group_col])
#         nodes_b = sim_nodes_b
#         rev_b = sim_rev_b
#         weighted_sum_b = sim_weighted_sum_b
#         weight_b = sim_weight_b

# # Final assignment
# df["test_group"] = np.where(df[group_col].isin(group_a), "A", "B")

# # Summary check
# summary = df.groupby("test_group").apply(
#     lambda x: pd.Series({
#         "skus": x[group_col].nunique(),
#         "sku_node_revenue_sum": x["sku_node_revenue"].sum(),
#         "weighted_avg_delta": np.average(x["Delta % test"], weights=x["sku_node_revenue"])
#     })
# ).reset_index()

# print(summary)

  test_group    skus  sku_node_revenue_sum  weighted_avg_delta
0          A  3127.0            2569057.18            0.024811
1          B  3127.0            2569065.44            0.024811


C:\Users\franc\AppData\Local\Temp\ipykernel_3208\541965670.py:78: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = df.groupby("test_group").apply(


In [ ]:
# dfTestFinal = dfTest.merge(df[[group_col, "test_group"]], how="left", on=group_col)
# # dfTestFinal where test_group is A Then Final test Price = Test price, else Final test Price = Final node level cost

# dfTestFinal["Sub-group"] = np.where(dfTestFinal["test_group"] == "A","Increased", "No change")

# dfTestFinal["Final test price"] = np.where(dfTestFinal["Sub-group"] == "Increased", dfTestFinal["Test price"], dfTestFinal["current_nlc_price"])
# dfTestFinal["Final price change %"] = round((dfTestFinal["Final test price"] - dfTestFinal["current_nlc_price"])/dfTestFinal["current_nlc_price"], 3)
# dfTestFinal["Final price change category"] = np.where(dfTestFinal["Final price change %"] < 0, "Decrease",
#                                                      np.where(dfTestFinal["Final price change %"] > 0, "Increase", "No change"))

# #calculate sum of sku_node revenue by gruop, and final price change % average by group
# summary_df = dfTestFinal.groupby("Sub-group").agg({"sku_node_revenue": "sum", "Final price change %": "mean"}).reset_index()
# summary_df["Final price change %"] = summary_df["Final price change %"].round(4)
# summary_df

,Sub-group,sku_node_revenue,Final price change %
0,Increased,2569057.18,0.0278
1,No change,2569065.44,0.0000


In [ ]:
# dfTestDSV= dfTestFinal[["Product Code", "Identifier","Final test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Final test price": "Price"})
# dfTestDSV

,SKU,Source,Price,SKU-Node
0,ACCE0221317570H,628326463,30.86,ACCE0221317570H-628326463
1,ACCE0221518560H,628326468,37.61,ACCE0221518560H-628326468
2,ACCE0221520560V,628326984,57.08,ACCE0221520560V-628326984
3,ACCE0221520570H,628326433,50.00,ACCE0221520570H-628326433
4,ACCE0221721565HXL,6283261264,70.53,ACCE0221721565HXL-6283261264
...,...,...,...,...
101516,ZENN0012026550VXL,62832614,113.83,ZENN0012026550VXL-62832614
101517,ZENN0012224530WXL,6283261551,84.78,ZENN0012224530WXL-6283261551
101518,ZENN0012226540VXL,628326435,108.64,ZENN0012226540VXL-628326435
101519,ZENN0041417565T,62832617,36.22,ZENN0041417565T-62832617


In [ ]:
# usecols = ["Product Code", "Identifier", "Final test price", "Final price change %", "Final price change category", "Node type", "Min units","Sub-group","Final node level cost category"]
# dfTestTracker = dfTestFinal[usecols].copy()

# dfTestTracker["Final target"] = "Increase test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Final node level cost category": "Final price category",
#     "Final price change category": "Final price change category final",
#    "Final price change %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

,Product Code,Identifier,Final test price,Final delta vs current %,Final price change category final,Node type,Min units,Sub-group,Final price category,Final target,SKU-Node,Start date
0,ACCE0221317570H,628326463,30.86,0.000,No change,HUB,8,No change,11%,Increase test,ACCE0221317570H-628326463,2026-03-30
1,ACCE0221518560H,628326468,37.61,0.039,Increase,HUB,8,Increased,11%,Increase test,ACCE0221518560H-628326468,2026-03-30
2,ACCE0221520560V,628326984,57.08,0.003,Increase,HUB,8,Increased,8%,Increase test,ACCE0221520560V-628326984,2026-03-30
3,ACCE0221520570H,628326433,50.00,0.000,No change,HUB,8,No change,11%,Increase test,ACCE0221520570H-628326433,2026-03-30
4,ACCE0221721565HXL,6283261264,70.53,0.018,Increase,HUB,8,Increased,11%,Increase test,ACCE0221721565HXL-6283261264,2026-03-30
...,...,...,...,...,...,...,...,...,...,...,...,...
101516,ZENN0012026550VXL,62832614,113.83,0.000,No change,HUB,4,Increased,6%,Increase test,ZENN0012026550VXL-62832614,2026-03-30
101517,ZENN0012224530WXL,6283261551,84.78,0.000,No change,HUB,4,Increased,8%,Increase test,ZENN0012224530WXL-6283261551,2026-03-30
101518,ZENN0012226540VXL,628326435,108.64,0.000,No change,HUB,4,No change,11%,Increase test,ZENN0012226540VXL-628326435,2026-03-30
101519,ZENN0041417565T,62832617,36.22,0.040,Increase,HUB,4,Increased,11%,Increase test,ZENN0041417565T-62832617,2026-03-30


## Test less sales

### Original

In [71]:
# dfOutputLessSales = dfOutput.merge(finalTargetWithGroupsMerge, how="left", on="Product Code")
# dfOutputLessSales = dfOutputLessSales[dfOutputLessSales["group"].notna()].copy()
# dfOutputLessSalesFilt = dfOutputLessSales[dfOutputLessSales["current_nlc_margin"]>=0.109].copy()
# dfOutputLessSalesFilt= dfOutputLessSalesFilt.merge(dfCurrentTests[["SKU-Node","Final target"]], how="left", left_on="SKU-Node", right_on="SKU-Node")
# dfOutputLessSalesFilt = dfOutputLessSalesFilt[dfOutputLessSalesFilt["Final target"] != "Margin test"].copy()
# print("Total rows with less sales:", len(dfOutputLessSales))
# display(dfOutputLessSales["current_nlc_margin category"].value_counts())
# print("Filtered:", len(dfOutputLessSalesFilt))
# display(dfOutputLessSalesFilt["current_nlc_margin category"].value_counts())
# display(dfOutputLessSalesFilt["group"].value_counts())

# dfOutputLessSalesFiltCheck = dfOutputLessSalesFilt.groupby(["current_nlc_margin category", "group"]).agg({"Product Code": "count"})

# dfOutputLessSalesFiltCheck = dfOutputLessSalesFiltCheck.rename(columns={"Product Code": "count"}).reset_index()
# # dfOutputLessSalesFiltCheck pivot by group
 
# dfOutputLessSalesFiltCheck = dfOutputLessSalesFiltCheck.pivot(index="current_nlc_margin category", columns="group", values="count").fillna(0).astype(int)

# dfOutputLessSalesFiltCheck

In [72]:
# dfOutputLessSalesFilt["New NLC"].value_counts()

In [73]:
# dfTest = dfOutputLessSalesFilt.copy()
# dfTest = dfTest[(dfTest["Target for node level cost? - 11% margin"] == "Yes") &
#                                       (dfTest["group"].notna()) &
#                                       (dfTest["current_nlc_price"].notna())].copy()

# print("Total rows in test:", len(dfTest))

# dfTest["Group"] = np.where(dfTest["group"] == 1, "10%",
#                                       np.where(dfTest["group"] == 2, "8%",
#                                               np.where(dfTest["group"] == 3, "6%", "No group")))
# dfTest["Test price"]= np.where(dfTest["group"] == 1, dfTest["Node level cost - 10% margin"],
#                                       np.where(dfTest["group"] == 2, dfTest["Node level cost - 8% margin"],
#                                               np.where(dfTest["group"] == 3, dfTest["Node level cost - 6% margin"], np.nan)))

# dfTest["Delta test to current NLC %"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Delta test to current NLC category"] = np.where(dfTest["Delta test to current NLC %"] < 0, "Decrease",
#                                                                 np.where(dfTest["Delta test to current NLC %"] > 0, "Increase", "No change"))

# relevantCols = ["Product Code", "Identifier","SKU-Node", "Group", "Test price", "Delta test to current NLC %", "Delta test to current NLC category",
#                     "Node type", "Min units","current_nlc_margin","qty_pct_change_L2W_vs_P90"]



# dfTest = dfTest[relevantCols].copy().rename(columns = {"current_nlc_margin": "NLC margin before update"})

# # Now display # of increases, decreases, no change by each Group + average of "Delta test to current NLC %" by group
# summary = (
#     dfTest
#         .groupby("Group")
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )
# summary

In [74]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [75]:
# usecols = ["Product Code", "Identifier","Group", "Test price", "Delta test to current NLC category", "Delta test to current NLC %", 
#            "Node type", "Min units","NLC margin before update","qty_pct_change_L2W_vs_P90"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Less sales"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Group": "Final price category",
#     "Delta test to current NLC category": "Final price change category final",
#     "Delta test to current NLC %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [76]:
# dfTestTracker.to_csv("Test.csv", index=False)

In [77]:
# dfCurrentTestsAllCheck = dfCurrentTestsAll.copy()
# dfCurrentTestsAllCheck["SKU-Node"] = dfCurrentTestsAllCheck["Product Code"] + "-" + dfCurrentTestsAllCheck["Identifier"]

# df = dfCurrentTestsAllCheck[dfCurrentTestsAllCheck["SKU-Node"].isin(dfOutputLessSalesFilt["SKU-Node"])].copy()#["Final target"].value_counts()
# dfMargin = df[df["Final target"] == "Added"].copy()
# dfMargin["Start date"].value_counts()

### Pivot back

In [78]:
# dfPrevDSVOriginal = dfPrevDSVOriginal.rename(columns={"sku": "Product Code",  "source":"Identifier", "walmart_dsv_price": "prev_nlc_price"}).drop(columns=["date"])
# dfPrevDSVOriginal

In [79]:
# dfTestLessSales = dfCurrentTestsAll[(dfCurrentTestsAll["Final target"] == "Less sales")& (dfCurrentTestsAll["Start date"] == "2026-02-11")].copy()
# dfTest = dfTestLessSales[dfTestLessSales["Final price category"].isin(["6%", "8%"])].copy()
# dfTest = dfTest.merge(dfPrevDSVOriginal, how="left", on=["Product Code", "Identifier"])
# dfTest["SKU-Node"] = dfTest["Product Code"] + "-" + dfTest["Identifier"]
# dfTest

In [80]:
# dfTestDSV = dfTest[["Product Code", "Identifier","prev_nlc_price"]].copy().rename(columns={"Product Code": "SKU", "Identifier": "Source", "prev_nlc_price": "Price"})
# dfTestDSV

In [81]:
# usecols = ["Product Code", "Identifier","prev_nlc_price", "Final price change category final","Final price category",
#            "Node type", "Min units","qty_pct_change_L2W_vs_P90"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Less sales back"
# dfTestTracker["Final price change category final"] = "No change"

# dfTestTracker = dfTestTracker.rename(columns={
#     "prev_nlc_price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

### Revert test

In [82]:
# change = ["Less sales","Less sales back"]

# dfOutputChange = dfOutput[dfOutput["Final target"].isin(change) == True].copy()

# display(dfOutputChange["Final price change category"].value_counts())


In [83]:
# #dfOutputChange have an idea by each Final target, how many are Increase, Decrease, No change in terms of price change category, and average of price change % by Final target
# summary = (
#     dfOutputChange.groupby("Final target")
#                  .agg(
#                      count=("Final price change category", "count"),
#                      increases=("Final price change category", lambda x: (x == "Increase").sum()),
#                      decreases=("Final price change category", lambda x: (x == "Decrease").sum()),
#                      no_change=("Final price change category", lambda x: (x == "No change").sum()),
#                      avg_price_change_pct=("Final price change %", "mean")
#                  )
#                  .reset_index()
# )
# summary

In [84]:
# dfTest = dfOutputChange.copy()
# # dfTest["Test price"] where Final price change category = Increase then Test price = Final node level cost, else = current_nlc_price
# dfTest["Final target"] = "Normal strategy"
# dfTest["Final price"] = dfTest.apply(lambda row: row["Final node level cost"] if row["Final price change category"] == "Increase" else row["current_nlc_price"], axis=1)
# dfTest["Final delta vs current %"] = round((dfTest["Final price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Final price change category final"] = np.where(dfTest["Final delta vs current %"] < 0, "Decrease", np.where(dfTest["Final delta vs current %"] > 0, "Increase", "No change"))
# relevantCols = ["Product Code", "Identifier","Final target","Final price", "Final delta vs current %", "Final price change category final", "Node type", "Min units"]

# dfTest = dfTest[relevantCols].copy()
# dfTest

In [85]:
# dfTestDSV = dfTest[["Product Code", "Identifier","Final price"]].copy().rename(columns={"Product Code": "SKU", "Identifier": "Source", "Final price": "Price"})
# dfTestDSV["SKU-Node"] = dfTestDSV["SKU"] + "-" + dfTestDSV["Source"].astype(str)
# dfTestDSV

In [86]:
# dfTestTracker = dfTest.copy()

# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

## Test high margin by brand

In [87]:
# pathAssignments = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_09 Documentation\Price tests docs & analysis\Incr margin brand phase 2\pricing_test_assignments.xlsx"
# read_cols = ["brand_code","price_level_pct","allocation_pct","test_group"]
# dfAssignments = pd.read_excel(pathAssignments, sheet_name="assignments_long", dtype={"SKU-Node": str}, usecols = read_cols)
# dfAssignments = dfAssignments.sort_values(["brand_code","allocation_pct","price_level_pct","test_group"], ascending=[True, False, True, True]).reset_index(drop=True)
# dfAssignments

In [88]:
#create dataframe where just 2 colums, allocation_pct and group, where for allocation_pct = 0.5 then group = "group_1", for allocation_pct = 0.3 then group = "Group 2", for allocation_pct = 0.2 then group = "Group 3"

In [89]:
# dfOutputTarget = dfOutput[dfOutput["Brand code"].isin(dfAssignments["brand_code"].unique())].copy()

# # dfOutputTarget = dfOutput.copy()


# exclude_targets =["Shipping cost added", "Wm margin split test"]
# dfOutputTarget = dfOutputTarget[~dfOutputTarget["Final target"].isin(exclude_targets)]


# target_col = "Target for node level cost? - 14% margin"
# dfOutputTarget = dfOutputTarget[dfOutputTarget[target_col] == "Yes"].copy()
# dfOutputTarget = dfOutputTarget[dfOutputTarget["current_nlc_margin"] < 0.16].copy()
# dfOutputTarget = dfOutputTarget[dfOutputTarget["sku_revenue"] > 500].copy()

# # dfOutputTarget = dfOutputTarget[dfOutputTarget["current_nlc_margin"] < 0.15].copy()
# print("Total sku-nodes", len(dfOutputTarget))

# dfOutputTargetAnalysis = dfOutputTarget.groupby("Brand code").agg({"SKU-Node": "count", "sku_node_revenue": "sum"}).reset_index().rename(columns={"SKU-Node": "Count SKU-Node", "sku_node_revenue": "Total inv amount"})
# dfOutputTargetAnalysis["Total inv amount"] = dfOutputTargetAnalysis["Total inv amount"].round(2)
# dfOutputTargetAnalysis["% inv amount"] = round(dfOutputTargetAnalysis["Total inv amount"] / total_revenue,3)
# print("Total % inv amt sku-node:", dfOutputTargetAnalysis["% inv amount"].sum().round(3))
# perc_rev_sku = dfSKURevenue[dfSKURevenue["Product Code"].isin(dfOutputTarget["Product Code"].unique()) == True]["sku_revenue"].sum()/total_revenue
# print("Total % inv amt sku:", round(perc_rev_sku,3))
# dfOutputTargetAnalysis.sort_values("% inv amount", ascending=False)

In [90]:
# import pandas as pd
# import numpy as np

# # Copy
# df = dfOutputTarget.copy()

# # ----------------------------
# # 1) Build SKU-level table
# # ----------------------------
# def mode_or_nan(x):
#     m = x.mode(dropna=True)
#     return m.iloc[0] if len(m) > 0 else np.nan

# sku_summary = (
#     df.groupby("Product Code")
#       .apply(lambda g: pd.Series({
#           "Brand code": mode_or_nan(g["Brand code"]),
#           "sku_sales_category": mode_or_nan(g["sku_sales_category"]),
#           "Current walmart margin at NLC category": mode_or_nan(g["Current walmart margin at NLC category"]),
#           "current_nlc_margin category": mode_or_nan(g["current_nlc_margin category"]),
#           "num_sku_nodes": g["SKU-Node"].nunique()
#       }))
#       .reset_index()
# )

# # ----------------------------
# # 2) Assign groups within each Brand code + sku_sales_category
# # ----------------------------
# # For every Brand code, split each sku_sales_category bucket:
# #   50% -> group_1
# #   25% -> group_2
# #   25% -> group_3
# # Then concatenate all brands together.

# def assign_groups_within_brand_and_sales_category(g):
#     g = g.sample(frac=1, random_state=42).copy()

#     n = len(g)
#     n_g1 = int(round(n * 0.50))
#     n_g2 = int(round(n * 0.25))
#     n_g3 = n - n_g1 - n_g2

#     groups = (
#         ["group_1"] * n_g1 +
#         ["group_2"] * n_g2 +
#         ["group_3"] * n_g3
#     )

#     g["test_group"] = groups
#     return g

# sku_assignment = (
#     sku_summary
#       .groupby(["Brand code", "sku_sales_category"], group_keys=False)
#       .apply(assign_groups_within_brand_and_sales_category)
#       .reset_index(drop=True)
# )

# # ----------------------------
# # 3) Merge assignment back to SKU-Nodes
# # ----------------------------
# dfTestSelectionGrouped = df.merge(
#     sku_assignment[["Product Code", "test_group"]],
#     on="Product Code",
#     how="left"
# )

# # ----------------------------
# # 4) Checks
# # ----------------------------
# print("SKU counts by group:")
# print(sku_assignment["test_group"].value_counts(dropna=False))

# print("\nSKU share by group:")
# print(sku_assignment["test_group"].value_counts(normalize=True, dropna=False))

# print("\nSplit of Brand code across groups:")
# print(
#     pd.crosstab(
#         sku_assignment["Brand code"],
#         sku_assignment["test_group"],
#         margins=True
#     )
# )

# print("\nRow-wise share of Brand code across groups:")
# print(
#     pd.crosstab(
#         sku_assignment["Brand code"],
#         sku_assignment["test_group"],
#         normalize="index"
#     )
# )

# print("\nSplit of sku_sales_category across groups:")
# print(
#     pd.crosstab(
#         [sku_assignment["Brand code"], sku_assignment["sku_sales_category"]],
#         sku_assignment["test_group"]
#     )
# )

# print("\nOther category balance check:")
# print(
#     pd.crosstab(
#         [
#             sku_assignment["Brand code"],
#             sku_assignment["Current walmart margin at NLC category"],
#             sku_assignment["current_nlc_margin category"]
#         ],
#         sku_assignment["test_group"]
#     )
# )

# # Optional revenue check
# sku_assignment = sku_assignment.merge(
#     dfSKURevenue[["Product Code", "sku_revenue"]],
#     how="left",
#     on="Product Code"
# )

# sku_assignment = sku_assignment.sort_values("sku_revenue", ascending=False)

# dfTestSelectionGrouped

In [91]:
# # sku_assignment group by test_group and get the counts for the different columns, e.g. sku_sales_category, Current walmart margin at NLC category, current_nlc_margin category, num_sku_nodes
# summary = (
#     sku_assignment.groupby("test_group")
#                .agg(
#                     total_revenue =("sku_revenue", "sum"),
#                    count_skus=("Product Code", "count"),
#                    avg_num_sku_nodes=("num_sku_nodes", "mean"),
#                    sales_category_dist=("sku_sales_category", lambda x: x.value_counts(normalize=True).to_dict()),
#                    walmart_margin_dist=("Current walmart margin at NLC category", lambda x: x.value_counts(normalize=True).to_dict()),
#                    nlc_margin_dist=("current_nlc_margin category", lambda x: x.value_counts(normalize=True).to_dict())
#                )
#                .reset_index()
# )

# summary 

In [92]:
# # dfTest = dfOutputAll.merge(finalGroups, how="left", on="Product Code")
# # dfTest = dfTest[(dfTest["Target for node level cost? - 15% margin"] == "Yes") &
# #                                       (dfTest["group"].notna()) &
# #                                       (dfTest["current_nlc_price"].notna())].copy()
# # dfTest
# # dfTest["Group"] = np.where(dfTest["group"] == 1, "11%",
# #                                       np.where(dfTest["group"] == 2, "13%",
# #                                               np.where(dfTest["group"] == 3, "15%", "No group")))
# # dfTest["Test price"]= np.where(dfTest["group"] == 1, dfTest["Node level cost - 11% margin"],
# #                                       np.where(dfTest["group"] == 2, dfTest["Node level cost - 13% margin"],
# #                                               np.where(dfTest["group"] == 3, dfTest["Node level cost - 15% margin"], np.nan)))


# dfTest = dfOutputTarget.merge(
#     sku_assignment[["Product Code", "test_group"]],
#     on="Product Code",
#     how="left"
# )
# dfTest["brand_code"] = dfTest["Product Code"].str[:4]

# dfTest = dfTest.merge(dfAssignments[["brand_code", "test_group","price_level_pct"]], how="left", on=["brand_code", "test_group"])

# dfTest['price_level_pct'] = dfTest['price_level_pct'].astype(str)

# dfTest["Test price"] =  np.where(dfTest['price_level_pct'] == "11", dfTest["Node level cost - 11% margin"],
#                                        np.where(dfTest['price_level_pct'] == "12", dfTest["Node level cost - 12% margin"],
#                                                np.where(dfTest['price_level_pct'] == "13", dfTest["Node level cost - 13% margin"], 
#                                                         np.where(dfTest['price_level_pct'] == "14", dfTest["Node level cost - 14% margin"], np.nan))))

# dfTest["Delta test to current NLC %"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Delta test to current NLC category"] = np.where(dfTest["Delta test to current NLC %"] < 0, "Decrease",
#                                                                 np.where(dfTest["Delta test to current NLC %"] > 0, "Increase", "No change"))

# # relevantCols = ["Product Code", "Identifier","SKU-Node", "price_level_pct", "Test price", "Delta test to current NLC %", "Delta test to current NLC category",
# #                     "Node type", "Min units"]
# # dfTest = dfTest[relevantCols].copy()



In [93]:

# summary = (
#     dfTest
#         .groupby(["price_level_pct"])
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )

# # Now display # of increases, decreases, no change by each Group + average of "Delta test to current NLC %" by group
# summary_brand = (
#     dfTest
#         .groupby(["brand_code","price_level_pct"])
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )
# display(summary)
# display(summary_brand)

In [94]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [95]:
# dfTest

In [96]:
# dfCurrentTestsAll

In [97]:
# usecols = ["Product Code", "Identifier","price_level_pct", "Test price", "Delta test to current NLC category", "Delta test to current NLC %", "Node type", "Min units"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Margin test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "price_level_pct": "Final price category",
#     "Delta test to current NLC category": "Final price change category final",
#     "Delta test to current NLC %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [98]:
# dfTestTracker["Final price category"].value_counts()

## Margin test by walmart margin

In [99]:
# dfOutputSalesAll = dfOutput.merge(dfSalesFiltered[["Product Code", "Sales category","revenue"]], how="left", on="Product Code")
# dfOutputSales = dfOutputSalesAll[dfOutputSalesAll["Sales category"].isin(["Top 20%", "Top 50%", "Top 80%"])].copy()
# dfOutputSalesTest = dfOutputSales[dfOutputSales["Current walmart margin at NLC"] > 0.13].copy()
# dfOutputSalesTest["Total margin"] = round(dfOutputSalesTest["Current walmart margin at NLC"] + dfOutputSalesTest["current_nlc_margin"],4)
# dfOutputSalesTest["New Walmart margin test"] = round(dfOutputSalesTest["Total margin"]/2,4)
# dfOutputSalesTest["New test price"] = round(dfOutputSalesTest["offer_price"] * (1 - dfOutputSalesTest["New Walmart margin test"]),2)
# dfOutputSalesTest["Price change with new test price %"] = round((dfOutputSalesTest["New test price"] - dfOutputSalesTest["current_nlc_price"])/ dfOutputSalesTest["current_nlc_price"], 3)

# dfOutputSalesTest["Price change with new test price % category"] = pd.cut(dfOutputSalesTest["Price change with new test price %"], bins=[0.01, 0.03,0.06,0.1, np.inf], labels=["1% to 3%", "3% to 6%", "6% to 10%", ">10%"])

# dfOutputSalesTest["Delta price change normal vs test"] = round(dfOutputSalesTest["Price change with new test price %"] - dfOutputSalesTest["Final price change %"], 3)

# dfOutputSalesTest["Delta price change normal vs test category"] = pd.cut(dfOutputSalesTest["Delta price change normal vs test"], bins=[0.01, 0.03,0.06,0.1, np.inf], labels=["1% to 3%", "3% to 6%", "6% to 10%", ">10%"])


# dfOutputSalesTest["New test price category"] = np.where(dfOutputSalesTest["Delta price change normal vs test"] < 0, "Decrease",
#                                                      np.where(dfOutputSalesTest["Delta price change normal vs test"] > 0, "Increase", "No change"))

# dfOutputSalesTest["New Walmart margin test 2"] = round(dfOutputSalesTest["Total margin"]*0.4,4)
# dfOutputSalesTest["New test price 2"] = round(dfOutputSalesTest["offer_price"] * (1 - dfOutputSalesTest["New Walmart margin test 2"]),2)
# dfOutputSalesTest["Price change with new test price % 2"] = round((dfOutputSalesTest["New test price 2"] - dfOutputSalesTest["current_nlc_price"])/ dfOutputSalesTest["current_nlc_price"], 3)



# dfTest = dfOutputSalesTest[(dfOutputSalesTest["Delta price change normal vs test"] > 0.01) &
#                            (dfOutputSalesTest["Delta price change normal vs test"] < 0.1) &
#                            (dfOutputSalesTest["Price change with new test price %"] > 0.01) &
#                            (dfOutputSalesTest["Delta price change normal vs test"] < 0.1) &
#                            (dfOutputSalesTest["current_nlc_margin"]> 0.08)].copy()


# dfTest = dfTest.merge(dfSalesSKUNodeGrouped, how="left", on=["SKU-Node"])

# print("Total SKU-Nodes above 13% wm margin:", len(dfTest))
# display(dfTest["Final target"].value_counts())
# display(dfTest["current_nlc_margin category"].value_counts())
# print("Mean current Tires Easy NLC (node level cost) margin:", dfTest["current_nlc_margin"].mean().round(4))
# print("Mean price change with new test price %:", dfTest["Price change with new test price %"].mean().round(4))
# print("Mean price change with new test price % 2:", dfTest["Price change with new test price % 2"].mean().round(4))

# print("Mean delta price change normal vs test:", dfTest["Delta price change normal vs test"].mean().round(4))
# print("Revenue contribution:", dfTest["total_inv_amount"].sum()/total_rev)
# display(dfTest["Final price change category"].value_counts())

In [100]:
# usefulCols = ["Product Code", "Identifier","SKU-Node", "total_inv_amount", "Sales category","Current walmart margin at NLC category","current_nlc_margin category",
#               "Price change with new test price % category","current_nlc_price","Final price change category","Final node level cost", "New test price", "New test price 2",
#               "Node type", "Min units"]
# dfTestSelection = dfTest[usefulCols].copy()
# dfTestSelection

In [101]:
# import pandas as pd
# import numpy as np

# # Copy
# df = dfTestSelection.copy()

# # ----------------------------
# # 1) Build SKU-level table
# # ----------------------------
# # Use Product Code as the SKU key
# # For stratification, keep the main categorical dimensions at SKU level.
# # Since a SKU can have multiple SKU-Nodes, we summarize numeric weight and
# # take the most representative category using weighted mode by inventory amount.

# def weighted_mode(x, weights):
#     tmp = pd.DataFrame({"x": x, "w": weights})
#     tmp = tmp.dropna(subset=["x"])
#     if len(tmp) == 0:
#         return np.nan
#     return tmp.groupby("x", dropna=False)["w"].sum().idxmax()

# sku_summary = (
#     df.groupby("Product Code", as_index=False)
#       .apply(lambda g: pd.Series({
#           "sku_weight": g["total_inv_amount"].sum(),
#           "Sales category": weighted_mode(g["Sales category"], g["total_inv_amount"]),
#           "Current walmart margin at NLC category": weighted_mode(g["Current walmart margin at NLC category"], g["total_inv_amount"]),
#           "current_nlc_margin category": weighted_mode(g["current_nlc_margin category"], g["total_inv_amount"]),
#           "Price change with new test price % category": weighted_mode(g["Price change with new test price % category"], g["total_inv_amount"]),
#           "num_sku_nodes": g["SKU-Node"].nunique()
#       }))
#       .reset_index(drop=True)
# )

# # ----------------------------
# # 2) Create stratification key
# # ----------------------------
# # This keeps balance across your important dimensions.
# strat_cols = [
#     "Sales category",
#     "Current walmart margin at NLC category",
#     "current_nlc_margin category",
#     "Price change with new test price % category"
# ]

# sku_summary["stratum"] = sku_summary[strat_cols].astype(str).agg(" | ".join, axis=1)

# # ----------------------------
# # 3) Assign groups within each stratum
# # ----------------------------
# # Group sizes:
# #   group 1 = 50%
# #   group 2 = 25%
# #   group 3 = 25%
# #
# # We randomize at SKU level, not SKU-Node level.

# rng = np.random.default_rng(42)

# def assign_groups_within_stratum(g):
#     g = g.sample(frac=1, random_state=42).copy()  # shuffle rows
    
#     n = len(g)
#     n_g1 = int(round(n * 0.50))
#     n_g2 = int(round(n * 0.25))
#     n_g3 = n - n_g1 - n_g2
    
#     groups = (
#         ["group_1"] * n_g1 +
#         ["group_2"] * n_g2 +
#         ["group_3"] * n_g3
#     )
    
#     # In case rounding creates mismatch
#     groups = groups[:n]
#     if len(groups) < n:
#         groups += ["group_3"] * (n - len(groups))
    
#     g["test_group"] = groups
#     return g

# sku_assignment = (
#     sku_summary.groupby("stratum", group_keys=False)
#                .apply(assign_groups_within_stratum)
#                .reset_index(drop=True)
# )

# # ----------------------------
# # 4) Merge assignment back to SKU-Nodes
# # ----------------------------
# dfTestSelectionGrouped = df.merge(
#     sku_assignment[["Product Code", "test_group"]],
#     on="Product Code",
#     how="left"
# )

# # ----------------------------
# # 5) Quick checks
# # ----------------------------
# print("SKU counts by group:")
# print(
#     sku_assignment["test_group"].value_counts(dropna=False)
# )

# print("\nSKU share by group:")
# print(
#     sku_assignment["test_group"].value_counts(normalize=True, dropna=False)
# )

# print("\nWeighted inventory share by group:")
# print(
#     sku_assignment.groupby("test_group")["sku_weight"].sum() /
#     sku_assignment["sku_weight"].sum()
# )

# print("\nSKU-Node counts by group:")
# print(
#     dfTestSelectionGrouped.groupby("test_group")["SKU-Node"].nunique()
# )

# dfTestSelectionGrouped.head()

In [102]:
# for col in strat_cols:
#     print(f"\nDistribution for {col}")
#     display(pd.crosstab(sku_assignment["test_group"], sku_assignment[col], normalize="index"))

In [103]:
# dfTestSelectionGrouped["Current walmart margin at NLC category"].value_counts()

In [104]:
# import numpy as np

# df = dfTestSelectionGrouped.copy()

# # ----------------------------
# # Sub-group labels
# # ----------------------------
# df["Sub-group"] = np.select(
#     [
#         df["test_group"] == "group_1",
#         df["test_group"] == "group_2",
#         df["test_group"] == "group_3",
#     ],
#     [
#         "Baseline",
#         "50% split",
#         "60% split",
#     ],
#     default=np.nan
# )

# # ----------------------------
# # Test price logic
# # group_1 -> baseline price only when baseline says Increase
# #            otherwise keep current price
# # group_2 -> New test price
# # group_3 -> New test price 2
# # ----------------------------
# df["Test price"] = np.select(
#     [
#         (df["test_group"] == "group_1") & (df["Final price change category"] == "Increase"),
#         (df["test_group"] == "group_1") & (df["Final price change category"] != "Increase"),
#         df["test_group"] == "group_2",
#         df["test_group"] == "group_3",
#     ],
#     [
#         df["Final node level cost"],   # baseline price
#         df["current_nlc_price"],       # keep current price when baseline is not Increase
#         df["New test price"],          # 50% split
#         df["New test price 2"],        # 60% split
#     ],
#     default=np.nan
# )

# # ----------------------------
# # Test price change %
# # vs current price
# # ----------------------------
# df["Test price change %"] = (df["Test price"] / df["current_nlc_price"]) - 1

# # Optional: percentage points / percent formatting helper
# df["Test price change % pct"] = df["Test price change %"] * 100

# # ----------------------------
# # Quick checks
# # ----------------------------
# print(df[["test_group", "Sub-group"]].drop_duplicates().sort_values("test_group"))

# print("\nCounts by Sub-group:")
# print(df["Sub-group"].value_counts(dropna=False))

# print("\nTest price summary by Sub-group:")
# print(
#     df.groupby("Sub-group")["Test price change %"]
#       .agg(["count", "mean", "median", "min", "max"])
#       .sort_index()
# )

# df

In [105]:
# df["Test price change % category"] =np.where(df["Test price change %"] < 0, "Decrease",
#                                                      np.where(df["Test price change %"] > 0, "Increase", "No change"))

# df["Test price change % category"].value_counts()

In [106]:
# #df by Sub-group I want split of "Test price change % category" within each Sub-group + average of "Test price change %"
# summary = (
#     df.groupby("Sub-group")
#       .agg(
#           count=("Test price change % category", "count"),
#           increases=("Test price change % category", lambda x: (x == "Increase").sum()),
#           decreases=("Test price change % category", lambda x: (x == "Decrease").sum()),
#           no_change=("Test price change % category", lambda x: (x == "No change").sum()),
#           avg_test_price_change_pct=("Test price change %", "mean")
#       )
#       .reset_index()
# )
# summary

In [107]:
# dfTest = df.copy()
# dfTest

In [108]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [109]:
# usecols = ["Product Code", "Identifier","Sub-group", "Test price", "Final price change category", "Test price change %", "Node type", "Min units",
#            "Sales category"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Wm margin split test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Group": "Final price category",
#     "Final price change category": "Final price change category final",
#     "Test price change %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [110]:
# dfTestCheckMAP = dfTest.merge(dfMAP, how = "left", on = "Product Code")

In [111]:
# dfTestCheckMAP["New price above MAP"] = dfTestCheckMAP["Test price"] > dfTestCheckMAP["MAP"] 

# dfTestCheckMAP["New price +5% above MAP"] = dfTestCheckMAP["Test price"]*1.05 > dfTestCheckMAP["MAP"] 

# dfTestCheckMAP["New price +5% above MAP"].value_counts()

In [112]:
# dfTestCheckMAP[dfTestCheckMAP["MAP"].notna()]

## Cost + shipping

In [113]:
# test = True
# dfOutputCostNode = dfOutput[dfOutput["Identifier"].isin(dfCostNode["Identifier"].unique())].copy()
# dfOutputCostNode = dfOutputCostNode.merge(dfCostNode, how="left", on="Identifier")

# cost_column = "Final node level cost + shipping"

# dfOutputCostNode[cost_column] = dfOutputCostNode["Final node level cost"] + dfOutputCostNode["Shipping cost"]

# price_change_col = "Final price change %"
# price_change_cat_col = "Final price change category"

# dfOutputCostNode[price_change_col] = round((dfOutputCostNode[cost_column] - dfOutputCostNode["current_nlc_price"])/dfOutputCostNode["current_nlc_price"],4)
# dfOutputCostNode[price_change_cat_col] = np.where(dfOutputCostNode[price_change_col] < 0, "Decrease",
#                                                         np.where(dfOutputCostNode[price_change_col] > 0, "Increase", "No change"))
# dfOutputCostNode[price_change_col].mean()

# dfTest = dfOutputCostNode.copy()

In [114]:
# dfTestDSV= dfTest[["Product Code", "Identifier",cost_column,"SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", cost_column: "Price"})
# dfTestDSV

In [115]:
# usecols = ["Product Code", "Identifier",cost_column, price_change_cat_col, price_change_col, 
#            "Node type", "Min units","Final node level cost category"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Shipping cost added"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Final node level cost category":"Final price category",
#     price_change_cat_col: "Final price change category final",
#     price_change_col: "Final delta vs current %",
#     cost_column: "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

# Regular updates

## Prices increases test

In [84]:
dfPriceIncr = dfOutput[dfOutput["Final target"] == "Increase test"].copy()
# If sub gruop = Increased then Final node level cost, else do the maximum of the current_nlc_price and the Node level cost - 6% margin
dfPriceIncr["Price"] = np.where(dfPriceIncr["Sub-group"] == "Increased", dfPriceIncr["Final node level cost"],
                                np.where(dfPriceIncr["current_nlc_margin"] < 0.06, dfPriceIncr["Final node level cost"], dfPriceIncr["current_nlc_price"]))

dfPriceIncr["Price change %"] = round((dfPriceIncr["Price"] - dfPriceIncr["current_nlc_price"])/dfPriceIncr["current_nlc_price"],4)
dfPriceIncr["Price change category"] = np.where(dfPriceIncr["Price change %"] < 0, "Decrease",
                                                     np.where(dfPriceIncr["Price change %"] > 0, "Increase", "No change"))

# dfWmMarginSplitDSVUpdate only where Price change % in aboslute is greater than 1%
dfPriceIncrUpdate = dfPriceIncr[abs(dfPriceIncr["Price change %"]) >= 0.01].copy()

print("Total SKU-Nodes to update in DSV:", len(dfPriceIncrUpdate))

#But do avg price change % only for the ones where Price change category is Increase and the same for decreases
summary = (
    dfPriceIncrUpdate.groupby("Sub-group")
                   .agg(
                       count=("Price change category", "count"),
                       increases=("Price change category", lambda x: (x == "Increase").sum()),
                       decreases=("Price change category", lambda x: (x == "Decrease").sum()),
                       no_change=("Price change category", lambda x: (x == "No change").sum()),
                       avg_price_change_pct_increase=("Price change %", lambda x: x[dfPriceIncrUpdate["Price change category"] == "Increase"].mean()),
                       avg_price_change_pct_decrease=("Price change %", lambda x: x[dfPriceIncrUpdate["Price change category"] == "Decrease"].mean())
                   )
                   .reset_index()
)
summary

Total SKU-Nodes to update in DSV: 21226


,Sub-group,count,increases,decreases,no_change,avg_price_change_pct_increase,avg_price_change_pct_decrease
0,Increased,16821,10529,6292,0,0.031774,-0.028308
1,No change,4405,4405,0,0,0.077400,NaN


In [85]:
dfPriceIncrDSV = dfPriceIncrUpdate[["Product Code", "Identifier","Price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Price": "Price"})
dfPriceIncrDSV

,SKU,Source,Price,SKU-Node
2560,ACCE0551521575S,628326916,82.10,ACCE0551521575S-628326916
3382,ACCE0671722555WXL,628326984,71.59,ACCE0671722555WXL-628326984
3697,ACCE0672024535YXL,6283261151,73.37,ACCE0672024535YXL-6283261151
3700,ACCE0672024535YXL,62832645,73.37,ACCE0672024535YXL-62832645
4894,ADVN0131825565T,628326686,141.49,ADVN0131825565T-628326686
...,...,...,...,...
2201403,YOKO3961626570T,628326890,183.29,YOKO3961626570T-628326890
2203217,YOKO3961827565H,628326922,227.13,YOKO3961827565H-628326922
2203319,YOKO3961828575E,62832628,339.29,YOKO3961828575E-62832628
2203340,YOKO3961828575E,628326916,316.85,YOKO3961828575E-628326916


In [86]:
dfPriceIncrTracker = dfPriceIncrUpdate[["SKU-Node"]].copy()
dfPriceIncrTracker = dfPriceIncrTracker.merge(dfCurrentTestsAll, on="SKU-Node", how="left")
dfPriceIncrTracker["Last price update"] = today_str
dfPriceIncrTracker

,SKU-Node,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,Last price update,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date
0,ACCE0551521575S-628326916,ACCE0551521575S,628326916,Increase test,11%,NaN,0.023,Increase,NaN,HUB,2026-03-30,8,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
1,ACCE0671722555WXL-628326984,ACCE0671722555WXL,628326984,Increase test,8%,NaN,0.000,No change,NaN,HUB,2026-03-30,8,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
2,ACCE0672024535YXL-6283261151,ACCE0672024535YXL,6283261151,Increase test,8%,NaN,0.000,No change,NaN,HUB,2026-03-30,8,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
3,ACCE0672024535YXL-62832645,ACCE0672024535YXL,62832645,Increase test,8%,NaN,0.000,No change,NaN,HUB,2026-03-30,8,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
4,ADVN0131825565T-628326686,ADVN0131825565T,628326686,Increase test,11%,NaN,0.031,Increase,NaN,SPOKE,2026-03-30,8,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21221,YOKO3961626570T-628326890,YOKO3961626570T,628326890,Increase test,11%,NaN,0.030,Increase,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
21222,YOKO3961827565H-628326922,YOKO3961827565H,628326922,Increase test,11%,NaN,0.030,Increase,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
21223,YOKO3961828575E-62832628,YOKO3961828575E,62832628,Increase test,6%,NaN,0.000,No change,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
21224,YOKO3961828575E-628326916,YOKO3961828575E,628326916,Increase test,8%,NaN,0.000,No change,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN


## DSVD test

In [87]:
pathNodes = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_09 Documentation\Price tests docs & analysis\DSVD cost test\DSVD cost test v3.xlsx"
dfDSVDTestAll = pd.read_excel(pathNodes, dtype={"node": str})
dfDSVDTest = dfDSVDTestAll[dfDSVDTestAll["Target"] == "Yes"].copy()
dfDSVDTest = dfDSVDTest.rename(columns={"node": "Identifier"})

dfDSVDTestUseful = dfDSVDTest[["Identifier", "Average shipping overall last 30 days"]]
dfDSVDTestUseful = dfDSVDTestUseful.rename(columns={"Average shipping overall last 30 days": "Shipping cost DSVD"})
dfDSVDTestUseful

,Identifier,Shipping cost DSVD
35,6283261032,4.166670
36,6283261033,7.692310
37,6283261034,8.571430
41,628326104,2.232558
72,6283261069,6.428570
...,...,...
1198,62832693,3.000000
1207,62832694,3.643064
1229,62832696,3.230772
1248,62832698,3.600000


In [88]:
dfDSVD = dfOutput[dfOutput["Final target"] == "DSVD test"].copy()

dfDSVD = dfDSVD.merge(dfDSVDTestUseful, how="left", on="Identifier")

# dfDSVD create column Price where if Sub-group = 60% split then do Price margin split 60%, if Sub-group = 50% split then do Price margin split 50%, if Sub-group = Baseline then Price = Final node level cost when Final price change category = Increase, else Price = current_nlc_price
dfDSVD["Price"] = np.where(dfDSVD["Sub-group"] == "No shipping", dfDSVD["Final node level cost"],
                                    np.where(dfDSVD["Sub-group"] == "Shipping cost added", dfDSVD["Final node level cost"] + dfDSVD["Shipping cost DSVD"],
                                                      dfDSVD["current_nlc_price"]))

dfDSVD["Price change %"] = round((dfDSVD["Price"] - dfDSVD["current_nlc_price"])/dfDSVD["current_nlc_price"],4)
dfDSVD["Price change category"] = np.where(dfDSVD["Price change %"] < 0, "Decrease",
                                                     np.where(dfDSVD["Price change %"] > 0, "Increase", "No change"))

# dfDSVDUpdate only where Price change % in aboslute is greater than 1%
dfDSVDUpdate = dfDSVD[abs(dfDSVD["Price change %"]) >= 0.01].copy()

print("Total SKU-Nodes to update in DSV:", len(dfDSVDUpdate))

#But do avg price change % only for the ones where Price change category is Increase and the same for decreases
summary = (
    dfDSVDUpdate.groupby("Sub-group")
                   .agg(
                       count=("Price change category", "count"),
                       increases=("Price change category", lambda x: (x == "Increase").sum()),
                       decreases=("Price change category", lambda x: (x == "Decrease").sum()),
                       no_change=("Price change category", lambda x: (x == "No change").sum()),
                       avg_price_change_pct_increase=("Price change %", lambda x: x[dfDSVDUpdate["Price change category"] == "Increase"].mean()),
                       avg_price_change_pct_decrease=("Price change %", lambda x: x[dfDSVDUpdate["Price change category"] == "Decrease"].mean())
                   )
                   .reset_index()
)
summary

Total SKU-Nodes to update in DSV: 103971


,Sub-group,count,increases,decreases,no_change,avg_price_change_pct_increase,avg_price_change_pct_decrease
0,No shipping,53919,34453,19466,0,0.036619,-0.034115
1,Shipping cost added,50052,33105,16947,0,0.032552,-0.031942


In [120]:
dfDSVDDSV = dfDSVDUpdate[["Product Code", "Identifier","Price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Price": "Price"})
dfDSVDDSV

,SKU,Source,Price,SKU-Node
0,ACCE0221316580T,628326443,43.380000,ACCE0221316580T-628326443
1,ACCE0221316580T,628326470,47.839066,ACCE0221316580T-628326470
2,ACCE0221316580T,628326570,43.380000,ACCE0221316580T-628326570
3,ACCE0322026550VXL,628326443,116.220000,ACCE0322026550VXL-628326443
4,ACCE0322026550VXL,628326570,116.220000,ACCE0322026550VXL-628326570
...,...,...,...,...
257773,WHEELWTIS03120108X650,628326104,196.130000,WHEELWTIS03120108X650-628326104
257868,YOKO20819524570H,628326443,448.280000,YOKO20819524570H-628326443
257876,YOKO34519524570H,628326470,429.099066,YOKO34519524570H-628326470
257877,YOKO34519528570H,628326470,475.919066,YOKO34519528570H-628326470


In [90]:
dfDSVDTracker = dfDSVDUpdate[["SKU-Node"]].copy()
dfDSVDTracker = dfDSVDTracker.merge(dfCurrentTestsAll, on="SKU-Node", how="left")
dfDSVDTracker["Last price update"] = today_str
dfDSVDTracker

,SKU-Node,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,Last price update,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date
0,ACCE0221316580T-628326443,ACCE0221316580T,628326443,DSVD test,11%,NaN,-0.010,Decrease,NaN,HUB,2026-03-24,8,No shipping,Yes,2026-04-02,0.1100,15% to 20%,95-100%,2026-03-30
1,ACCE0221316580T-628326470,ACCE0221316580T,628326470,DSVD test,11%,NaN,0.108,Increase,NaN,HUB,2026-03-24,8,Shipping cost added,Yes,2026-04-02,0.2046,0% to 10%,95-100%,2026-03-30
2,ACCE0221316580T-628326570,ACCE0221316580T,628326570,DSVD test,11%,NaN,-0.010,Decrease,NaN,HUB,2026-03-24,8,No shipping,Yes,2026-04-02,0.1100,15% to 20%,95-100%,2026-03-30
3,ACCE0322026550VXL-628326443,ACCE0322026550VXL,628326443,DSVD test,11%,NaN,-0.010,Decrease,NaN,HUB,2026-03-24,8,No shipping,Yes,2026-04-02,0.1100,15% to 20%,95-100%,2026-03-30
4,ACCE0322026550VXL-628326570,ACCE0322026550VXL,628326570,DSVD test,11%,NaN,-0.010,Decrease,NaN,HUB,2026-03-24,8,No shipping,Yes,2026-04-02,0.1100,15% to 20%,95-100%,2026-03-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103966,WHEELWTIS03120108X650-628326104,WHEELWTIS03120108X650,628326104,DSVD test,11%,NaN,0.000,No change,NaN,HUB,2026-03-24,4,No shipping,Yes,2026-04-02,0.1100,>=30%,No sales,2026-03-30
103967,YOKO20819524570H-628326443,YOKO20819524570H,628326443,DSVD test,11%,NaN,0.000,No change,NaN,HUB,2026-03-24,4,No shipping,Yes,2026-04-02,0.1100,15% to 20%,No sales,2026-03-30
103968,YOKO34519524570H-628326470,YOKO34519524570H,628326470,DSVD test,11%,NaN,0.056,Increase,NaN,HUB,2026-03-24,4,Shipping cost added,Yes,2026-04-02,0.1188,20% to 25%,No sales,2026-03-30
103969,YOKO34519528570H-628326470,YOKO34519528570H,628326470,DSVD test,11%,NaN,-0.000,No change,NaN,HUB,2026-03-24,4,Shipping cost added,Yes,2026-04-02,0.1180,15% to 20%,No sales,2026-03-30


## Walmart margin split test update

In [91]:
dfWmMarginSplit = dfOutput[dfOutput["Final target"] == "Wm margin split test"].copy()
dfWmMarginSplit["Sub-group"].value_counts()
# dfWmMarginSplit create column Price where if Sub-group = 60% split then do Price margin split 60%, if Sub-group = 50% split then do Price margin split 50%, if Sub-group = Baseline then Price = Final node level cost when Final price change category = Increase, else Price = current_nlc_price
dfWmMarginSplit["Price"] = np.where(dfWmMarginSplit["Sub-group"] == "60% split", dfWmMarginSplit["Price margin split 60%"],
                                    np.where(dfWmMarginSplit["Sub-group"] == "50% split", dfWmMarginSplit["Price margin split 50%"],
                                             np.where((dfWmMarginSplit["Sub-group"] == "Baseline") & (dfWmMarginSplit["Final price change category"] == "Increase"), dfWmMarginSplit["Final node level cost"],
                                                      dfWmMarginSplit["current_nlc_price"])))

dfWmMarginSplit["Price change %"] = round((dfWmMarginSplit["Price"] - dfWmMarginSplit["current_nlc_price"])/dfWmMarginSplit["current_nlc_price"],4)
dfWmMarginSplit["Price change category"] = np.where(dfWmMarginSplit["Price change %"] < 0, "Decrease",
                                                     np.where(dfWmMarginSplit["Price change %"] > 0, "Increase", "No change"))

# dfWmMarginSplitDSVUpdate only where Price change % in aboslute is greater than 1%
dfWmMarginSplitUpdate = dfWmMarginSplit[abs(dfWmMarginSplit["Price change %"]) >= 0.01].copy()

print("Total SKU-Nodes to update in DSV:", len(dfWmMarginSplitUpdate))

#But do avg price change % only for the ones where Price change category is Increase and the same for decreases
summary = (
    dfWmMarginSplitUpdate.groupby("Sub-group")
                   .agg(
                       count=("Price change category", "count"),
                       increases=("Price change category", lambda x: (x == "Increase").sum()),
                       decreases=("Price change category", lambda x: (x == "Decrease").sum()),
                       no_change=("Price change category", lambda x: (x == "No change").sum()),
                       avg_price_change_pct_increase=("Price change %", lambda x: x[dfWmMarginSplit["Price change category"] == "Increase"].mean()),
                       avg_price_change_pct_decrease=("Price change %", lambda x: x[dfWmMarginSplit["Price change category"] == "Decrease"].mean())
                   )
                   .reset_index()
)
summary

Total SKU-Nodes to update in DSV: 21447


,Sub-group,count,increases,decreases,no_change,avg_price_change_pct_increase,avg_price_change_pct_decrease
0,50% split,6862,2513,4349,0,0.035785,-0.033196
1,60% split,7170,4321,2849,0,0.031220,-0.031254
2,Baseline,7415,7415,0,0,0.050595,NaN


In [92]:
dfWmMarginSplitDSV = dfWmMarginSplitUpdate[["Product Code", "Identifier","Price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Price": "Price"})
dfWmMarginSplitDSV

,SKU,Source,Price,SKU-Node
114,ACCE0221418570H,6283261123,44.08,ACCE0221418570H-6283261123
121,ACCE0221418570H,62832627,44.08,ACCE0221418570H-62832627
264,ACCE0221520560V,628326443,46.72,ACCE0221520560V-628326443
266,ACCE0221520560V,628326460,45.57,ACCE0221520560V-628326460
270,ACCE0221520560V,628326470,46.72,ACCE0221520560V-628326470
...,...,...,...,...
2176927,WEST1211623575T,628326437,91.48,WEST1211623575T-628326437
2176932,WEST1211623575T,628326759,91.48,WEST1211623575T-628326759
2176933,WEST1211623575T,628326770,91.48,WEST1211623575T-628326770
2194767,YOKO1811722565H,628326922,159.48,YOKO1811722565H-628326922


In [93]:
dfWmMarginSplitTracker = dfWmMarginSplitUpdate[["SKU-Node"]].copy()
dfWmMarginSplitTracker = dfWmMarginSplitTracker.merge(dfCurrentTestsAll, on="SKU-Node", how="left")
dfWmMarginSplitTracker["Last price update"] = today_str
dfWmMarginSplitTracker

,SKU-Node,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,Last price update,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date
0,ACCE0221418570H-6283261123,ACCE0221418570H,6283261123,Wm margin split test,NaN,39.80,0.000000,Decrease,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1319,20% to 25%,80-90%,2026-03-30
1,ACCE0221418570H-62832627,ACCE0221418570H,62832627,Wm margin split test,NaN,39.80,0.000000,Decrease,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1319,20% to 25%,80-90%,2026-03-30
2,ACCE0221520560V-628326443,ACCE0221520560V,628326443,Wm margin split test,NaN,44.08,0.023213,Increase,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1100,20% to 25%,70-80%,2026-03-30
3,ACCE0221520560V-628326460,ACCE0221520560V,628326460,Wm margin split test,NaN,44.35,0.000000,Decrease,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1154,20% to 25%,70-80%,2026-03-30
4,ACCE0221520560V-628326470,ACCE0221520560V,628326470,Wm margin split test,NaN,44.08,0.023213,Increase,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1100,20% to 25%,70-80%,2026-03-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21442,WEST1211623575T-628326437,WEST1211623575T,628326437,Wm margin split test,NaN,86.49,0.000000,No change,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1100,10% to 15%,70-80%,2026-03-30
21443,WEST1211623575T-628326759,WEST1211623575T,628326759,Wm margin split test,NaN,86.49,0.000000,No change,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1100,10% to 15%,70-80%,2026-03-30
21444,WEST1211623575T-628326770,WEST1211623575T,628326770,Wm margin split test,NaN,86.49,0.000000,No change,NaN,HUB,2026-03-09,4,Baseline,Yes,2026-04-02,0.1100,10% to 15%,70-80%,2026-03-30
21445,YOKO1811722565H-628326922,YOKO1811722565H,628326922,Wm margin split test,NaN,157.90,0.000000,No change,NaN,HUB,2026-03-09,8,Baseline,Yes,2026-04-02,0.1100,0% to 10%,30-40%,2026-03-30


## Brands margin test update

In [94]:
dfMarginTest = dfOutput[(dfOutput["Final target"] == "Margin test") & (dfOutput["Start date"].isin(["2026-03-12"]))].copy()

# dfMarginTest create column Price where if Sub-group = 11% then do Node level cost - 11% margin, if Sub-group = 12% then do Node level cost - 12% margin, if Sub-group = 13% then do Node level cost - 13% margin, if Sub-group = 14% then do Node level cost - 14% margin, else do current_nlc_price , if 15%  then do Node level cost - 15% margin
dfMarginTest["Price"] = np.where(dfMarginTest["Sub-group"] == "11%", dfMarginTest["Node level cost - 11% margin"],
                                 np.where(dfMarginTest["Sub-group"] == "12%", dfMarginTest["Node level cost - 12% margin"],
                                         np.where(dfMarginTest["Sub-group"] == "13%", dfMarginTest["Node level cost - 13% margin"],
                                                 np.where(dfMarginTest["Sub-group"] == "14%", dfMarginTest["Node level cost - 14% margin"],
                                                         np.where(dfMarginTest["Sub-group"] == "15%", dfMarginTest["Node level cost - 15% margin"], dfMarginTest["current_nlc_price"])))))


dfMarginTest["Price change %"] = round((dfMarginTest["Price"] - dfMarginTest["current_nlc_price"])/dfMarginTest["current_nlc_price"],4)
dfMarginTest["Price change category"] = np.where(dfMarginTest["Price change %"] < 0, "Decrease",
                                                     np.where(dfMarginTest["Price change %"] > 0, "Increase", "No change"))

# dfWmMarginSplitDSVUpdate only where Price change % in aboslute is greater than 1%
dfMarginTestUpdate = dfMarginTest[abs(dfMarginTest["Price change %"]) >= 0.01].copy()

print("Total SKU-Nodes to update in DSV:", len(dfMarginTestUpdate))

#But do avg price change % only for the ones where Price change category is Increase and the same for decreases
summary = (
    dfMarginTestUpdate.groupby("Sub-group")
                   .agg(
                       count=("Price change category", "count"),
                       increases=("Price change category", lambda x: (x == "Increase").sum()),
                       decreases=("Price change category", lambda x: (x == "Decrease").sum()),
                       no_change=("Price change category", lambda x: (x == "No change").sum()),
                       avg_price_change_pct_increase=("Price change %", lambda x: x[dfMarginTestUpdate["Price change category"] == "Increase"].mean()),
                       avg_price_change_pct_decrease=("Price change %", lambda x: x[dfMarginTestUpdate["Price change category"] == "Decrease"].mean())
                   )
                   .reset_index()
)

summary

Total SKU-Nodes to update in DSV: 67482


,Sub-group,count,increases,decreases,no_change,avg_price_change_pct_increase,avg_price_change_pct_decrease
0,11%,27236,20919,6317,0,0.071406,-0.038884
1,12%,7,0,7,0,NaN,-0.034157
2,13%,24382,13097,11285,0,0.069384,-0.033506
3,14%,15857,10720,5137,0,0.083320,-0.038310


In [95]:
dfMarginTestDSV = dfMarginTestUpdate[["Product Code", "Identifier","Price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Price": "Price"})
dfMarginTestDSV

,SKU,Source,Price,SKU-Node
51,ACCE0221417565H,6283261123,39.01,ACCE0221417565H-6283261123
57,ACCE0221417565H,62832627,39.01,ACCE0221417565H-62832627
58,ACCE0221417565H,62832628,33.97,ACCE0221417565H-62832628
60,ACCE0221417565H,628326443,42.10,ACCE0221417565H-628326443
64,ACCE0221417565H,628326470,42.10,ACCE0221417565H-628326470
...,...,...,...,...
1992345,NEXE1632025555VXL,6283261318,171.82,NEXE1632025555VXL-6283261318
1992346,NEXE1632025555VXL,6283261337,171.82,NEXE1632025555VXL-6283261337
1992347,NEXE1632025555VXL,6283261347,171.82,NEXE1632025555VXL-6283261347
1992445,NEXE1632025555VXL,628326693,171.82,NEXE1632025555VXL-628326693


In [96]:
dfMarginTestTracker = dfMarginTestUpdate[["SKU-Node"]].copy()
dfMarginTestTracker = dfMarginTestTracker.merge(dfCurrentTestsAll, on="SKU-Node", how="left")
dfMarginTestTracker["Last price update"] = today_str
dfMarginTestTracker

,SKU-Node,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,Last price update,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date
0,ACCE0221417565H-6283261123,ACCE0221417565H,6283261123,Margin test,13%,34.86,0.0125,Increase,NaN,HUB,2026-03-12,8,13%,Yes,2026-04-02,0.1299,>=30%,80-90%,2026-03-30
1,ACCE0221417565H-62832627,ACCE0221417565H,62832627,Margin test,13%,34.86,0.0125,Increase,NaN,HUB,2026-03-12,8,13%,Yes,2026-04-02,0.1299,>=30%,80-90%,2026-03-30
2,ACCE0221417565H-62832628,ACCE0221417565H,62832628,Margin test,13%,34.34,0.0226,Increase,NaN,HUB,2026-03-12,8,13%,Yes,2026-04-02,0.1299,>=30%,80-90%,2026-03-30
3,ACCE0221417565H-628326443,ACCE0221417565H,628326443,Margin test,13%,33.18,0.0128,Increase,NaN,HUB,2026-03-12,8,13%,Yes,2026-04-02,0.1299,>=30%,80-90%,2026-03-30
4,ACCE0221417565H-628326470,ACCE0221417565H,628326470,Margin test,13%,33.18,0.0128,Increase,NaN,HUB,2026-03-12,8,13%,Yes,2026-04-02,0.1299,>=30%,80-90%,2026-03-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67477,NEXE1632025555VXL-6283261318,NEXE1632025555VXL,6283261318,Margin test,11%,177.53,0.0464,Increase,NaN,SPOKE,2026-03-12,4,11%,Yes,2026-04-02,0.1100,0% to 10%,80-90%,2026-03-30
67478,NEXE1632025555VXL-6283261337,NEXE1632025555VXL,6283261337,Margin test,11%,177.53,0.0464,Increase,NaN,SPOKE,2026-03-12,4,11%,Yes,2026-04-02,0.1100,0% to 10%,80-90%,2026-03-30
67479,NEXE1632025555VXL-6283261347,NEXE1632025555VXL,6283261347,Margin test,11%,177.53,0.0464,Increase,NaN,SPOKE,2026-03-12,4,11%,Yes,2026-04-02,0.1100,0% to 10%,80-90%,2026-03-30
67480,NEXE1632025555VXL-628326693,NEXE1632025555VXL,628326693,Margin test,11%,177.53,0.0464,Increase,NaN,SPOKE,2026-03-12,4,11%,Yes,2026-04-02,0.1100,0% to 10%,80-90%,2026-03-30


## Low price updates (to show inv/correct our profit.)

In [97]:
dfCurrDSVNLCCheck = dfOutput[["Product Code", "Identifier","Purchase Price+FET","Node level cost - 6% margin", "Final target",
                                                 "Final node level cost","Final node level cost category","Node type", "Min units","current_nlc_margin"]].copy()

dont_update_final_targets = ["Margin test", "Wm margin split test", "Shipping cost added","DSVD test","Increase test"]

dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCheck[(dfCurrDSVNLCCheck["current_nlc_margin"]< min_margin_update_prices)
                                              & (~dfCurrDSVNLCCheck["Final target"].isin(dont_update_final_targets))].copy()


dfCurrDSVNLCCPriceUpdates["Category inventory"] = np.where(dfCurrDSVNLCCPriceUpdates["current_nlc_margin"]< 0.04, "Not showing inventory", "Below 6% margin")
dfCurrDSVNLCCPriceUpdates["SKU-Node"] = dfCurrDSVNLCCPriceUpdates["Product Code"] + "-" + dfCurrDSVNLCCPriceUpdates["Identifier"].astype(str)


if test:
    dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[dfCurrDSVNLCCPriceUpdates["SKU-Node"].isin(dfTest["SKU-Node"]) == False].copy()
# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[dfCurrDSVNLCCPriceUpdates["SKU-Node"].isin(dfTrackerLessSalesChng["SKU-Node"]) == False].copy()


# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[~dfCurrDSVNLCCPriceUpdates["Product Code"].str.contains("MICH|BFGO")].copy()


dfCurrDSVNLCCPriceUpdates

,Product Code,Identifier,Purchase Price+FET,Node level cost - 6% margin,Final target,Final node level cost,Final node level cost category,Node type,Min units,current_nlc_margin,Category inventory,SKU-Node
10,ACCE0221316580T,628326590,38.61,41.07,Normal strategy,43.38,11%,HUB,8,-0.0201,Not showing inventory,ACCE0221316580T-628326590
318,ACCE0221521565HXL,6283261123,61.09,64.99,No sales test,66.40,8%,HUB,8,0.0115,Not showing inventory,ACCE0221521565HXL-6283261123
321,ACCE0221521565HXL,62832627,61.09,64.99,No sales test,66.40,8%,HUB,8,0.0115,Not showing inventory,ACCE0221521565HXL-62832627
382,ACCE0221522560V,628326869,56.26,59.85,Added,63.21,11%,HUB,8,0.0440,Below 6% margin,ACCE0221522560V-628326869
1170,ACCE0322028550WXL,628326590,103.95,110.59,Normal strategy,116.80,11%,HUB,8,-0.0137,Not showing inventory,ACCE0322028550WXL-628326590
...,...,...,...,...,...,...,...,...,...,...,...,...
2198350,YOKO2052027565E,628326570,340.94,362.70,Decreased - margin > 20%,383.08,11%,HUB,4,0.0150,Not showing inventory,YOKO2052027565E-628326570
2198444,YOKO20520331250E,628326570,354.42,377.04,Decreased - margin > 20%,385.24,8%,HUB,4,0.0137,Not showing inventory,YOKO20520331250E-628326570
2198495,YOKO20520351250E,628326590,344.52,366.51,Decreased - margin > 20%,387.10,11%,HUB,4,0.0113,Not showing inventory,YOKO20520351250E-628326590
2201518,YOKO3961627570H,6283261134,180.18,198.68,Added,209.45,11%,HUB,4,0.0083,Not showing inventory,YOKO3961627570H-6283261134


In [98]:
usecols = ["Product Code", "Identifier", "Final node level cost", "Final node level cost category", "Category inventory", "Node type", "Min units"]
dfCurrDSVNLCCPriceUpdatesforTracker = dfCurrDSVNLCCPriceUpdates[usecols].copy()
dfCurrDSVNLCCPriceUpdatesforTracker["Final target"] = "Updated"

dfCurrDSVNLCCPriceUpdatesforTracker = dfCurrDSVNLCCPriceUpdatesforTracker.rename(columns={
    "Final node level cost category": "Final price category",
    "Final node level cost": "Final price"})
dfCurrDSVNLCCPriceUpdatesforTracker["SKU-Node"] = dfCurrDSVNLCCPriceUpdatesforTracker["Product Code"] + "-" + dfCurrDSVNLCCPriceUpdatesforTracker["Identifier"].astype(str)
dfCurrDSVNLCCPriceUpdatesforTracker["Min units"] = dfCurrDSVNLCCPriceUpdatesforTracker["Min units"].astype(int)
dfCurrDSVNLCCPriceUpdatesforTracker["Start date"] = date_str
dfCurrDSVNLCCPriceUpdatesforTracker

,Product Code,Identifier,Final price,Final price category,Category inventory,Node type,Min units,Final target,SKU-Node,Start date
10,ACCE0221316580T,628326590,43.38,11%,Not showing inventory,HUB,8,Updated,ACCE0221316580T-628326590,2026-04-02
318,ACCE0221521565HXL,6283261123,66.40,8%,Not showing inventory,HUB,8,Updated,ACCE0221521565HXL-6283261123,2026-04-02
321,ACCE0221521565HXL,62832627,66.40,8%,Not showing inventory,HUB,8,Updated,ACCE0221521565HXL-62832627,2026-04-02
382,ACCE0221522560V,628326869,63.21,11%,Below 6% margin,HUB,8,Updated,ACCE0221522560V-628326869,2026-04-02
1170,ACCE0322028550WXL,628326590,116.80,11%,Not showing inventory,HUB,8,Updated,ACCE0322028550WXL-628326590,2026-04-02
...,...,...,...,...,...,...,...,...,...,...
2198350,YOKO2052027565E,628326570,383.08,11%,Not showing inventory,HUB,4,Updated,YOKO2052027565E-628326570,2026-04-02
2198444,YOKO20520331250E,628326570,385.24,8%,Not showing inventory,HUB,4,Updated,YOKO20520331250E-628326570,2026-04-02
2198495,YOKO20520351250E,628326590,387.10,11%,Not showing inventory,HUB,4,Updated,YOKO20520351250E-628326590,2026-04-02
2201518,YOKO3961627570H,6283261134,209.45,11%,Not showing inventory,HUB,4,Updated,YOKO3961627570H-6283261134,2026-04-02


In [99]:
dfCurrDSVNLCCPriceUpdates["Brand code"] = dfCurrDSVNLCCPriceUpdates["Product Code"].str[:4]
dfCurrDSVNLCCPriceUpdates["Brand code"].value_counts()

Brand code
GENE    15919
CONT    13153
FIRE     6737
FALK     6382
BRID     5624
        ...  
ZETA        1
CRLS        1
SAMS        1
RONE        1
LION        1
Name: count, Length: 113, dtype: int64

In [100]:
dfNLCPriceUpdatesFinal = dfCurrDSVNLCCPriceUpdates[["Product Code", "Identifier", "Final node level cost"]].rename(
                        columns={"Identifier": "Source","Final node level cost":"Price","Product Code":"SKU"})

dfNLCPriceUpdatesFinal["SKU-Node"] = dfNLCPriceUpdatesFinal["SKU"] + "-" + dfNLCPriceUpdatesFinal["Source"]
dfNLCPriceUpdatesFinal

,SKU,Source,Price,SKU-Node
10,ACCE0221316580T,628326590,43.38,ACCE0221316580T-628326590
318,ACCE0221521565HXL,6283261123,66.40,ACCE0221521565HXL-6283261123
321,ACCE0221521565HXL,62832627,66.40,ACCE0221521565HXL-62832627
382,ACCE0221522560V,628326869,63.21,ACCE0221522560V-628326869
1170,ACCE0322028550WXL,628326590,116.80,ACCE0322028550WXL-628326590
...,...,...,...,...
2198350,YOKO2052027565E,628326570,383.08,YOKO2052027565E-628326570
2198444,YOKO20520331250E,628326570,385.24,YOKO20520331250E-628326570
2198495,YOKO20520351250E,628326590,387.10,YOKO20520351250E-628326590
2201518,YOKO3961627570H,6283261134,209.45,YOKO3961627570H-6283261134


In [101]:
dfCurrentTestsToUpdate = dfCurrentTests[dfCurrentTests["SKU-Node"].isin(dfNLCPriceUpdatesFinal["SKU-Node"])].copy()
dfCurrentTestsToUpdate["Final target"].value_counts()

Final target
Added                       32180
Normal strategy             15166
No sales test                6808
Updated                      4980
Decreased - margin > 20%     4218
Updates test                 2693
Less sales                    142
Less sales back               121
Name: count, dtype: int64

In [102]:
# dfCurrentTestsToUpdate["Brand code"] = dfCurrentTestsToUpdate["SKU-Node"].str[:4]
# dfCurrentTestsToUpdate[dfCurrentTestsToUpdate["Final target"] == "Margin test"]#["Brand code"].value_counts()

## High price updates

In [103]:
dfCurrDSVNLCCheck = dfOutput[["Product Code", "Identifier","Purchase Price+FET","Node level cost - 20% margin","Node type", "Min units",
                              "current_nlc_margin","current_nlc_price","Final target","Target for node level cost? - 20% margin","Sub-group"]].copy()
dfCurrDSVNLCCPriceUpdatesDecr= dfCurrDSVNLCCheck[dfCurrDSVNLCCheck["current_nlc_margin"]> max_margin_update_prices].copy()
dfCurrDSVNLCCPriceUpdatesDecr["Category inventory"] = ""

dfCurrDSVNLCCPriceUpdatesDecr["SKU-Node"] = dfCurrDSVNLCCPriceUpdatesDecr["Product Code"] + "-" + dfCurrDSVNLCCPriceUpdatesDecr["Identifier"].astype(str)


dfCurrDSVNLCCPriceUpdatesDecr =dfCurrDSVNLCCPriceUpdatesDecr[~dfCurrDSVNLCCPriceUpdatesDecr["Final target"].isin(dont_update_final_targets)].copy()

if test:
    dfCurrDSVNLCCPriceUpdatesDecr = dfCurrDSVNLCCPriceUpdatesDecr[dfCurrDSVNLCCPriceUpdatesDecr["SKU-Node"].isin(dfTest["SKU-Node"]) == False].copy()

# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[dfCurrDSVNLCCPriceUpdates["SKU-Node"].isin(dfTrackerLessSalesChng["SKU-Node"]) == False].copy()
# dfCurrDSVNLCCPriceUpdates = dfCurrDSVNLCCPriceUpdates[~dfCurrDSVNLCCPriceUpdates["Product Code"].str.contains("MICH|BFGO")].copy()

dfCurrDSVNLCCPriceUpdatesDecr["Final delta vs current %"] = round((dfCurrDSVNLCCPriceUpdatesDecr["Node level cost - 20% margin"] - dfCurrDSVNLCCPriceUpdatesDecr["current_nlc_price"])/dfCurrDSVNLCCPriceUpdatesDecr["current_nlc_price"],4)

dfCurrDSVNLCCPriceUpdatesDecr["Final price change category final"] = np.where(dfCurrDSVNLCCPriceUpdatesDecr["Final delta vs current %"] < 0, "Decrease",
                                                                np.where(dfCurrDSVNLCCPriceUpdatesDecr["Final delta vs current %"] > 0, "Increase", "No change"))
dfCurrDSVNLCCPriceUpdatesDecr

,Product Code,Identifier,Purchase Price+FET,Node level cost - 20% margin,Node type,Min units,current_nlc_margin,current_nlc_price,Final target,Target for node level cost? - 20% margin,Sub-group,Category inventory,SKU-Node,Final delta vs current %,Final price change category final
45,ACCE0221416570H,6283261117,36.96,46.20,HUB,8,0.2150,47.08,Normal strategy,Yes,11%,,ACCE0221416570H-6283261117,-0.0187,Decrease
46,ACCE0221416570H,628326864,36.96,46.20,HUB,8,0.2150,47.08,Normal strategy,Yes,11%,,ACCE0221416570H-628326864,-0.0187,Decrease
1050,ACCE0322026540YXL,6283261117,85.41,106.76,HUB,8,0.2235,110.00,Normal strategy,Yes,11%,,ACCE0322026540YXL-6283261117,-0.0295,Decrease
1056,ACCE0322026540YXL,628326864,85.41,106.76,HUB,8,0.2235,110.00,Normal strategy,Yes,11%,,ACCE0322026540YXL-628326864,-0.0295,Decrease
1113,ACCE0322027555VXL,6283261117,85.41,106.76,HUB,8,0.2156,108.88,Updates test,Yes,No change,,ACCE0322027555VXL-6283261117,-0.0195,Decrease
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2203886,YOKO3962028555E,628326916,274.58,343.22,HUB,4,0.2768,379.67,Added,Yes,6%,,YOKO3962028555E-628326916,-0.0960,Decrease
2204026,YOKO3962029555E,628326916,288.40,360.50,HUB,4,0.2159,367.82,Added,Yes,6%,,YOKO3962029555E-628326916,-0.0199,Decrease
2204081,YOKO3962029560E,628326916,314.43,393.04,HUB,4,0.2176,401.87,Added,Yes,NaN,,YOKO3962029560E-628326916,-0.0220,Decrease
2204167,YOKO39620331250E,628326460,322.60,403.25,HUB,4,0.2054,405.98,Added,Yes,6%,,YOKO39620331250E-628326460,-0.0067,Decrease


In [104]:
usecols = ["Product Code", "Identifier", "Node level cost - 20% margin", "Category inventory", "Node type", "Min units",
           "Final delta vs current %", "Final price change category final","Final target","SKU-Node","Sub-group"]
dfUpdateAboveMarginTracker = dfCurrDSVNLCCPriceUpdatesDecr[usecols].copy()

# where Final target = Wm margin split test, then leave it as is, otherwise put "Updated - above max margin" 
dfUpdateAboveMarginTracker["Final target"] = np.where(dfUpdateAboveMarginTracker["Final target"] == "Wm margin split test", "Wm margin split test", 
                                                      np.where(dfUpdateAboveMarginTracker["Final target"] == "Margin test","Margin test",
                                                               "Decreased - margin > 20%"))


dfUpdateAboveMarginTracker = dfUpdateAboveMarginTracker.rename(columns={
    "Final node level cost category": "Final price category",
    "Node level cost - 20% margin": "Final price"})

dfUpdateAboveMarginTracker["Final price category"] = "20%"

dfUpdateAboveMarginTracker["SKU-Node"] = dfUpdateAboveMarginTracker["Product Code"] + "-" + dfUpdateAboveMarginTracker["Identifier"].astype(str)
dfUpdateAboveMarginTracker["Min units"] = dfUpdateAboveMarginTracker["Min units"].astype(int)
dfUpdateAboveMarginTracker["Start date"] = date_str
dfUpdateAboveMarginTracker

,Product Code,Identifier,Final price,Category inventory,Node type,Min units,Final delta vs current %,Final price change category final,Final target,SKU-Node,Sub-group,Final price category,Start date
45,ACCE0221416570H,6283261117,46.20,,HUB,8,-0.0187,Decrease,Decreased - margin > 20%,ACCE0221416570H-6283261117,11%,20%,2026-04-02
46,ACCE0221416570H,628326864,46.20,,HUB,8,-0.0187,Decrease,Decreased - margin > 20%,ACCE0221416570H-628326864,11%,20%,2026-04-02
1050,ACCE0322026540YXL,6283261117,106.76,,HUB,8,-0.0295,Decrease,Decreased - margin > 20%,ACCE0322026540YXL-6283261117,11%,20%,2026-04-02
1056,ACCE0322026540YXL,628326864,106.76,,HUB,8,-0.0295,Decrease,Decreased - margin > 20%,ACCE0322026540YXL-628326864,11%,20%,2026-04-02
1113,ACCE0322027555VXL,6283261117,106.76,,HUB,8,-0.0195,Decrease,Decreased - margin > 20%,ACCE0322027555VXL-6283261117,No change,20%,2026-04-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2203886,YOKO3962028555E,628326916,343.22,,HUB,4,-0.0960,Decrease,Decreased - margin > 20%,YOKO3962028555E-628326916,6%,20%,2026-04-02
2204026,YOKO3962029555E,628326916,360.50,,HUB,4,-0.0199,Decrease,Decreased - margin > 20%,YOKO3962029555E-628326916,6%,20%,2026-04-02
2204081,YOKO3962029560E,628326916,393.04,,HUB,4,-0.0220,Decrease,Decreased - margin > 20%,YOKO3962029560E-628326916,NaN,20%,2026-04-02
2204167,YOKO39620331250E,628326460,403.25,,HUB,4,-0.0067,Decrease,Decreased - margin > 20%,YOKO39620331250E-628326460,6%,20%,2026-04-02


In [105]:
# dfUpdateAboveMarginTracker["Final target"].value_counts() also want the average of "Final delta vs current %" by "Final target"
summary = (
    dfUpdateAboveMarginTracker.groupby("Final target")
    .agg(
        count=("Final target", "count"),
        avg_final_delta_vs_current_pct=("Final delta vs current %", "mean")
    )
    .reset_index()
)
summary

,Final target,count,avg_final_delta_vs_current_pct
0,Decreased - margin > 20%,30407,-0.028132


In [106]:
dfNLCPriceDecreasesDSV = dfCurrDSVNLCCPriceUpdatesDecr[["Product Code", "Identifier", "Node level cost - 20% margin"]].rename(
                        columns={"Identifier": "Source","Node level cost - 20% margin":"Price","Product Code":"SKU"})

dfNLCPriceDecreasesDSV["SKU-Node"] = dfNLCPriceDecreasesDSV["SKU"] + "-" + dfNLCPriceDecreasesDSV["Source"]
dfNLCPriceDecreasesDSV

,SKU,Source,Price,SKU-Node
45,ACCE0221416570H,6283261117,46.20,ACCE0221416570H-6283261117
46,ACCE0221416570H,628326864,46.20,ACCE0221416570H-628326864
1050,ACCE0322026540YXL,6283261117,106.76,ACCE0322026540YXL-6283261117
1056,ACCE0322026540YXL,628326864,106.76,ACCE0322026540YXL-628326864
1113,ACCE0322027555VXL,6283261117,106.76,ACCE0322027555VXL-6283261117
...,...,...,...,...
2203886,YOKO3962028555E,628326916,343.22,YOKO3962028555E-628326916
2204026,YOKO3962029555E,628326916,360.50,YOKO3962029555E-628326916
2204081,YOKO3962029560E,628326916,393.04,YOKO3962029560E-628326916
2204167,YOKO39620331250E,628326460,403.25,YOKO39620331250E-628326460


In [107]:
check = dfNLCPriceDecreasesDSV.copy()
check["Brand code"] = check["SKU"].str[:4]
check["Brand code"].value_counts()

Brand code
KUMH    14696
IRON     5116
CARL     2530
HIRU     1086
YOKO      956
        ...  
ZETA        1
GLAD        1
GRND        1
ROVE        1
RBPO        1
Name: count, Length: 90, dtype: int64

In [108]:
dfCurrDSVNLCCPriceUpdatesDecr["Final target"].value_counts()

Final target
Decreased - margin > 20%    17951
Normal strategy              5134
Added                        3009
Updated                      2207
No sales test                1158
Updates test                  874
Less sales back                43
Less sales                     26
Name: count, dtype: int64

In [109]:
check = dfCurrDSVNLCCPriceUpdatesDecr[dfCurrDSVNLCCPriceUpdatesDecr["Final target"]== "Margin test"]
tests_check = dfCurrentTestsAll.copy()
tests_check["SKU-Node"] = tests_check["Product Code"] + "-" + tests_check["Identifier"].astype(str)
tests_check[tests_check["SKU-Node"].isin(check["SKU-Node"])]["Start date"].value_counts()

Series([], Name: count, dtype: int64)

## New sku-nodes

In [110]:
dfNewSKUNodes = dfOutput[dfOutput["New NLC"] == "Yes"][["Product Code", "Identifier", "Final node level cost","Final node level cost category","Node type", "Min units"]].copy()
dfNewSKUNodes["Final target"] = "Added"
dfNewSKUNodes["Final price change category final"] = dfNewSKUNodes["Final target"]
dfNewSKUNodes["SKU-Node"] = dfNewSKUNodes["Product Code"] + "-" + dfNewSKUNodes["Identifier"].astype(str)
#SKU-Node not in dfTest SKU NODE

if test:
    dfNewSKUNodes = dfNewSKUNodes[~dfNewSKUNodes["SKU-Node"].isin(dfTest["SKU-Node"])].copy()

dfNewSKUNodesFinal = dfNewSKUNodes.rename(columns={"Product Code":"SKU","Final node level cost": "Price", "Identifier":"Source"}).drop(columns=["Final node level cost category"])

dfNewSKUNodesFinal

,SKU,Source,Price,Node type,Min units,Final target,Final price change category final,SKU-Node
201,ACCE0221519560H,6283261113,40.91,HUB,8,Added,Added,ACCE0221519560H-6283261113
202,ACCE0221519560H,6283261122,41.80,HUB,8,Added,Added,ACCE0221519560H-6283261122
203,ACCE0221519560H,6283261134,48.16,HUB,8,Added,Added,ACCE0221519560H-6283261134
204,ACCE0221519560H,6283261137,48.42,HUB,8,Added,Added,ACCE0221519560H-6283261137
205,ACCE0221519560H,6283261147,48.16,HUB,8,Added,Added,ACCE0221519560H-6283261147
...,...,...,...,...,...,...,...,...
2204347,ZEET0251523575TXL,628326592,74.16,HUB,4,Added,Added,ZEET0251523575TXL-628326592
2204373,ZEET1291620565V,6283261551,58.43,HUB,4,Added,Added,ZEET1291620565V-6283261551
2204374,ZEET1291620565V,62832617,57.80,HUB,4,Added,Added,ZEET1291620565V-62832617
2204408,ZENN0041822545WXL,628326435,62.87,HUB,4,Added,Added,ZENN0041822545WXL-628326435


In [111]:
dfNewSKUNodesTracker = dfNewSKUNodes.rename(columns={
    "Final node level cost category": "Final price category",
    "Final node level cost": "Final price"})
dfNewSKUNodesTracker["Start date"] = date_str
# dfNewSKUNodesTracker["SKU-Node"] = dfNewSKUNodesTracker["Product Code"] + "-" + dfNewSKUNodesTracker["Identifier"].astype(str)

dfNewSKUNodesTracker

,Product Code,Identifier,Final price,Final price category,Node type,Min units,Final target,Final price change category final,SKU-Node,Start date
201,ACCE0221519560H,6283261113,40.91,11%,HUB,8,Added,Added,ACCE0221519560H-6283261113,2026-04-02
202,ACCE0221519560H,6283261122,41.80,11%,HUB,8,Added,Added,ACCE0221519560H-6283261122,2026-04-02
203,ACCE0221519560H,6283261134,48.16,11%,HUB,8,Added,Added,ACCE0221519560H-6283261134,2026-04-02
204,ACCE0221519560H,6283261137,48.42,11%,HUB,8,Added,Added,ACCE0221519560H-6283261137,2026-04-02
205,ACCE0221519560H,6283261147,48.16,11%,HUB,8,Added,Added,ACCE0221519560H-6283261147,2026-04-02
...,...,...,...,...,...,...,...,...,...,...
2204347,ZEET0251523575TXL,628326592,74.16,11%,HUB,4,Added,Added,ZEET0251523575TXL-628326592,2026-04-02
2204373,ZEET1291620565V,6283261551,58.43,11%,HUB,4,Added,Added,ZEET1291620565V-6283261551,2026-04-02
2204374,ZEET1291620565V,62832617,57.80,11%,HUB,4,Added,Added,ZEET1291620565V-62832617,2026-04-02
2204408,ZENN0041822545WXL,628326435,62.87,11%,HUB,4,Added,Added,ZENN0041822545WXL-628326435,2026-04-02


In [112]:
dfNewSKUNodes["Final node level cost category"].value_counts()

Final node level cost category
11%    58676
8%      3798
6%      2531
Name: count, dtype: int64

# Other - Price changes check (no)

In [106]:
dfOutput[dfOutput["sku_node_revenue"]>0]

,Product Code,Identifier,Warehouse Code,Available,Purchase Price+FET,date,Node type,offer_price,unit_cost,MAP,Shipping cost,Cost+Shipping,Is MAP now?,Node level cost - 6% margin,Walmart Margin - 6% margin,Node cost < MAP - min margin% - 6% margin,Node cost - MAP - 6% margin,Target for node level cost? - 6% margin,Node level cost - 8% margin,Walmart Margin - 8% margin,Node cost < MAP - min margin% - 8% margin,Node cost - MAP - 8% margin,Target for node level cost? - 8% margin,Node level cost - 11% margin,Walmart Margin - 11% margin,Node cost < MAP - min margin% - 11% margin,Node cost - MAP - 11% margin,Target for node level cost? - 11% margin,Node level cost - 12% margin,Walmart Margin - 12% margin,Node cost < MAP - min margin% - 12% margin,Node cost - MAP - 12% margin,Target for node level cost? - 12% margin,Node level cost - 13% margin,Walmart Margin - 13% margin,Node cost < MAP - min margin% - 13% margin,Node cost - MAP - 13% margin,Target for node level cost? - 13% margin,Node level cost - 14% margin,Walmart Margin - 14% margin,Node cost < MAP - min margin% - 14% margin,Node cost - MAP - 14% margin,Target for node level cost? - 14% margin,Node level cost - 15% margin,Walmart Margin - 15% margin,Node cost < MAP - min margin% - 15% margin,Node cost - MAP - 15% margin,Target for node level cost? - 15% margin,Node level cost - 20% margin,Walmart Margin - 20% margin,Node cost < MAP - min margin% - 20% margin,Node cost - MAP - 20% margin,Target for node level cost? - 20% margin,current_nlc_price,Final node level cost category,Final node level cost,Final walmart margin,Final price change %,Final price change category,SKU-Node,Brand code,New NLC,Min units,current_nlc_margin,current_nlc_margin category,Current walmart margin at National,Current walmart margin at NLC,Current walmart margin at NLC category,Walmart margin at new NLC category,Total margin,Wmt margin 50% group,Price margin split 50%,Wmt margin 60% group,Price margin split 60%,sku_node_revenue,sku_revenue,sku_sales_category,Final target,Start date,Sub-group
3,ACCE0221316580T,6283261134,5234,8,33.35,2026-03-31,HUB,45.930000,37.94,NaN,7.0,40.35,No,42.48,0.2275,No,NaN,Yes,43.25,0.2108,No,NaN,Yes,44.47,0.1842,No,NaN,Yes,44.90,0.1748,No,NaN,Yes,45.33,0.1655,No,NaN,Yes,45.78,0.1557,No,NaN,Yes,46.24,0.1457,No,NaN,Yes,48.69,0.0923,No,NaN,Yes,41.69,11%,44.47,0.1842,0.067,Increase,ACCE0221316580T-6283261134,ACCE,No,8,0.0321,0% to <6%,0.1740,0.0923,0% to 10%,15% to 20%,0.1244,0.0622,43.07,0.0498,43.64,177.88,309.02,95-100%,Shipping cost added,2026-03-07,20%
5,ACCE0221316580T,628326433,5578,55,33.61,2026-03-31,HUB,45.930000,37.94,NaN,0.0,33.61,No,35.76,0.2214,No,NaN,Yes,36.53,0.2047,No,NaN,Yes,37.76,0.1779,No,NaN,Yes,38.19,0.1685,No,NaN,Yes,38.63,0.1589,No,NaN,Yes,39.08,0.1491,No,NaN,Yes,39.54,0.1391,No,NaN,Yes,42.01,0.0853,No,NaN,Yes,37.76,11%,37.76,0.1779,0.000,No change,ACCE0221316580T-628326433,ACCE,No,8,0.1099,10.8% to <11.2%,0.1740,0.1779,15% to 20%,15% to 20%,0.2878,0.1439,39.32,0.1151,40.64,37.76,309.02,95-100%,Normal strategy,2026-01-05,11%
12,ACCE0221316580T,628326916,6012,16,41.13,2026-03-31,HUB,45.930000,37.94,NaN,0.0,41.13,No,43.76,0.0472,No,NaN,Yes,44.71,0.0266,No,NaN,Yes,46.21,-0.0061,No,NaN,No,46.74,-0.0176,No,NaN,No,47.28,-0.0294,No,NaN,No,47.83,-0.0414,No,NaN,No,48.39,-0.0536,No,NaN,No,51.41,-0.1193,No,NaN,No,46.69,8%,44.71,0.0266,-0.042,Decrease,ACCE0221316580T-628326916,ACCE,No,8,0.1191,11.2% to <13%,0.1740,-0.0165,-10% to 0%,0% to 10%,0.1026,0.0513,43.57,0.0410,44.05,93.38,309.02,95-100%,Normal strategy,2026-01-05,11%
14,ACCE0221317570H,6283261122,5297,13,28.86,2026-03-29,HUB,44.363333,32.09,NaN,0.0,28.86,No,30.70,0.3080,No,NaN,Yes,31.37,0.2929,No,NaN,Yes,32.43,0.2690,No,NaN,Yes,32.80,0.2607,No,NaN,Yes,33.17,0.2523,No,NaN,Yes,33.56,0.2435,No,NaN,Yes,33.95,0.2347,No,NaN,Yes,36.07,0.1869,No,NaN,Yes,32.43,11%,32.43,0.2690,0.000,No change,ACCE0221317570H-6283261122,ACCE,No,8,0.1101,10.8% to <11.2%,0.2767,0.2690,25% to 30%,25% to 30%,0.3791,0.1896,35.95,0.1516,36.07,6

In [104]:
dfOutputChange = dfOutput[(dfOutput["New NLC"] == "No")].copy()

dontChange = []  #["Margin test", "Less sales","Updates test","Less sales back"]

change = ["Less sales","Less sales back","Normal strategy", "Added","Updated","Decreased - margin > 20%"] #,"Margin test","Wm margin split test","Shipping cost added","DSVD test","Increase test"]

dfOutputChange = dfOutputChange[(dfOutputChange["Final target"].isin(dontChange) == False)
                                & (dfOutputChange["Final target"].isin(change) == True)
                               & (dfOutputChange["sku_node_revenue"]>0) ].copy()

#Group by Final price change category and count the number of SKU-Nodes in each category and avg Final price change %
#Also sum sku_node revenue and calculate % of total_revenue, total_revenue is already defined dont reset it pls


summary = (
    dfOutputChange.groupby("Final price change category")
    .agg(
        count=("Final price change category", "count"),
        avg_final_price_change_pct=("Final price change %", "mean"),
        total_revenue_change_category=("sku_node_revenue", "sum")
    )
    .reset_index()
)
summary["% of total revenue"] = summary["total_revenue_change_category"] / total_revenue

summary

,Final price change category,count,avg_final_price_change_pct,total_revenue_change_category,% of total revenue
0,Decrease,17078,-0.044094,12031624.54,0.131235
1,Increase,2543,0.013953,1576425.46,0.017195
2,No change,12328,0.000000,8621814.65,0.094042


In [105]:
total_revenue

91680232.21999997

In [ ]:
# wmMargin = 0.25

# dfHighWMMargin = dfOutput[(dfOutput["Current walmart margin at NLC"] > wmMargin) &  (~dfOutput["Final target"].isin(dontChange))].copy()
# display(dfHighWMMargin["current_nlc_margin category"].value_counts())
# dfHighWMMargin["Current walmart margin at NLC category"].value_counts()

current_nlc_margin category
10.8% to <11.2%    145512
8% to <10.8%        81060
13% to <15%         57415
11.2% to <13%       48722
>=17%               30696
15% to <17%         25730
6% to <8%           19799
0% to <6%            3301
<0%                   535
Name: count, dtype: int64

Current walmart margin at NLC category
25% to 30%    257715
>=30%         155055
<-10%              0
-10% to 0%         0
0% to 10%          0
10% to 15%         0
15% to 20%         0
20% to 25%         0
Name: count, dtype: int64

In [ ]:
# dfOutputChangeWithSales = dfOutputChange[(dfOutputChange["Product Code"].isin(dfSalesLastXDaysUniqueSKUs["Product Code"])) &
#                           (~dfOutputChange["Final target"].isin(dontChange))].copy()

# # dfOutputChangeWithSales = dfOutputChangeWithSales.merge(dfSalesSKUNodeFeb, how="left", on="SKU-Node")

# display(dfOutputChangeWithSales["NLC Price change category"].value_counts())

NLC Price change category
Decrease     147732
No change    128183
Increase      84181
Name: count, dtype: int64

In [ ]:
# dfOutputChangeWithSalesChange = dfOutputChangeWithSales[(dfOutputChangeWithSales["NLC Price change category"] != "No change") &
#                                                         (dfOutputChangeWithSales["NLC Price change %"].abs() > 0.01)&
#                                                         (dfOutputChangeWithSales["Start date"].isin(["2026-01-05","2026-01-20"]))].copy()
# display(dfOutputChangeWithSalesChange["NLC Price change category"].value_counts())
# display(dfOutputChangeWithSalesChange["Final target"].value_counts())
# display(dfOutputChangeWithSalesChange["Start date"].value_counts())

# dfCheck = dfOutputChangeWithSalesChange.groupby(["Final target", "Start date"]).agg({"Product Code": "count"}).reset_index()
# dfCheck = dfCheck.rename(columns={"Product Code": "count"})
# dfCheck

NLC Price change category
Decrease    35203
Increase    18846
Name: count, dtype: int64

Final target
Normal strategy    43935
Added               5161
No sales test       4404
Updated              549
Name: count, dtype: int64

Start date
2026-01-05    48339
2026-01-20     5710
Name: count, dtype: int64

,Final target,Start date,count
0,Added,2026-01-20,5161
1,No sales test,2026-01-05,4404
2,Normal strategy,2026-01-05,43935
3,Updated,2026-01-20,549


## Test update prices

In [ ]:
# dfTestSKUs = dfOutputChangeWithSalesChange.groupby("Product Code").agg({"total_inv_amount": "sum"}).reset_index()
# dfTestSKUs = split_skus_into_groups(dfTestSKUs,2,"Product Code","total_inv_amount")
# total_inv_test = dfTestSKUs["total_inv_amount"].sum()
# dfTestSKUs = dfTestSKUs[["Product Code", "group"]].copy()
# dfTestSKUs

In [ ]:
# total_inv_test/totalFebInvAmt

In [ ]:
# dfTestUpdate = dfOutputChangeWithSalesChange.merge(dfTestSKUs, how="left", on="Product Code")
# dfTestUpdate

In [ ]:
# dfTest = dfTestUpdate.copy()
# dfTest = dfTest[(dfTest["group"].notna()) & (dfTest["current_nlc_price"].notna())].copy()

# print("Total rows in test:", len(dfTest))

# dfTest["Group"] = np.where(dfTest["group"] == 1, "Update", np.where(dfTest["group"] == 2, "No change","Other"))

# dfTest["Test price"]= np.where(dfTest["group"] == 1, dfTest["Final node level cost"],
#                                       np.where(dfTest["group"] == 2, dfTest["current_nlc_price"], np.nan))

# dfTest["Delta test to current NLC %"] = round((dfTest["Test price"] - dfTest["current_nlc_price"])/dfTest["current_nlc_price"],4)
# dfTest["Delta test to current NLC category"] = np.where(dfTest["Delta test to current NLC %"] < 0, "Decrease",
#                                                                 np.where(dfTest["Delta test to current NLC %"] > 0, "Increase", "No change"))

# relevantCols = ["Product Code", "Identifier","SKU-Node", "Group", "Test price", "Delta test to current NLC %", "Delta test to current NLC category",
#                     "Node type", "Min units","current_nlc_margin","Final node level cost category"]



# dfTest = dfTest[relevantCols].copy().rename(columns = {"current_nlc_margin": "NLC margin before update"})

# # Now display # of increases, decreases, no change by each Group + average of "Delta test to current NLC %" by group
# summary = (
#     dfTest
#         .groupby("Group")
#         .agg(
#             count=("Delta test to current NLC category", "count"),
#             increases=("Delta test to current NLC category", lambda x: (x == "Increase").sum()),
#             decreases=("Delta test to current NLC category", lambda x: (x == "Decrease").sum()),
#             no_change=("Delta test to current NLC category", lambda x: (x == "No change").sum()),
#             avg_delta_test_to_current_nlc_pct=("Delta test to current NLC %", "mean"),
#         )
#         .reset_index()
# )
# summary

In [ ]:
# dfTestDSV= dfTest[["Product Code", "Identifier","Test price","SKU-Node"]].rename(columns = {"Product Code": "SKU", "Identifier": "Source", "Test price": "Price"})
# dfTestDSV

In [ ]:
# usecols = ["Product Code", "Identifier","Group", "Test price", "Delta test to current NLC category", "Delta test to current NLC %", 
#            "Node type", "Min units","NLC margin before update","Final node level cost category"]
# dfTestTracker = dfTest[usecols].copy()

# dfTestTracker["Final target"] = "Updates test"

# dfTestTracker = dfTestTracker.rename(columns={
#     "Group": "Sub-group",
#     "Final node level cost category":"Final price category",
#     "Delta test to current NLC category": "Final price change category final",
#     "Delta test to current NLC %": "Final delta vs current %",
#     "Test price": "Final price"})
# dfTestTracker["SKU-Node"] = dfTestTracker["Product Code"] + "-" + dfTestTracker["Identifier"].astype(str)
# dfTestTracker["Min units"] = dfTestTracker["Min units"].astype(int)
# dfTestTracker["Start date"] = date_str
# dfTestTracker

In [ ]:
# dfOutputChangeIncrease = dfOutputChange[dfOutputChange["NLC Price change category"] == "Decrease"].copy()
# display(dfOutputChangeIncrease["current_nlc_margin category"].value_counts())

In [ ]:
# len(dfOutputChange) + len(dfNLCPriceUpdatesFinal) + len(dfNewSKUNodesFinal) == len(dfOutput)

# Create new DSV

## Check original dsv + dsv starting point

In [113]:
dfCurrDSVOriginal

,sku,walmart_dsv_price,source
0,IRON0081417565H,40.11,628326503
1,IRON0061417565H,40.11,628326106
2,IRON0061417565H,40.11,6283261141
3,APLS0031518555V,40.11,628326916
4,IRON0061417565H,40.11,6283261227
...,...,...,...
3296321,MATR0011520575C2,53.99,NaN
3296322,MATR0011522575D2,69.39,NaN
3296323,MATR0011623585E2,80.09,NaN
3296324,RADA0101824545WXL,79.43,NaN


In [129]:
dfCurrDSVStart = dfCurrDSVOriginal.rename(columns={"sku": "SKU", "walmart_dsv_price":"Price", "source":"Source"})#.drop(columns=["date"])
dfCurrDSVStart["SKU-Node"] = dfCurrDSVStart["SKU"] + "-" + dfCurrDSVStart["Source"]
dfCurrDSVStart

,SKU,Price,Source,SKU-Node
0,IRON0081417565H,40.11,628326503,IRON0081417565H-628326503
1,IRON0061417565H,40.11,628326106,IRON0061417565H-628326106
2,IRON0061417565H,40.11,6283261141,IRON0061417565H-6283261141
3,APLS0031518555V,40.11,628326916,APLS0031518555V-628326916
4,IRON0061417565H,40.11,6283261227,IRON0061417565H-6283261227
...,...,...,...,...
3296321,MATR0011520575C2,53.99,NaN,NaN
3296322,MATR0011522575D2,69.39,NaN,NaN
3296323,MATR0011623585E2,80.09,NaN,NaN
3296324,RADA0101824545WXL,79.43,NaN,NaN


## Update national prices (no)

In [115]:
dfCurrDSVStartNational = dfCurrDSVStart[dfCurrDSVStart["Source"].isna()].copy()
dfCurrDSVStartNational

,SKU,Price,Source,SKU-Node
3234125,KELL0011418570T,69.57,NaN,NaN
3234126,WDTO0314410B,4.14,NaN,NaN
3234127,WDTO073411400B,4.81,NaN,NaN
3234128,WDTO0734410B,4.84,NaN,NaN
3234129,OTRO022411400B,11.28,NaN,NaN
...,...,...,...,...
3296321,MATR0011520575C2,53.99,NaN,NaN
3296322,MATR0011522575D2,69.39,NaN,NaN
3296323,MATR0011623585E2,80.09,NaN,NaN
3296324,RADA0101824545WXL,79.43,NaN,NaN


In [116]:
pathNewPrices = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-03\Walmart B2B Item Report 3.25.26.xlsx"
# sheetNewPrices = "DSV prices no increase"
# sheetNewPrices = "Final prices for the DSV"
sheetNewPrices = "National prices"
dfNewPrices = pd.read_excel(pathNewPrices, sheet_name=sheetNewPrices, dtype={"SKU": str}, skiprows=2)
dfNewPrices = dfNewPrices.rename(columns={"Min of Final price": "New Price", "Product Code": "SKU"})
dfNewPrices

,SKU,New Price
0,-,45.00
1,000060094858,180.30
2,00850004743751,42.00
3,00850004743768,45.00
4,00850004743775,45.00
...,...,...
64189,ZETA0411825555VXL,79.57
64190,ZETA0411826560V,92.61
64191,ZETA0411826565H,96.27
64192,ZETA0411827565H,101.21


In [117]:
dfNewNational = dfCurrDSVStartNational.merge(dfNewPrices, how="left", on="SKU")
dfNewNational["Final DSV price"] = np.where(dfNewNational["New Price"].isna(), dfNewNational["Price"], dfNewNational["New Price"])
dfNewNational["Price change %"] = round((dfNewNational["Final DSV price"] - dfNewNational["Price"]) / dfNewNational["Price"],4)
dfNewNational["Price change category"] = np.where(dfNewNational["Price change %"]>0, "Increase", np.where(dfNewNational["Price change %"]<0, "Decrease", "No change"))
display(dfNewNational["Price change category"].value_counts())
dfNewNationalFinal = dfNewNational[["SKU", "Final DSV price"]].rename(columns={"Final DSV price": "Price"})
dfNewNationalFinal

Price change category
No change    62201
Name: count, dtype: int64

,SKU,Price
0,KELL0011418570T,69.57
1,WDTO0314410B,4.14
2,WDTO073411400B,4.81
3,WDTO0734410B,4.84
4,OTRO022411400B,11.28
...,...,...
62196,MATR0011520575C2,53.99
62197,MATR0011522575D2,69.39
62198,MATR0011623585E2,80.09
62199,RADA0101824545WXL,79.43


In [40]:
dfCurrDSVNLCForDSV = dfCurrDSVNLC.rename(columns={"Product Code": "SKU", "current_nlc_price": "Price","Identifier": "Source"})
dfCurrDSVNLCForDSV

,SKU,Price,Source
0,IRON0081417565H,40.11,628326503
1,IRON0061417565H,40.11,628326106
2,IRON0061417565H,40.11,6283261141
3,APLS0031518555V,40.11,628326916
4,IRON0061417565H,40.11,6283261227
...,...,...,...
3296321,YOKO3661827565H,208.84,628326687
3296322,YOKO39615311050C,188.82,628326687
3296323,YOKO3961624575T,164.34,628326687
3296324,YOKO3961629575E,243.97,6283261143


In [41]:
dfDSVNewNational = pd.concat([dfCurrDSVNLCForDSV, dfNewNationalFinal], ignore_index=True)
dfDSVNewNational["Minimum margin"] = "4%"
dfDSVNewNational = dfDSVNewNational[["SKU", "Price", "Minimum margin", "Source"]]
dfDSVNewNational

,SKU,Price,Minimum margin,Source
0,IRON0081417565H,40.11,4%,628326503
1,IRON0061417565H,40.11,4%,628326106
2,IRON0061417565H,40.11,4%,6283261141
3,APLS0031518555V,40.11,4%,628326916
4,IRON0061417565H,40.11,4%,6283261227
...,...,...,...,...
3296321,MATR0011520575C2,53.99,4%,NaN
3296322,MATR0011522575D2,69.39,4%,NaN
3296323,MATR0011623585E2,80.09,4%,NaN
3296324,RADA0101824545WXL,79.43,4%,NaN


In [42]:
dfDSVNewNational[dfDSVNewNational["SKU"].isin(dfRollbacks["Product Code"])]["Source"].unique()

array([nan], dtype=object)

In [ ]:
# outputPath = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-01\National pricing submission"
# file_name = f"Walmart B2B NLC Price Updates no Price increases.csv"
# full_path = f"{outputPath}\\{file_name}"
# dfDSVNewNational.to_csv(full_path, index=False)

In [43]:
dfCurrDSVStart = dfDSVNewNational.copy()
dfCurrDSVStart["SKU-Node"] = dfCurrDSVStart["SKU"] + "-" + dfCurrDSVStart["Source"]
dfCurrDSVStart

,SKU,Price,Minimum margin,Source,SKU-Node
0,IRON0081417565H,40.11,4%,628326503,IRON0081417565H-628326503
1,IRON0061417565H,40.11,4%,628326106,IRON0061417565H-628326106
2,IRON0061417565H,40.11,4%,6283261141,IRON0061417565H-6283261141
3,APLS0031518555V,40.11,4%,628326916,APLS0031518555V-628326916
4,IRON0061417565H,40.11,4%,6283261227,IRON0061417565H-6283261227
...,...,...,...,...,...
3296321,MATR0011520575C2,53.99,4%,NaN,NaN
3296322,MATR0011522575D2,69.39,4%,NaN,NaN
3296323,MATR0011623585E2,80.09,4%,NaN,NaN
3296324,RADA0101824545WXL,79.43,4%,NaN,NaN


## RBs updates (no)

In [57]:
dfCurrDSVStartNLC = dfCurrDSVStart[dfCurrDSVStart["Source"].notna()].copy()
dfCurrDSVStartNational = dfCurrDSVStart[dfCurrDSVStart["Source"].isna()].copy()

dfCurrDSVStartNLCNoRbs = dfCurrDSVStartNLC[~dfCurrDSVStartNLC["SKU"].isin(dfRollbacks["Product Code"])].copy()
print("Removed", len(dfCurrDSVStartNLC) - len(dfCurrDSVStartNLCNoRbs), "SKUs with rollbacks from the NLC DSV start list")

dfCurrDSVStart = pd.concat([dfCurrDSVStartNLCNoRbs, dfCurrDSVStartNational], ignore_index=True)
dfCurrDSVStart

Removed 65560 SKUs with rollbacks from the NLC DSV start list


,SKU,Price,Source,SKU-Node
0,IRON0061417565H,40.11,62832635,IRON0061417565H-62832635
1,IRON0081417565H,40.11,628326503,IRON0081417565H-628326503
2,IRON0061417565H,40.11,628326106,IRON0061417565H-628326106
3,IRON0061417565H,40.11,6283261141,IRON0061417565H-6283261141
4,APLS0031518555V,40.11,628326916,APLS0031518555V-628326916
...,...,...,...,...
3222579,MATR0011520575C2,57.17,NaN,NaN
3222580,MATR0011522575D2,69.39,NaN,NaN
3222581,MATR0011623585E2,94.62,NaN,NaN
3222582,RADA0101824545WXL,79.43,NaN,NaN


In [58]:
dfCurrNationalRB = dfCurrDSVStartNational.merge(dfRollbacksSKUPrice, how="left", on="SKU")
dfCurrNationalRB["Delta price"] = round((dfCurrNationalRB["RB price"] - dfCurrNationalRB["Price"])/dfCurrNationalRB["Price"],4)
dfCurrNationalRB["Delta price category"] = np.where(dfCurrNationalRB["Delta price"] < 0, "Decrease",
                                                     np.where(dfCurrNationalRB["Delta price"] > 0, "Increase", "No change"))
display(dfCurrNationalRB[dfCurrNationalRB["Delta price category"] != "No change"])
display(dfCurrNationalRB["Delta price category"].value_counts())

dfCurrNationalRB["Price"] = np.where(dfCurrNationalRB["RB price"].isna(), dfCurrNationalRB["Price"], dfCurrNationalRB["RB price"])
dfCurrNationalRB = dfCurrNationalRB[["SKU", "Price"]].copy()

,SKU,Price,Source,SKU-Node,RB price,Delta price,Delta price category
77,ACCE0221519560H,40.33,NaN,NaN,38.27,-0.0511,Decrease
90,ARMS0041519555V,40.47,NaN,NaN,37.36,-0.0768,Decrease
109,PRIX0011518555V,40.64,NaN,NaN,39.95,-0.0170,Decrease
148,LEXA0431418560H,40.91,NaN,NaN,35.10,-0.1420,Decrease
167,BLHK0131518560HXL,41.08,NaN,NaN,40.92,-0.0039,Decrease
...,...,...,...,...,...,...,...
60188,SAIL3072228545HXL,163.02,NaN,NaN,152.50,-0.0645,Decrease
60222,TOYO38617371250E,428.24,NaN,NaN,384.67,-0.1017,Decrease
60225,TOYO38618371350E,463.86,NaN,NaN,408.22,-0.1199,Decrease
60226,TOYO38620351150E,436.15,NaN,NaN,395.43,-0.0934,Decrease


Delta price category
No change    61404
Decrease       783
Increase        14
Name: count, dtype: int64

In [59]:
dfCurrNationalRB

,SKU,Price
0,KELL0011418570T,69.57
1,WDTO0314410B,4.14
2,WDTO073411400B,4.81
3,WDTO0734410B,4.84
4,OTRO022411400B,12.85
...,...,...
62196,MATR0011520575C2,57.17
62197,MATR0011522575D2,69.39
62198,MATR0011623585E2,94.62
62199,RADA0101824545WXL,79.43


In [ ]:
# dfCurrDSVStart[dfCurrDSVStart["SKU"].isin(dfRollbacks["Product Code"])]["Source"].unique()
# dfCurrDSVStartNoRbs = dfCurrDSVStart[~dfCurrDSVStart["SKU"].isin(dfRollbacks["Product Code"])].copy()

# dfCurrDSVStartRbsOk = pd.concat([dfCurrDSVStartNoRbs, dfRollbacksSKUPrice], ignore_index=True)
# dfCurrDSVStartRbsOk

In [60]:
dfRollbacksSKUPriceCheck = dfCurrNationalRB.merge(dfCurrDSVStartNational, how="left", on="SKU")
dfRollbacksSKUPriceCheck["Delta price"] = round(dfRollbacksSKUPriceCheck["Price_x"] - dfRollbacksSKUPriceCheck["Price_y"],2)
dfRollbacksSKUPriceCheck["Delta Price category"] = dfRollbacksSKUPriceCheck["Delta price category"] = np.where(dfRollbacksSKUPriceCheck["Delta price"] < 0, "Decrease",
                                                     np.where(dfRollbacksSKUPriceCheck["Delta price"] > 0, "Increase", "No change"))
dfRollbacksSKUPriceCheck["Delta Price category"].value_counts()

Delta Price category
No change    61404
Decrease       783
Increase        14
Name: count, dtype: int64

In [61]:
dfCurrDSVStart = pd.concat([dfCurrDSVStartNLCNoRbs, dfCurrNationalRB.rename(columns={"Price": "Price", "SKU": "SKU", "Source": "Source"})], ignore_index=True)
dfCurrDSVStart

,SKU,Price,Source,SKU-Node
0,IRON0061417565H,40.11,62832635,IRON0061417565H-62832635
1,IRON0081417565H,40.11,628326503,IRON0081417565H-628326503
2,IRON0061417565H,40.11,628326106,IRON0061417565H-628326106
3,IRON0061417565H,40.11,6283261141,IRON0061417565H-6283261141
4,APLS0031518555V,40.11,628326916,APLS0031518555V-628326916
...,...,...,...,...
3222579,MATR0011520575C2,57.17,NaN,NaN
3222580,MATR0011522575D2,69.39,NaN,NaN
3222581,MATR0011623585E2,94.62,NaN,NaN
3222582,RADA0101824545WXL,79.43,NaN,NaN


## Normal update

In [150]:
original_rows = len(dfCurrDSVStart)

list_dfs_updates = [dfPriceIncrDSV, dfDSVDDSV, dfWmMarginSplitDSV, 
                    dfMarginTestDSV, dfNLCPriceUpdatesFinal, 
                    dfNLCPriceDecreasesDSV]

if test:
    list_dfs_updates = list_dfs_updates + [dfTestDSV]


dfUpdatesDSV = pd.concat(list_dfs_updates, ignore_index=True)

#dfUpdatesDSV price round to 2 decimals
dfUpdatesDSV["Price"] = dfUpdatesDSV["Price"].round(2)

rows_to_update = len(dfUpdatesDSV)  

rows_to_add = len(dfNewSKUNodesFinal)

# if test:
#       rows_test = len(dfTest)
#       rows_added_by_test = len(dfTestDSV[~dfTestDSV["SKU-Node"].isin(dfCurrDSVStart["SKU-Node"])])
#       rows_update_test = rows_test - rows_added_by_test
#       rows_to_add+= rows_added_by_test
#       rows_to_update+= rows_update_test

final_rows = original_rows + rows_to_add

print("Original rows in Walmart DSV:", original_rows
      ,"\nRows to update:", rows_to_update
      # ,"\nRows to update (test):", rows_to_update_test
      ,"\nRows to add:", rows_to_add
      ,"\nFinal rows after update and add:", final_rows
      ,"\nTotal rows to update and add:", rows_to_add +rows_to_update) # + rows_to_update_test
final_rows

Original rows in Walmart DSV: 3296326 
Rows to update: 310861 
Rows to add: 65005 
Final rows after update and add: 3361331 
Total rows to update and add: 375866


3361331

In [151]:
dfCurrDSVRemovedUpdates = dfCurrDSVStart[~dfCurrDSVStart["SKU-Node"].isin(dfUpdatesDSV["SKU-Node"])].copy()

dfNewDSV = pd.concat([dfCurrDSVRemovedUpdates, dfUpdatesDSV,dfNewSKUNodesFinal], ignore_index=True).drop(columns=["SKU-Node"])
dfNewDSV["Minimum margin"] = "4%"
dfNewDSV = dfNewDSV[["SKU", "Price", "Minimum margin", "Source"]]
print("Len before dropping duplicates:", len(dfNewDSV))
dfNewDSV = dfNewDSV.drop_duplicates(subset=["SKU", "Source"], keep="last").reset_index(drop=True)
print("Len after dropping duplicates:", len(dfNewDSV))
dfNewDSV

Len before dropping duplicates: 3361331
Len after dropping duplicates: 3361331


,SKU,Price,Minimum margin,Source
0,IRON0081417565H,40.11,4%,628326503
1,APLS0031518555V,40.11,4%,628326916
2,IRON0061417565H,40.11,4%,6283261227
3,ACCE0221417570T,40.11,4%,62832652
4,IRON0061417565H,40.11,4%,6283261400
...,...,...,...,...
3361326,ZEET0251523575TXL,74.16,4%,628326592
3361327,ZEET1291620565V,58.43,4%,6283261551
3361328,ZEET1291620565V,57.80,4%,62832617
3361329,ZENN0041822545WXL,62.87,4%,628326435


In [ ]:
# dfNewDSV = dfCurrDSVStart.copy()
# dfNewDSV["Minimum margin"] = "4%"
# dfNewDSV = dfNewDSV[["SKU", "Price", "Minimum margin", "Source"]]
# dfNewDSV = dfNewDSV.drop_duplicates()
# dfNewDSV

,SKU,Price,Minimum margin,Source
0,IRON0081417565H,40.11,4%,628326503
1,IRON0061417565H,40.11,4%,628326106
2,IRON0061417565H,40.11,4%,6283261141
3,APLS0031518555V,40.11,4%,628326916
4,IRON0061417565H,40.11,4%,6283261227
...,...,...,...,...
3296321,MATR0011520575C2,53.99,4%,NaN
3296322,MATR0011522575D2,69.39,4%,NaN
3296323,MATR0011623585E2,80.09,4%,NaN
3296324,RADA0101824545WXL,79.43,4%,NaN


In [ ]:
# pathAddSKUsList = [r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Walmart B2B_New SKUs from March 2026.xlsx",
#                    r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Order 66_Priority 1 Phase 2A & 2B 030926.xlsx"]

# dfAddSKUsList = []
# for path in pathAddSKUsList:
#     df = pd.read_excel(path, usecols=["SKU", "Unit Cost","Order 66"])
#     #keep only when Order 66 = "No"
#     df = df[df["Order 66"] == "No"].copy()
#     #Drop col "Order 66"
#     df = df.drop(columns=["Order 66"])
#     df.rename(columns={ "Unit Cost": "Price"}, inplace=True)
#     df["Minimum margin"] = "4%"
#     df["Source"] = np.nan
#     print(len(df), "rows read from file:", path)

#     dfAddSKUsList.append(df)
    
# dfAddSKUs = pd.concat(dfAddSKUsList, ignore_index=True).drop_duplicates()
# dfAddSKUs


1906 rows read from file: H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Walmart B2B_New SKUs from March 2026.xlsx
18 rows read from file: H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_02 Pricing\2026-02\New SKUs\Item Report_Order 66_Priority 1 Phase 2A & 2B 030926.xlsx


,Price,SKU,Minimum margin,Source
0,144.41,ACCE0322123550V,4%,NaN
1,50.38,ACCE0531518555HXL,4%,NaN
2,119.12,ACCE0532228535WXL,4%,NaN
3,74.44,AIRL0038201000B,4%,NaN
4,80.10,APLS0151621570H,4%,NaN
...,...,...,...,...
1919,57.17,MATR0011520575C2,4%,NaN
1920,69.39,MATR0011522575D2,4%,NaN
1921,94.62,MATR0011623585E2,4%,NaN
1922,79.43,RADA0101824545WXL,4%,NaN


In [ ]:
# dfNewDSV = dfCurrDSVStart.copy()
# dfAddSKUsAlreadyInDSV = dfAddSKUs[dfAddSKUs["SKU"].isin(dfNewDSV["SKU"])].copy()
# dfAddNewSKUs = dfAddSKUs[~dfAddSKUs["SKU"].isin(dfNewDSV["SKU"])].copy()

# # dfNewDSVRemove = dfNewDSV[dfNewDSV["SKU"].isin(dfAddSKUs["SKU"])].copy()
# # dfNewDSVNoOverlap = dfNewDSV[~dfNewDSV["SKU"].isin(dfAddSKUs["SKU"])].copy()
# print("Number of overlapping SKUs between new DSV and add SKUs:", len(dfAddSKUsAlreadyInDSV))

# print("# SKUs to add:", len(dfAddNewSKUs))

# total_skus = len(dfNewDSV) + len(dfAddNewSKUs)
# print("Total SKUs in new DSV after adding new SKUs:", total_skus)

# dfNewDSV = pd.concat([dfNewDSV, dfAddNewSKUs], ignore_index=True)

# #Drop SKU-Node
# dfNewDSV = dfNewDSV.drop(columns=["SKU-Node"])

# dfNewDSV["Minimum margin"] = "4%"

# dfNewDSV

Number of overlapping SKUs between new DSV and add SKUs: 1
# SKUs to add: 1915
Total SKUs in new DSV after adding new SKUs: 3234726


,SKU,Price,Source,Minimum margin
0,IRON0061417565H,40.11,62832635,4%
1,IRON0081417565H,40.11,628326503,4%
2,IRON0061417565H,40.11,628326106,4%
3,IRON0061417565H,40.11,6283261141,4%
4,APLS0031518555V,40.11,628326916,4%
...,...,...,...,...
3234721,MATR0011520575C2,57.17,NaN,4%
3234722,MATR0011522575D2,69.39,NaN,4%
3234723,MATR0011623585E2,94.62,NaN,4%
3234724,RADA0101824545WXL,79.43,NaN,4%


In [ ]:
current_year_month = datetime.now().strftime("%Y-%m")
outputPath = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\DSV Files"
outputPathMonth = os.path.join(outputPath, current_year_month)
if not os.path.exists(outputPathMonth):
    os.makedirs(outputPathMonth)
outputFileName = f"New WalmartB2B DSV to upload {date_str}.csv"
dfNewDSV.to_csv(os.path.join(outputPathMonth, outputFileName), index=False)

In [152]:
#Check we need to do every time:

dfNewDSVCheck = dfNewDSV.copy()
dfNewDSVCheck["Source"] = dfNewDSVCheck["Source"].fillna("National")
dfNewDSVCheck["SKU-Node"] = dfNewDSVCheck["SKU"] + "-" + dfNewDSVCheck["Source"]

dfCurrDSVOriginalToCheck = dfCurrDSVOriginal.copy()
dfCurrDSVOriginalToCheck = dfCurrDSVOriginalToCheck.copy()#.drop(columns=["date"])
dfCurrDSVOriginalToCheck["source"] = dfCurrDSVOriginalToCheck["source"].fillna("National")
dfCurrDSVOriginalToCheck["SKU-Node"] = dfCurrDSVOriginalToCheck["sku"] + "-" + dfCurrDSVOriginalToCheck["source"]


# rows_to_update_test_ok = len(dfTest[dfTest["Delta test to current NLC category"]!= "No change"])
# rows_to_update_test_ok  = len(dfTest)


dfNewDSVCheckMerged = dfNewDSVCheck.merge(dfCurrDSVOriginalToCheck, how="left", on="SKU-Node", suffixes=("_New", "_Old"))
dfNewDSVCheckMerged["Price change"] = round(dfNewDSVCheckMerged["Price"] - dfNewDSVCheckMerged["walmart_dsv_price"],2)
dfNewDSVCheckMerged["Price change category"] = np.where(dfNewDSVCheckMerged["Price change"]>0, "Increase",
                                                      np.where(dfNewDSVCheckMerged["Price change"]<0, "Decrease", 
                                                      np.where(dfNewDSVCheckMerged["walmart_dsv_price"].isna(),"New","No change")))

if test:
    dfNewDSVCheckMerged["Is in Test SKU-Nodes"] = dfNewDSVCheckMerged["SKU-Node"].isin(dfTestDSV["SKU-Node"])
    no_change_test = len(dfTestTracker[dfTestTracker["Final delta vs current %"]== 0])

dfNewDSVCheckMergedChanged = dfNewDSVCheckMerged[dfNewDSVCheckMerged["Price change"] != 0].copy()
print("Should be", rows_to_update + rows_to_add, "rows") #+ rows_to_update_test_ok

if test:
    print("Removing no change in test should be", rows_to_update + rows_to_add - no_change_test, "rows") #+ rows_to_update_test_ok

dfNewDSVCheckMergedChanged["Is national?"] = np.where(dfNewDSVCheckMergedChanged["Source"] == "National", "Yes", "No")
display(dfNewDSVCheckMergedChanged["Is national?"].value_counts())
display(dfNewDSVCheckMergedChanged["Price change category"].value_counts())
dfNewDSVCheckMergedChanged

Should be 375866 rows


Is national?
No    375854
Name: count, dtype: int64

Price change category
Increase    207792
Decrease    103057
New          65005
Name: count, dtype: int64

,SKU,Price,Minimum margin,Source,SKU-Node,sku,walmart_dsv_price,source,Price change,Price change category,Is national?
2985465,ACCE0551521575S,82.10,4%,628326916,ACCE0551521575S-628326916,ACCE0551521575S,87.25,628326916,-5.15,Decrease,No
2985466,ACCE0671722555WXL,71.59,4%,628326984,ACCE0671722555WXL-628326984,ACCE0671722555WXL,73.14,628326984,-1.55,Decrease,No
2985467,ACCE0672024535YXL,73.37,4%,6283261151,ACCE0672024535YXL-6283261151,ACCE0672024535YXL,70.98,6283261151,2.39,Increase,No
2985468,ACCE0672024535YXL,73.37,4%,62832645,ACCE0672024535YXL-62832645,ACCE0672024535YXL,70.98,62832645,2.39,Increase,No
2985469,ADVN0131825565T,141.49,4%,628326686,ADVN0131825565T-628326686,ADVN0131825565T,149.44,628326686,-7.95,Decrease,No
...,...,...,...,...,...,...,...,...,...,...,...
3361326,ZEET0251523575TXL,74.16,4%,628326592,ZEET0251523575TXL-628326592,NaN,NaN,NaN,NaN,New,No
3361327,ZEET1291620565V,58.43,4%,6283261551,ZEET1291620565V-6283261551,NaN,NaN,NaN,NaN,New,No
3361328,ZEET1291620565V,57.80,4%,62832617,ZEET1291620565V-62832617,NaN,NaN,NaN,NaN,New,No
3361329,ZENN0041822545WXL,62.87,4%,628326435,ZENN0041822545WXL-628326435,NaN,NaN,NaN,NaN,New,No


In [ ]:
# pathDSVCheck = r"H:\Shared drives\DevOps Projects\Python projecs\python-automations\WMT_pricing\outputs\New WalmartB2B DSV to upload 2026-03-20.csv"
# dfCheck = pd.read_csv(pathDSVCheck, dtype = {"Source": str})
# dfCheck = dfCheck.rename(columns={"SKU": "sku", "Price": "walmart_dsv_price", "Source": "source"})
# # dfCheck["source"] = dfCheck["source"].astype(str).fillna("National")
# dfCheck

,sku,walmart_dsv_price,Minimum margin,source
0,IRON0061417565H,40.11,4%,62832635
1,IRON0081417565H,40.11,4%,628326503
2,IRON0061417565H,40.11,4%,628326106
3,IRON0061417565H,40.11,4%,6283261141
4,APLS0031518555V,40.11,4%,628326916
...,...,...,...,...
3246992,YOKO3962028555E,335.53,4%,628326896
3246993,YOKO3962028555E,335.53,4%,62832691
3246994,YOKO3962028555E,335.53,4%,62832692
3246995,ZEET1231825570H,105.89,4%,62832617


# Update tracker

In [156]:
dfCurrentTestsAllNew = dfCurrentTestsAll.copy()
dfCurrentTestsAllNew["SKU-Node"] = dfCurrentTestsAllNew["Product Code"] + "-" + dfCurrentTestsAllNew["Identifier"]

df_final_to_append_tracker = pd.concat([dfNewSKUNodesTracker, dfCurrDSVNLCCPriceUpdatesforTracker, 
                                            dfUpdateAboveMarginTracker, dfWmMarginSplitTracker, dfMarginTestTracker,
                                            dfDSVDTracker,dfPriceIncrTracker], ignore_index=True)

if test:
    df_final_to_append_tracker = pd.concat([df_final_to_append_tracker, dfTestTracker], ignore_index=True)

dfCurrentTestsAllNewNotUpdated = dfCurrentTestsAllNew[~dfCurrentTestsAllNew["SKU-Node"].isin(df_final_to_append_tracker["SKU-Node"])].copy()
dfCurrentTestsAllNew = pd.concat([dfCurrentTestsAllNewNotUpdated, df_final_to_append_tracker], ignore_index=True)
dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop(columns=["SKU-Node"])

#Check if therea re duplicates in dfCurrentTestsAllNew by Product Code and Identifier
duplicates = dfCurrentTestsAllNew[dfCurrentTestsAllNew.duplicated(subset=["Product Code", "Identifier"], keep=False)]

if not duplicates.empty:
    print("Warning: There are duplicates in dfCurrentTestsAllNew based on Product Code and Identifier:")
    dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop_duplicates(subset=["Product Code", "Identifier"], keep="last")
    display(duplicates.sort_values(by=["Product Code", "Identifier"]))

# dfCurrentTestsAllNew = dfCurrentTestsAllNew.merge(dfCurrentTestsAllPrev, how="left", on=["Product Code", "Identifier"])

print("Final count of tests:", len(dfCurrentTestsAllNew))
print("Should be:", len(dfCurrentTestsAll) + len(dfNewSKUNodesTracker))

display(dfCurrentTestsAllNew["Final target"].value_counts().sort_index())
display(dfCurrentTestsAllNew["Start date"].value_counts().sort_index())

dfCurrentTestsAllNew

Final count of tests: 3381812
Should be: 3419593


Final target
Added                       1160441
DSVD test                    291097
Decreased - margin > 20%     149472
Increase test                101521
Less sales                    13930
Less sales back               14381
Margin test                  298305
No sales test                238847
Normal strategy              489856
Shipping cost added           31584
Updated                      390573
Updates test                 107829
Wm margin split test          93976
Name: count, dtype: int64

Start date
2026-01-05    608896
2026-01-16     93876
2026-01-20    219970
2026-01-27     47125
2026-01-28     13967
2026-01-29      3717
2026-01-30    340891
2026-02-03     76023
2026-02-04     14946
2026-02-05     13782
2026-02-06    107805
2026-02-09     25916
2026-02-11     38465
2026-02-12    132525
2026-02-16     42177
2026-02-17     14299
2026-02-18     20314
2026-02-19     27928
2026-02-20      9247
2026-02-24     18418
2026-02-25     47773
2026-02-26      8628
2026-02-27      8229
2026-03-02     14358
2026-03-03     22542
2026-03-04     13803
2026-03-05      7122
2026-03-06     10206
2026-03-07     30504
2026-03-09    101883
2026-03-10     49320
2026-03-11       229
2026-03-12    313922
2026-03-13     82525
2026-03-16     28459
2026-03-17     81354
2026-03-18     18104
2026-03-19      7521
2026-03-20     11763
2026-03-23     22751
2026-03-24    310383
2026-03-26     22060
2026-03-27      7940
2026-03-30    119879
2026-03-31      8527
2026-04-02    161740
Name: count, dtype: int

,Product Code,Identifier,Final target,Final price category,Final price,Final delta vs current %,Final price change category final,Category inventory,Node type,Start date,Min units,Sub-group,Is in stock?,Last price update,current_nlc_margin,Current walmart margin at NLC category,sku_sales_category,current_nlc_margin_date
0,ACCE0131318560V,6283261261,Normal strategy,11%,70.06,0.00,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-04-02
1,ACCE0131318560V,6283261264,Normal strategy,11%,70.76,0.00,No change,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-04-02
2,ACCE0131318560V,6283261391,Normal strategy,11%,67.87,NaN,New,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-04-02
3,ACCE0131418555V,6283261391,Normal strategy,11%,56.13,-0.02,Decrease,NaN,HUB,2026-01-05,8,11%,No,NaN,NaN,NaN,No sales,2026-04-02
4,ACCE0221316580T,6283261113,Normal strategy,11%,43.25,0.03,Increase,NaN,HUB,2026-01-05,8,11%,Yes,NaN,0.1188,0% to 10%,95-100%,2026-04-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3381807,YOKO3961626570T,628326890,Increase test,11%,NaN,0.03,Increase,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
3381808,YOKO3961827565H,628326922,Increase test,11%,NaN,0.03,Increase,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
3381809,YOKO3961828575E,62832628,Increase test,6%,NaN,0.00,No change,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN
3381810,YOKO3961828575E,628326916,Increase test,8%,NaN,0.00,No change,NaN,HUB,2026-03-30,4,Increased,NaN,2026-04-02,NaN,NaN,NaN,NaN


In [158]:
pathTrackerBk = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Bk tracker\Final node level costs tracker_bk.csv"
pathTrackerBk = pathTracker.replace("Final node level costs tracker.csv", f"Bk tracker\Final node level costs tracker_{date_str}.csv")
dfCurrentTestsAllNew.to_csv(pathTracker, index=False)
print("Tracker updated")
dfCurrentTestsAllNew.to_csv(pathTrackerBk, index=False)
print("Backup created")

Tracker updated
Backup created


In [165]:
dfCurrentTestsAllNew["Final target"].value_counts()

Final target
Added                       1162198
Normal strategy              546485
Updated                      346644
Margin test                  303443
DSVD test                    291097
No sales test                252075
Decreased - margin > 20%     148411
Updates test                 126438
Wm margin split test          95983
Shipping cost added           31918
Less sales                    15329
Less sales back               14996
Name: count, dtype: int64

In [166]:
dfCurrentTestsAllNew["Start date"].value_counts()

Start date
2026-01-05    656492
2026-01-30    366062
2026-03-12    321307
2026-03-24    311845
2026-01-20    224247
2026-02-12    153364
2026-02-06    126755
2026-03-13    107187
2026-03-09    104946
2026-01-16    102147
2026-03-17     87739
2026-02-03     81193
2026-03-10     56238
2026-02-25     53274
2026-01-27     50249
2026-02-16     44892
2026-02-11     40989
2026-03-16     34711
2026-03-03     30948
2026-03-07     30774
2026-02-19     30344
2026-02-09     28454
2026-03-23     26281
2026-03-26     25111
2026-02-18     22917
2026-02-24     20307
2026-03-18     19844
2026-02-04     16073
2026-03-02     15945
2026-02-17     15624
2026-03-04     15619
2026-02-05     14978
2026-01-28     14820
2026-03-20     12663
2026-03-06     12110
2026-02-20     10236
2026-02-26      9633
2026-02-27      9100
2026-03-27      8849
2026-03-19      8370
2026-03-05      8075
2026-01-29      4021
2026-03-11       284
Name: count, dtype: int64

In [167]:
# dfPrevDSVOriginalFix = dfPrevDSVOriginal.copy()
# dfPrevDSVOriginalFix["SKU-Node"] = dfPrevDSVOriginalFix["sku"] + "-" + dfPrevDSVOriginalFix["source"]
# dfPrevDSVOriginalFix["SKU-Node-Price"] = dfPrevDSVOriginalFix["sku"] + "-" + dfPrevDSVOriginalFix["source"] + "-" + dfPrevDSVOriginalFix["walmart_dsv_price"].astype(str)

# dfPrevDSVOriginalFix = dfPrevDSVOriginalFix.rename(columns={"walmart_dsv_price": "prev_nlc_price", "sku": "Product Code", "source": "Identifier"}).drop(columns=["date"])
# dfPrevDSVOriginalFix

In [168]:
# dfCurrDSVOriginalFix = dfCurrDSVOriginal.copy()
# dfCurrDSVOriginalFix["SKU-Node"] = dfCurrDSVOriginalFix["sku"] + "-" + dfCurrDSVOriginalFix["source"]
# dfCurrDSVOriginalFix["SKU-Node-Price"] = dfCurrDSVOriginalFix["sku"] + "-" + dfCurrDSVOriginalFix["source"] + "-" + dfCurrDSVOriginalFix["walmart_dsv_price"].astype(str)

# dfCurrDSVOriginalFix["Start date"] = '2026-02-24'
# dfCurrDSVOriginalFix["Min units"] = 8
# dfCurrDSVOriginalFix["Node type"] = "HUB"
# dfCurrDSVOriginalFix["Final price category"] = "11%"
# dfCurrDSVOriginalFix = dfCurrDSVOriginalFix.rename(columns={"walmart_dsv_price": "Final price","sku": "Product Code", "source": "Identifier"}).drop(columns=["date"])
# dfCurrDSVOriginalFix["Final target"] = ""
# dfCurrDSVOriginalFix["Category inventory"] = ""
# dfCurrDSVOriginalFix

In [169]:
# dfNewSKUNodesTrackerPrev = dfCurrDSVOriginalFix[~dfCurrDSVOriginalFix["SKU-Node"].isin(dfPrevDSVOriginalFix["SKU-Node"])].copy()
# dfNewSKUNodesTrackerPrev["Final target"] = "Added"
# dfNewSKUNodesTrackerPrev

In [170]:
# dfNewSKUNodesTracker = pd.concat([dfNewSKUNodesTrackerPrev, dfNewSKUNodesTracker], ignore_index=True)
# dfNewSKUNodesTracker["Start date"].value_counts()

In [171]:
# dfCurrDSVNLCCPriceUpdatesforTrackerPrev = dfCurrDSVOriginalFix[~dfCurrDSVOriginalFix["SKU-Node-Price"].isin(dfPrevDSVOriginalFix["SKU-Node-Price"])].copy()
# dfCurrDSVNLCCPriceUpdatesforTrackerPrev = dfCurrDSVNLCCPriceUpdatesforTrackerPrev[~dfCurrDSVNLCCPriceUpdatesforTrackerPrev["SKU-Node"].isin(dfNewSKUNodesTrackerPrev["SKU-Node"])].copy()

# dfCurrDSVNLCCPriceUpdatesforTrackerPrev["Final target"] = "Updated"
# dfCurrDSVNLCCPriceUpdatesforTrackerPrev

In [172]:
# dfCurrDSVNLCCPriceUpdatesforTracker = pd.concat([dfCurrDSVNLCCPriceUpdatesforTrackerPrev, dfCurrDSVNLCCPriceUpdatesforTracker], ignore_index=True)
# dfCurrDSVNLCCPriceUpdatesforTracker["Start date"].value_counts()

In [173]:
# dfTargets = dfCurrentTestsAll[["Final target"]].drop_duplicates().copy()
# dfTargets.to_excel("Targets mapping.xlsx", index=False)
# dfTargetsFull = pd.read_excel("Targets mapping.xlsx")
# dfTargetsFull
# dfCurrentTestsAllNew[["Final target","Old final target","Start date"]].value_counts()
# dfCurrentTestsAllNew[["Final target", "Old final target", "Start date", "Min units", "Node type"]].value_counts().reset_index(name="count")
# 

In [174]:
# dfCurrentTestsAllPrev = dfCurrentTestsAll.copy()
# dfCurrentTestsAllPrev["Final target prev"] = dfCurrentTestsAllPrev["Final target"]
# dfCurrentTestsAllPrev["Start date prev"] = dfCurrentTestsAllPrev["Start date"]
# dfCurrentTestsAllPrev = dfCurrentTestsAllPrev[["Product Code", "Identifier", "Final target prev", "Start date prev"]].copy()
# dfCurrentTestsAllPrev

In [175]:
# dfCurrentTestsAllNew["SKU-Node"] = dfCurrentTestsAllNew["Product Code"] + "-" + dfCurrentTestsAllNew["Identifier"]
# dfCurrentTestsAllNew = dfCurrentTestsAllNew.merge(dfOutput[["SKU-Node","current_nlc_margin"]], how = "left", on="SKU-Node")
# dfCurrentTestsAllNew["current_nlc_margin_date"] = today_str
# dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop(columns=["SKU-Node"])

In [176]:
# dfCurrentTestsAllNew = dfCurrentTestsAll.copy()

# #dfCurrentTestsAllNew["Final target prev"] is a copy of Final target

# dfCurrentTestsAllNew["SKU-Node"] = dfCurrentTestsAllNew["Product Code"] + "-" + dfCurrentTestsAllNew["Identifier"]

# dfCurrentTestsAllNewNotUpdated = dfCurrentTestsAllNew[~dfCurrentTestsAllNew["SKU-Node"].isin(dfDSVChanges["SKU-Node"])].copy()
# dfCurrentTestsAllNew = pd.concat([dfCurrentTestsAllNewNotUpdated, dfDSVChanges], ignore_index=True)


# # dfCurrentTestsAllNew["Node type"] = dfCurrentTestsAllNew["Node type"].fillna("HUB")
# dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop(columns=["SKU-Node"])

# #Check if therea re duplicates in dfCurrentTestsAllNew by Product Code and Identifier
# duplicates = dfCurrentTestsAllNew[dfCurrentTestsAllNew.duplicated(subset=["Product Code", "Identifier"], keep=False)]

# if not duplicates.empty:
#     print("Warning: There are duplicates in dfCurrentTestsAllNew based on Product Code and Identifier:")
#     dfCurrentTestsAllNew = dfCurrentTestsAllNew.drop_duplicates(subset=["Product Code", "Identifier"], keep="last")
#     display(duplicates)


# Errors in price updates checks

Wait 3 hours since we uploaded the DSV file into hybris before running this. Flag to Fran if the % 

In [20]:
creds_path = ftp_server.create_path_cred(file_name = "ftp_credentials_fran.json")
host, port, username, password = ftp_server.read_credentials(creds_path)
ftp = ftp_server.connect_ftp(host, port, username, password)
path = "/external-merchants/WalmartB2B/price"

import pandas as pd

def ftp_list_names_only(ftp, path):
    ftp.cwd(path)
    names = ftp.nlst()
    return pd.DataFrame({"filename": names})

df = ftp_list_names_only(ftp,path)
# 1) extract datetime string
dt_str = df["filename"].str.extract(r"_(\d{8}-\d{6})_", expand=False)

# 2) convert to datetime
df["file_datetime"] = pd.to_datetime(dt_str, format="%Y%m%d-%H%M%S")

# 3) tag response files
df["is_response"] = df["filename"].str.contains("_response", case=False, na=False)

df["File date"] = df["file_datetime"].dt.date
display(df["File date"].value_counts().sort_index(ascending=False))

dfUseful = df[df["File date"] >= pd.to_datetime(yesterday_str).date()].copy()
dfUseful = dfUseful[dfUseful["is_response"] == True].copy()
dfUseful

Connected to FTP server: upload.tires-easy.com


File date
2026-04-01    1179
2026-03-31     394
2026-03-30      32
2026-03-27       4
2026-03-26      22
2026-03-24     162
2026-03-23      25
2026-03-20      10
2026-03-19       8
2026-03-18      39
2026-03-17      56
2026-03-16      24
2026-03-13      44
2026-03-12     112
2026-03-11       4
2026-03-10      23
2026-03-09      32
2026-03-06      14
2026-03-05    1485
2026-03-04       8
2026-03-03      12
2026-03-02       8
2026-02-27       4
2026-02-26       6
2026-02-25      16
2026-02-24      10
2026-02-20       6
2026-02-19      31
2026-02-18      12
2026-02-17       9
2026-02-16      22
2026-02-13       2
2026-02-12     143
2026-02-11      96
2026-02-09      16
2026-02-06     110
2026-02-05      17
2026-02-04      33
2026-02-03    1290
2026-02-02    1283
2026-01-30     318
2026-01-29      28
2026-01-28       8
2026-01-26      29
2026-01-20     230
2026-01-19     182
2026-01-16      74
2026-01-05     927
2025-12-16       2
2025-12-15     156
2025-11-26      22
2025-11-25     263
20

,filename,file_datetime,is_response,File date
7634,price_feed_20260331-124348_ca560f96-f07e-44ab-...,2026-03-31 12:43:48,True,2026-03-31
7636,price_feed_20260331-124350_cea37589-0d27-45d0-...,2026-03-31 12:43:50,True,2026-03-31
7638,price_feed_20260331-124351_d2681c3e-2f81-4546-...,2026-03-31 12:43:51,True,2026-03-31
7640,price_feed_20260331-124352_837cd2da-3ad6-4c9e-...,2026-03-31 12:43:52,True,2026-03-31
7642,price_feed_20260331-124353_39bc5989-0430-468d-...,2026-03-31 12:43:53,True,2026-03-31
...,...,...,...,...
9197,price_feed_20260401-120721_f5253307-ce45-4430-...,2026-04-01 12:07:21,True,2026-04-01
9199,price_feed_20260401-121105_377877d9-1d92-43eb-...,2026-04-01 12:11:05,True,2026-04-01
9201,price_feed_20260401-121906_880f8a73-0b0a-452c-...,2026-04-01 12:19:06,True,2026-04-01
9203,price_feed_20260401-122106_d312d7d0-9d3f-46ca-...,2026-04-01 12:21:06,True,2026-04-01


In [21]:
import os

folder_price_updates = r"H:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_01 Node level costs\Price updates check"
folder_today = os.path.join(folder_price_updates, today_str)
folder_xml= os.path.join(folder_today, "response_files")
os.makedirs(folder_xml, exist_ok=True)

ftp.cwd(path) #"/external-merchants/WalmartB2B/price"

count = 0
for fname in dfUseful["filename"]:
    local_path = os.path.join(folder_xml, fname)

    with open(local_path, "wb") as f:
        ftp.retrbinary(f"RETR {fname}", f.write)

    count += 1
    if count % 10 == 0:
        print(f"Downloaded {count}/{len(dfUseful)}")

print(f"Downloaded {len(dfUseful)} files to {folder_xml}")

Downloaded 10/687
Downloaded 20/687
Downloaded 30/687
Downloaded 40/687
Downloaded 50/687
Downloaded 60/687
Downloaded 70/687
Downloaded 80/687
Downloaded 90/687
Downloaded 100/687
Downloaded 110/687
Downloaded 120/687
Downloaded 130/687
Downloaded 140/687
Downloaded 150/687
Downloaded 160/687
Downloaded 170/687
Downloaded 180/687
Downloaded 190/687
Downloaded 200/687
Downloaded 210/687
Downloaded 220/687
Downloaded 230/687
Downloaded 240/687
Downloaded 250/687
Downloaded 260/687
Downloaded 270/687
Downloaded 280/687
Downloaded 290/687
Downloaded 300/687
Downloaded 310/687
Downloaded 320/687
Downloaded 330/687
Downloaded 340/687
Downloaded 350/687
Downloaded 360/687
Downloaded 370/687
Downloaded 380/687
Downloaded 390/687
Downloaded 400/687
Downloaded 410/687
Downloaded 420/687
Downloaded 430/687
Downloaded 440/687
Downloaded 450/687
Downloaded 460/687
Downloaded 470/687
Downloaded 480/687
Downloaded 490/687
Downloaded 500/687
Downloaded 510/687
Downloaded 520/687
Downloaded 530/687
Do

In [22]:
def read_xml_file(file_path_xml):
    ns_uri = 'http://walmart.com/'
    ns = {'ns': ns_uri}
    tag_item_status = f'{{{ns_uri}}}itemIngestionStatus'

    records = []

    for event, elem in ET.iterparse(file_path_xml, events=('end',)):
        if elem.tag == tag_item_status:
            index_elem = elem.find('ns:index', ns)
            product_id_elem = elem.find('.//ns:productId', ns)
            ship_node = elem.find('ns:shipNode', ns)
            status_elem = elem.find('ns:ingestionStatus', ns)

            # ---- error handling ----
            error_elem = elem.find('.//ns:ingestionError', ns)

            error_type = error_elem.find('ns:type', ns).text if error_elem is not None and error_elem.find('ns:type', ns) is not None else None
            error_code = error_elem.find('ns:code', ns).text if error_elem is not None and error_elem.find('ns:code', ns) is not None else None
            error_field = error_elem.find('ns:field', ns).text if error_elem is not None and error_elem.find('ns:field', ns) is not None else None
            error_description = error_elem.find('ns:description', ns).text if error_elem is not None and error_elem.find('ns:description', ns) is not None else None

            records.append({
                'index': index_elem.text if index_elem is not None else None,
                'productId': product_id_elem.text if product_id_elem is not None else None,
                'shipNode': ship_node.text if ship_node is not None else None,
                'ingestionStatus': status_elem.text if status_elem is not None else None,
                'error_type': error_type,
                'error_code': error_code,
                'error_field': error_field,
                'error_description': error_description
            })

            elem.clear()  # free memory

    df_xml = pd.DataFrame(records)
    return df_xml

In [23]:
df_xml_all = pd.DataFrame()
count = 0
for file in os.listdir(folder_xml):
    file_path_xml = os.path.join(folder_xml, file)
    df_xml = read_xml_file(file_path_xml)
    # print(f"File: {file}, Rows: {len(df_xml)}")
    # display(df_xml["ingestionStatus"].value_counts())
    df_xml_all = pd.concat([df_xml_all, df_xml], ignore_index=True)

    count += 1
    #every 10 print
    if count % 10 == 0:
        print(f"Processed {count} files")

display(df_xml_all)

print("Overall ingestion status counts:")
display(df_xml_all["ingestionStatus"].value_counts())


df_xml_all["SKU-Node"] = df_xml_all["productId"] + "-" + df_xml_all["shipNode"]
df_errors = df_xml_all[df_xml_all["ingestionStatus"] != "SUCCESS"].copy()
df_success = df_xml_all[df_xml_all["ingestionStatus"] == "SUCCESS"].copy()
df_errors_not_success = df_errors[~df_errors["SKU-Node"].isin(df_success["SKU-Node"])].copy()
df_errors_not_success_unique = df_errors_not_success.drop_duplicates(subset=["SKU-Node"], keep="last")
df_errors_not_success_unique["ingestionStatus"].value_counts()

df_xml_all_unique = pd.concat([df_success, df_errors_not_success_unique], ignore_index=True)

print("Ingestion status counts for unique SKU-Nodes:")
display(df_xml_all_unique["ingestionStatus"].value_counts())
# df_errors_not_success_unique[["error_field", "error_description"]].value_counts().reset_index(name="count")

Processed 10 files
Processed 20 files
Processed 30 files
Processed 40 files
Processed 50 files
Processed 60 files
Processed 70 files
Processed 80 files
Processed 90 files
Processed 100 files
Processed 110 files
Processed 120 files
Processed 130 files
Processed 140 files
Processed 150 files
Processed 160 files
Processed 170 files
Processed 180 files
Processed 190 files
Processed 200 files
Processed 210 files
Processed 220 files
Processed 230 files
Processed 240 files
Processed 250 files
Processed 260 files
Processed 270 files
Processed 280 files
Processed 290 files
Processed 300 files
Processed 310 files
Processed 320 files
Processed 330 files
Processed 340 files
Processed 350 files
Processed 360 files
Processed 370 files
Processed 380 files
Processed 390 files
Processed 400 files
Processed 410 files
Processed 420 files
Processed 430 files
Processed 440 files
Processed 450 files
Processed 460 files
Processed 470 files
Processed 480 files
Processed 490 files
Processed 500 files
Processed

,index,productId,shipNode,ingestionStatus,error_type,error_code,error_field,error_description
0,0,YOKO20520351250E,628326524,SUCCESS,None,None,None,None
1,1,ADVN0611620555W~wm,628326527,SUCCESS,None,None,None,None
2,2,ADVN0611620555W,628326527,SUCCESS,None,None,None,None
3,3,ADVN0611620555W~wm,628326663,SUCCESS,None,None,None,None
4,4,ADVN0611620555W,628326663,SUCCESS,None,None,None,None
...,...,...,...,...,...,...,...,...
6307890,12,COOP1402026550VXL~wm,628326874,SUCCESS,None,None,None,None
6307891,13,COOP1402026550VXL~wm2,628326874,SUCCESS,None,None,None,None
6307892,14,COOP1402026550VXL,628326683,SUCCESS,None,None,None,None
6307893,15,COOP1402026550VXL~wm,628326683,SUCCESS,None,None,None,None


Overall ingestion status counts:


ingestionStatus
SUCCESS          4637614
TIMEOUT_ERROR    1309283
INPROGRESS        320461
DATA_ERROR         40537
Name: count, dtype: int64

Ingestion status counts for unique SKU-Nodes:


ingestionStatus
SUCCESS       4637614
DATA_ERROR      39877
Name: count, dtype: int64

In [24]:
# Total not successes
not_success = len(df_xml_all_unique[df_xml_all_unique["ingestionStatus"] != "SUCCESS"])
total_success = len(df_xml_all_unique[df_xml_all_unique["ingestionStatus"] == "SUCCESS"])

perc_failure = round(not_success / (not_success + total_success),4)
print(f"Failure rate: {perc_failure:.2%} with {not_success} failures and {total_success} successes.")
if perc_failure > perc_failure_flag:
    print(f"ALERT: High failure rate of {perc_failure:.2%} with {not_success} failures and {total_success} successes.")
    #Flag to Fran.

Failure rate: 0.85% with 39877 failures and 4637614 successes.


In [25]:
df_success_by_sku = df_success.groupby("productId").size().reset_index(name="count").sort_values(by="count", ascending=False)

# df_errors_content_sku = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "content.SKU"].copy()
# df_errors_content_sku = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "unitCost.currencyAmount"].copy()
df_errors_content_sku = df_errors_not_success_unique.copy()

df_errors_content_sku_agg = df_errors_content_sku.groupby(["productId","error_field"]).size().reset_index(name="count").sort_values(by="count", ascending=False)
df_errors_content_sku_agg_with_success = df_errors_content_sku_agg.merge(df_success_by_sku, how="left", on="productId", suffixes=("_errors", "_success"))

df_errors_content_sku_agg_with_success["count_success"] = df_errors_content_sku_agg_with_success["count_success"].fillna(0).astype(int)
df_errors_content_sku_agg_with_success["Has success"] = np.where(df_errors_content_sku_agg_with_success["count_success"]>0, "Yes", "No")
df_errors_content_sku_agg_with_success["% errors"] = round(df_errors_content_sku_agg_with_success["count_errors"] / (df_errors_content_sku_agg_with_success["count_errors"] + df_errors_content_sku_agg_with_success["count_success"]),4)
display(df_errors_content_sku_agg_with_success)

df_success_by_node = df_success.groupby("shipNode").size().reset_index(name="count").sort_values(by="count", ascending=False)
df_errors_content_node = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "content.legacyDistributorId"].copy()
# df_errors_content_sku = df_errors_not_success_unique[df_errors_not_success_unique["error_field"] == "unitCost.currencyAmount"].copy()

# df_errors_content_node = df_errors_not_success_unique.copy()

#Group by productId and count rows
df_errors_content_node_agg = df_errors_content_node.groupby("shipNode").size().reset_index(name="count").sort_values(by="count", ascending=False)

df_errors_content_node_agg_with_success = df_errors_content_node_agg.merge(df_success_by_node, how="left", on="shipNode", suffixes=("_errors", "_success"))


df_errors_content_node_agg_with_success["count_success"] = df_errors_content_node_agg_with_success["count_success"].fillna(0).astype(int)
df_errors_content_node_agg_with_success["Has success"] = np.where(df_errors_content_node_agg_with_success["count_success"]>0, "Yes", "No")
df_errors_content_node_agg_with_success["% errors"] = round(df_errors_content_node_agg_with_success["count_errors"] / (df_errors_content_node_agg_with_success["count_errors"] + df_errors_content_node_agg_with_success["count_success"]),4)
df_errors_content_node_agg_with_success = df_errors_content_node_agg_with_success.sort_values(by="% errors", ascending=False)
df_errors_content_node_agg_with_success

,productId,error_field,count_errors,count_success,Has success,% errors
0,TOYO3282025550HXL,unitCost.currencyAmount,550,0,No,1.0000
1,FALK1271620555WXL,unitCost.currencyAmount,511,0,No,1.0000
2,FALK0461827565T,content.SKU,509,0,No,1.0000
3,FALK0461728570T,content.SKU,507,0,No,1.0000
4,KUMH0341721555V,unitCost.currencyAmount,497,0,No,1.0000
...,...,...,...,...,...,...
2100,FOUM04720351250E,unitCost.currencyAmount,1,13,Yes,0.0714
2101,FOUM0261519545V~wm,unitCost.currencyAmount,1,10,Yes,0.0909
2102,FOUM0261519545V,unitCost.currencyAmount,1,10,Yes,0.0909
2103,HANK2111720560WXL,unitCost.currencyAmount,1,0,No,1.0000


,shipNode,count_errors,count_success,Has success,% errors
580,628326875,1,63,Yes,0.0156
399,6283261552,3,374,Yes,0.0080
497,6283261231,2,248,Yes,0.0080
499,6283261266,2,315,Yes,0.0063
328,6283261553,4,708,Yes,0.0056
...,...,...,...,...,...
521,628326686,1,9506,Yes,0.0001
522,628326685,1,14135,Yes,0.0001
523,628326690,1,8854,Yes,0.0001
686,628326457,1,8309,Yes,0.0001


In [26]:
#send to excel, one sheet with summary of df_xml_all["ingestionStatus"].value_counts(), other sheet the df_xml_all where ingestionStatus != "SUCCESS"
output_excel_path = os.path.join(folder_today, f"NLC Price update response summary {today_str}.xlsx")
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    df_summary = df_xml_all_unique["ingestionStatus"].value_counts().reset_index()
    df_summary["percent"] = (
        df_summary["count"] / df_summary["count"].sum()
    )


    df_summary.columns = ["ingestionStatus", "Counts", "Percentage"]
    df_summary.to_excel(writer, sheet_name='Summary', index=False)

    df_errors_content_sku_agg_with_success.to_excel(writer, sheet_name='Errors by SKU-Reason', index=False)

    df_summary_errors = df_errors_not_success_unique["error_description"].value_counts().reset_index()
    df_summary_errors.columns = ["error_description", "Counts"]

    df_summary_errors.to_excel(writer, sheet_name='Error Summary', index=False)

    df_errors_not_success_unique.to_excel(writer, sheet_name='Errors list', index=False)